In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q monai

In [ ]:
import monai
import os
import glob
import re
import pandas as pd
import torch
import nibabel as nib
import numpy as np
from monai.transforms import Compose, Resize, NormalizeIntensity

# ============================================================
# PARCHE: convertir cualquier salida de MONAI a torch.Tensor puro
# ============================================================
def to_plain_tensor(x, ensure_channel_dim=True):
    """
    Convierte Tensor / MetaTensor / ndarray a torch.Tensor puro,
    sin metadatos MONAI, en float32 y contiguo.
    """
    if torch.is_tensor(x):
        x = x.detach().clone()
    else:
        x = torch.as_tensor(x)

    x = x.to(torch.float32).contiguous()

    if ensure_channel_dim and x.ndim == 3:
        x = x.unsqueeze(0)   # -> (1, H, W, D)

    if x.ndim != 4:
        raise ValueError(f"Se esperaba tensor con shape (1,H,W,D), llegó {tuple(x.shape)}")

    # importante: reconstruir como Tensor puro para romper con MetaTensor
    x = torch.tensor(x.cpu().numpy(), dtype=torch.float32)
    return x.contiguous()


def save_plain_mri_pt(x, save_path):
    """
    Guarda SIEMPRE un dict sencillo con tensor puro.
    """
    x_plain = to_plain_tensor(x, ensure_channel_dim=True)
    torch.save({"x": x_plain}, save_path)

In [ ]:
"""
atlas_utils.py
==============
Section C of the Materials and Methods.

Defines an atlas manager that:
  1. Loads an integer-valued atlas in template space.
  2. Extracts ROI masks {R_k}_{k=1}^K.
  3. Resamples those masks either to image space or feature-map space.
  4. Normalizes each ROI mask so that its support sums to 1, matching the
     masked pooling formula used by ROITokenizer.

The module does not assume a specific atlas vendor. It only requires a NIfTI
label map with integer region identifiers.
"""



from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Sequence, Union

import nibabel as nib
import numpy as np
import torch
import torch.nn.functional as F

ArrayLikePath = Union[str, Path]


@dataclass
class AtlasConfig:
    label_values: Optional[Sequence[int]] = None
    drop_background: bool = True
    eps: float = 1e-8
    min_voxels_per_roi: int = 1


def load_label_atlas(path: ArrayLikePath):
    img = nib.load(str(path))
    img = nib.as_closest_canonical(img)
    atlas = img.get_fdata(dtype=np.float32)

    if atlas.ndim == 4:
        atlas = atlas[..., 0]

    atlas = np.rint(atlas).astype(np.int32)
    return img, atlas


def infer_label_values(
    atlas: np.ndarray,
    drop_background: bool = True,
    min_voxels_per_roi: int = 1,
) -> list[int]:
    vals = np.unique(atlas).tolist()
    vals = [int(v) for v in vals]

    if drop_background:
        vals = [v for v in vals if v != 0]

    if min_voxels_per_roi > 1:
        vals = [v for v in vals if int((atlas == v).sum()) >= min_voxels_per_roi]

    return sorted(vals)


class AtlasROIManager:
    def __init__(self, atlas_path: ArrayLikePath, config: Optional[AtlasConfig] = None):
        self.atlas_path = str(atlas_path)
        self.config = config or AtlasConfig()

        self.atlas_img, self.atlas_np = load_label_atlas(self.atlas_path)
        self.affine = self.atlas_img.affine.copy()
        self.shape = tuple(int(v) for v in self.atlas_np.shape)

        if self.config.label_values is not None:
            self.label_values = [int(v) for v in self.config.label_values]
        else:
            self.label_values = infer_label_values(
                self.atlas_np,
                drop_background=self.config.drop_background,
                min_voxels_per_roi=self.config.min_voxels_per_roi,
            )

        self.K = len(self.label_values)

        self._atlas_onehot = self._build_onehot(self.atlas_np, self.label_values)
        self.atlas_tensor = self._atlas_onehot  # alias público

        self.roi_volumes = self._atlas_onehot.flatten(1).sum(dim=1).long()

        self._validate_nonempty()

    @staticmethod
    def _build_onehot(atlas: np.ndarray, label_values: Sequence[int]) -> torch.Tensor:
        masks = []
        for lab in label_values:
            masks.append((atlas == int(lab)).astype(np.float32))

        if len(masks) == 0:
            raise ValueError("No ROI labels were found in the atlas.")

        onehot = np.stack(masks, axis=0)  # (K, H, W, D)
        return torch.from_numpy(onehot)

    def _validate_nonempty(self):
        empty = (self.roi_volumes <= 0).nonzero(as_tuple=False).flatten().tolist()
        if len(empty) > 0:
            bad_labels = [self.label_values[i] for i in empty]
            raise ValueError(
                f"El atlas contiene ROIs vacías después de cargarlo. "
                f"indices={empty}, labels={bad_labels}"
            )

    @staticmethod
    def _resize_masks(masks: torch.Tensor, target_shape: Sequence[int]) -> torch.Tensor:
        """
        masks: (K, H, W, D)
        output: (K, Ht, Wt, Dt)
        """
        if len(target_shape) != 3:
            raise ValueError(f"target_shape debe tener longitud 3, llegó: {target_shape}")

        x = masks.unsqueeze(1)  # (K,1,H,W,D)
        x = F.interpolate(x, size=tuple(int(v) for v in target_shape), mode="nearest")
        return x.squeeze(1)

    @staticmethod
    def _normalize_masks(masks: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
        flat = masks.flatten(1)
        denom = flat.sum(dim=1, keepdim=True).clamp_min(eps)
        flat = flat / denom
        return flat.view_as(masks)

    def get_masks(
        self,
        target_shape: Sequence[int],
        normalize: bool = True,
        device: Optional[torch.device] = None,
        dtype: torch.dtype = torch.float32,
    ) -> torch.Tensor:
        """
        Devuelve máscaras ROI remuestreadas al shape objetivo.
        Salida: (K, Ht, Wt, Dt)
        """
        masks = self._resize_masks(self._atlas_onehot.float(), target_shape)

        if normalize:
            masks = self._normalize_masks(masks, eps=self.config.eps)

        masks = masks.to(dtype=dtype)

        if device is not None:
            masks = masks.to(device)

        return masks

    def get_binary_masks(
        self,
        target_shape: Sequence[int],
        device: Optional[torch.device] = None,
        dtype: torch.dtype = torch.float32,
    ) -> torch.Tensor:
        masks = self._resize_masks(self._atlas_onehot.float(), target_shape)
        masks = (masks > 0.5).to(dtype=dtype)

        if device is not None:
            masks = masks.to(device)

        return masks

    def roi_weights_from_volume(
        self,
        power: float = 0.0,
        device: Optional[torch.device] = None,
        dtype: torch.dtype = torch.float32,
    ) -> torch.Tensor:
        """
        power = 0.0 -> pesos uniformes
        power > 0.0 -> inverse-volume weighting^power, renormalizado
        """
        vol = self._atlas_onehot.flatten(1).sum(dim=1).float().clamp_min(1.0)

        if power <= 0:
            w = torch.ones_like(vol)
        else:
            w = (1.0 / vol) ** power

        w = w / w.sum().clamp_min(self.config.eps)
        w = w.to(dtype=dtype)

        if device is not None:
            w = w.to(device)

        return w

    def maybe_validate_K(self, K_expected: int) -> None:
        if self.K != int(K_expected):
            raise ValueError(
                f"Atlas has K={self.K} regions, but the model/loss expects K={int(K_expected)}."
            )

    def summary(self) -> dict:
        return {
            "atlas_path": self.atlas_path,
            "shape": self.shape,
            "K": self.K,
            "label_min": int(min(self.label_values)) if self.K > 0 else None,
            "label_max": int(max(self.label_values)) if self.K > 0 else None,
            "n_background_voxels": int((self.atlas_np == 0).sum()),
            "roi_volumes_min": int(self.roi_volumes.min().item()) if self.K > 0 else None,
            "roi_volumes_max": int(self.roi_volumes.max().item()) if self.K > 0 else None,
        }

In [ ]:

"""
concept_targets.py
==================
Section K of the Materials and Methods.

This module implements a practical MRI-only anatomical target c_tilde_{n,k}
for each ROI. Because the Kaggle derivatives may not ship FreeSurfer-based
cortical thickness or per-region morphometric spreadsheets, we implement an
atlas-based tissue-loss summary that is directly computable from the 3D MRI.

The default biomarker is:
    s_{n,k} = fraction of ROI voxels below the subject-specific q-th
              percentile of the intracranial intensity distribution

This makes s_{n,k} a bounded regional tissue-loss proxy. It is then converted
into a concept target in [0,1] through a CN-referenced z-score and a sigmoid:
    c_tilde_{n,k} = sigmoid((s_{n,k} - mu_k_CN) / (sigma_k_CN + eps))

If you later obtain stronger morphometric measurements, only the extractor
function needs to be replaced; the normalizer and cache protocol can remain.
"""

from __future__ import annotations

import json
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional, Sequence, Union

import numpy as np
import torch

ArrayLikePath = Union[str, Path]

@dataclass
class ConceptTargetConfig:
    brain_threshold: float = 0.0
    low_intensity_percentile: float = 20.0
    eps: float = 1e-6
    normal_class_name: str = "CN"


def _safe_torch_load(path):
    obj = torch.load(str(path), map_location="cpu", weights_only=False)

    if torch.is_tensor(obj):
        x = obj
    elif isinstance(obj, dict):
        x = None
        for key in ["x", "image", "mri", "tensor", "volume"]:
            if key in obj:
                x = obj[key]
                break
        if x is None:
            raise KeyError(f"No se encontró tensor MRI en {path}")
    else:
        x = torch.as_tensor(obj)

    if not torch.is_tensor(x):
        x = torch.as_tensor(x)

    x = x.detach().to(torch.float32)
    x = torch.tensor(x.cpu().numpy(), dtype=torch.float32)
    return x


def _unwrap_tensorlike(obj):
    """
    Accept:
      - plain Tensor / MetaTensor
      - dict with keys like 'x', 'image', 'mri', 'tensor', 'volume'
    """
    if torch.is_tensor(obj):
        return obj

    if isinstance(obj, dict):
        for key in ["x", "image", "mri", "tensor", "volume"]:
            if key in obj:
                x = obj[key]
                if torch.is_tensor(x):
                    return x
                return torch.as_tensor(x)

    # final fallback
    return torch.as_tensor(obj)


def _to_numpy_volume(x: torch.Tensor | np.ndarray) -> np.ndarray:
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 4 and x.shape[0] == 1:
        x = x[0]
    if x.ndim != 3:
        raise ValueError(f"Expected 3D volume or (1,H,W,D), got shape {x.shape}.")
    return x.astype(np.float32)


def extract_tissue_loss_proxy(
    x: torch.Tensor | np.ndarray,
    atlas_mgr: AtlasROIManager,
    cfg: Optional[ConceptTargetConfig] = None,
) -> np.ndarray:
    cfg = cfg or ConceptTargetConfig()
    vol = _to_numpy_volume(x)
    roi_masks = atlas_mgr.get_binary_masks(vol.shape).cpu().numpy()   # (K,H,W,D)

    brain = vol[vol > cfg.brain_threshold]
    if brain.size == 0:
        raise ValueError("Empty brain mask after thresholding; cannot compute concept targets.")

    q = np.percentile(brain, cfg.low_intensity_percentile)
    proxy = np.zeros(atlas_mgr.K, dtype=np.float32)

    for k in range(atlas_mgr.K):
        mask = roi_masks[k] > 0
        if not np.any(mask):
            continue
        roi_vals = vol[mask]
        proxy[k] = float((roi_vals <= q).mean())

    return proxy.astype(np.float32)



def _to_torch_volume(x, device):
    if torch.is_tensor(x):
        vol = x.detach()
    else:
        vol = torch.as_tensor(x)

    vol = vol.float()

    # admitir (1,H,W,D) o (H,W,D)
    if vol.ndim == 4 and vol.shape[0] == 1:
        vol = vol[0]
    elif vol.ndim != 3:
        raise ValueError(f"Se esperaba volumen 3D o (1,H,W,D), llegó {tuple(vol.shape)}")

    return vol.to(device, non_blocking=True)


@dataclass
class ConceptNormalizer:
    mu: np.ndarray
    sigma: np.ndarray
    eps: float = 1e-6

    def transform(self, features: np.ndarray) -> np.ndarray:
        z = (features - self.mu[None, :]) / (self.sigma[None, :] + self.eps)
        return 1.0 / (1.0 + np.exp(-z))

    def save(self, path: ArrayLikePath) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "mu": self.mu.tolist(),
            "sigma": self.sigma.tolist(),
            "eps": float(self.eps),
        }
        path.write_text(json.dumps(payload, indent=2))

    @classmethod
    def load(cls, path: ArrayLikePath) -> "ConceptNormalizer":
        payload = json.loads(Path(path).read_text())
        return cls(
            mu=np.asarray(payload["mu"], dtype=np.float32),
            sigma=np.asarray(payload["sigma"], dtype=np.float32),
            eps=float(payload["eps"]),
        )


def fit_concept_normalizer(
    features: np.ndarray,
    labels: Sequence[str] | Sequence[int],
    normal_label: str | int = "CN",
    eps: float = 1e-6,
) -> ConceptNormalizer:
    features = np.asarray(features, dtype=np.float32)
    labels = np.asarray(labels)
    mask = labels == normal_label
    if mask.sum() == 0:
        raise ValueError(f"No samples found for normal_label={normal_label!r}.")
    ref = features[mask]
    mu = ref.mean(axis=0).astype(np.float32)
    sigma = ref.std(axis=0).astype(np.float32)
    return ConceptNormalizer(mu=mu, sigma=sigma, eps=eps)



def build_subject_concept_target(
    x: torch.Tensor | np.ndarray,
    atlas_mgr: AtlasROIManager,
    normalizer: ConceptNormalizer,
    cfg: Optional[ConceptTargetConfig] = None,
) -> torch.Tensor:
    feats = extract_tissue_loss_proxy(x, atlas_mgr, cfg=cfg)[None, :]
    c_tilde = normalizer.transform(feats)[0]
    return torch.from_numpy(c_tilde.astype(np.float32))


def precompute_concept_targets_from_dataframe(
    df,
    atlas_mgr: AtlasROIManager,
    x_column: str = "x_path",
    label_column: str = "label",
    subject_id_column: str = "subject_id",
    output_dir: ArrayLikePath = "./concept_targets",
    cfg: Optional[ConceptTargetConfig] = None,
) -> tuple[ConceptNormalizer, "pd.DataFrame"]:
    import pandas as pd

    cfg = cfg or ConceptTargetConfig()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    raw_features = []
    labels = []
    rows = []

    for _, row in df.iterrows():
        x = _safe_torch_load(row[x_column])
        feats = extract_tissue_loss_proxy(x, atlas_mgr, cfg=cfg)
        raw_features.append(feats)
        labels.append(row[label_column])

        rows.append({
            "subject_id": row[subject_id_column],
            "label": row[label_column],
            "x_path": row[x_column],
        })

    raw_features = np.stack(raw_features, axis=0)
    normalizer = fit_concept_normalizer(
        raw_features,
        labels=labels,
        normal_label=cfg.normal_class_name,
        eps=cfg.eps,
    )

    out_rows = []
    transformed = normalizer.transform(raw_features)

    for meta, c_tilde in zip(rows, transformed):
        save_path = output_dir / f"{meta['subject_id']}_c_target.pt"
        save_plain_vector_pt(torch.from_numpy(c_tilde.astype(np.float32)), save_path, key="c_target")

        out_rows.append({
            **meta,
            "concept_target_path": str(save_path),
        })

    normalizer.save(output_dir / "concept_normalizer.json")
    return normalizer, pd.DataFrame(out_rows)

In [ ]:

"""
jacobian_utils.py
=================
Section L of the Materials and Methods.

This module computes:
    g_{n,k}   = mean_{x in R_k} psi(J_n(x))
    g_bar     = normalized ROI-wise deformation summary

It supports two regimes:
  1. If a displacement field already exists, compute Jacobian directly.
  2. If only template and subject MRI are available, estimate a non-linear
     displacement field with SimpleITK (Demons registration).

The implementation is deliberately explicit because Jacobian-based terms are
part of the anatomical plausibility regularizer, not pathology ground truth.
"""

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Union

import nibabel as nib
import numpy as np
import torch

# from atlas_utils import AtlasROIManager

try:
    import SimpleITK as sitk
except Exception:  # pragma: no cover
    sitk = None

ArrayLikePath = Union[str, Path]

@dataclass
class JacobianConfig:
    psi: str = "neg_log"
    eps: float = 1e-6
    n_iterations: int = 50
    smooth_displacement_field: bool = True
    normalize_within_subject: bool = True


def _safe_torch_load(path):
    """
    Carga .pt confiables del pipeline propio.
    Soporta archivos antiguos que contienen MONAI MetaTensor.
    """
    obj = torch.load(str(path), map_location="cpu", weights_only=False)

    # Caso 1: tensor directo / MetaTensor
    if torch.is_tensor(obj):
        x = obj

    # Caso 2: dict con tensor MRI
    elif isinstance(obj, dict):
        x = None
        for key in ["x", "image", "mri", "tensor", "volume"]:
            if key in obj:
                x = obj[key]
                break
        if x is None:
            raise KeyError(f"No se encontró tensor MRI en {path}")
    else:
        x = torch.as_tensor(obj)

    # romper dependencia con MetaTensor
    if not torch.is_tensor(x):
        x = torch.as_tensor(x)

    x = x.detach().to(torch.float32)
    x = torch.tensor(x.cpu().numpy(), dtype=torch.float32)  # fuerza Tensor puro
    return x

def _unwrap_tensorlike(obj):
    if torch.is_tensor(obj):
        return obj

    if isinstance(obj, dict):
        for key in ["x", "image", "mri", "tensor", "volume"]:
            if key in obj:
                x = obj[key]
                if torch.is_tensor(x):
                    return x.detach().to(torch.float32)
                return torch.as_tensor(x, dtype=torch.float32)

    raise TypeError(f"Unsupported object type loaded from checkpoint: {type(obj)}")


def _tensor_to_3d_numpy(x) -> np.ndarray:
    if torch.is_tensor(x):
        x = x.detach().cpu().to(torch.float32)
        if x.ndim == 4 and x.shape[0] == 1:
            x = x[0]
        arr = x.numpy()
    else:
        arr = np.asarray(x, dtype=np.float32)
        if arr.ndim == 4 and arr.shape[0] == 1:
            arr = arr[0]

    if arr.ndim != 3:
        raise ValueError(f"Expected 3D volume or (1,H,W,D), got shape {arr.shape}")

    return np.ascontiguousarray(arr.astype(np.float32))


def _ensure_sitk():
    if sitk is None:
        raise ImportError(
            "SimpleITK is required for Jacobian computation from displacement fields "
            "or for Demons registration. Install SimpleITK before using jacobian_utils.py."
        )

def load_nifti_array(path: ArrayLikePath) -> np.ndarray:
    img = nib.load(str(path))
    img = nib.as_closest_canonical(img)
    arr = img.get_fdata(dtype=np.float32)
    if arr.ndim == 4:
        arr = arr[..., 0]
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def sitk_from_numpy(arr: np.ndarray) -> "sitk.Image":
    _ensure_sitk()
    img = sitk.GetImageFromArray(arr.astype(np.float32))
    img.SetSpacing((1.0, 1.0, 1.0))
    return img

def estimate_displacement_field(
    fixed_volume: np.ndarray,
    moving_volume: np.ndarray,
    cfg: Optional[JacobianConfig] = None,
) -> "sitk.Image":
    _ensure_sitk()
    cfg = cfg or JacobianConfig()

    fixed = sitk_from_numpy(fixed_volume)
    moving = sitk_from_numpy(moving_volume)

    matcher = sitk.HistogramMatchingImageFilter()
    matcher.SetNumberOfHistogramLevels(128)
    matcher.SetNumberOfMatchPoints(10)
    moving = matcher.Execute(moving, fixed)

    demons = sitk.DiffeomorphicDemonsRegistrationFilter()
    demons.SetNumberOfIterations(int(cfg.n_iterations))
    demons.SetStandardDeviations(1.0)
    displacement = demons.Execute(fixed, moving)

    if cfg.smooth_displacement_field:
        displacement = sitk.SmoothingRecursiveGaussian(displacement, 1.0)
    return displacement

def jacobian_determinant_from_displacement(displacement_field: "sitk.Image") -> np.ndarray:
    _ensure_sitk()
    jac = sitk.DisplacementFieldJacobianDeterminant(displacement_field)
    jac_np = sitk.GetArrayFromImage(jac).astype(np.float32)
    return np.nan_to_num(jac_np, nan=1.0, posinf=1.0, neginf=1.0)

def apply_psi(jac_det: np.ndarray, psi: str = "neg_log", eps: float = 1e-6) -> np.ndarray:
    jac_det = np.clip(jac_det, eps, None)
    if psi == "neg_log":
        return -np.log(jac_det).astype(np.float32)
    if psi == "identity":
        return jac_det.astype(np.float32)
    raise ValueError(f"Unknown psi={psi!r}")

def pool_roi_deformation(
    psi_jacobian: np.ndarray,
    atlas_mgr: AtlasROIManager,
) -> np.ndarray:
    masks = atlas_mgr.get_binary_masks(psi_jacobian.shape).cpu().numpy()
    out = np.zeros(atlas_mgr.K, dtype=np.float32)

    for k in range(atlas_mgr.K):
        mask = masks[k] > 0
        if np.any(mask):
            out[k] = float(psi_jacobian[mask].mean())
    return out

def normalize_roi_summary(g: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    g = np.asarray(g, dtype=np.float32)
    mu = float(g.mean())
    sigma = float(g.std())
    z = (g - mu) / (sigma + eps)
    return (1.0 / (1.0 + np.exp(-z))).astype(np.float32)

def compute_g_bar_from_template_and_subject(
    template_volume: np.ndarray,
    subject_volume: np.ndarray,
    atlas_mgr: AtlasROIManager,
    cfg: Optional[JacobianConfig] = None,
) -> torch.Tensor:
    cfg = cfg or JacobianConfig()
    displacement = estimate_displacement_field(template_volume, subject_volume, cfg=cfg)
    jac_det = jacobian_determinant_from_displacement(displacement)
    psi_jac = apply_psi(jac_det, psi=cfg.psi, eps=cfg.eps)
    g = pool_roi_deformation(psi_jac, atlas_mgr)

    if cfg.normalize_within_subject:
        g = normalize_roi_summary(g, eps=cfg.eps)
    return torch.from_numpy(g.astype(np.float32))

def to_plain_vector(x, expected_dim=1):
    """
    Convierte un vector ROI (por ejemplo g_bar o c_target) a torch.Tensor puro.
    """
    if torch.is_tensor(x):
        x = x.detach().clone()
    else:
        x = torch.as_tensor(x)

    x = x.to(torch.float32).contiguous()

    if expected_dim is not None and x.ndim != expected_dim:
        raise ValueError(f"Se esperaba tensor con ndim={expected_dim}, llegó shape={tuple(x.shape)}")

    # romper dependencia con MetaTensor si existiera
    x = torch.tensor(x.cpu().numpy(), dtype=torch.float32)
    return x.contiguous()


def save_plain_vector_pt(x, save_path, key="x"):
    """
    Guarda un vector anatómico puro, por ejemplo g_bar o c_target.
    """
    x_plain = to_plain_vector(x, expected_dim=1)
    torch.save({key: x_plain}, save_path)

def precompute_jacobians_from_dataframe(
    df,
    atlas_mgr: AtlasROIManager,
    template_x_path: ArrayLikePath,
    x_column: str = "x_path",
    subject_id_column: str = "subject_id",
    output_dir: ArrayLikePath = "./jacobian_targets",
    cfg: Optional[JacobianConfig] = None,
):
    import pandas as pd

    cfg = cfg or JacobianConfig()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    template_obj = _safe_torch_load(template_x_path)
    template_x = _tensor_to_3d_numpy(_safe_torch_load(template_x_path))

    rows = []
    for _, row in df.iterrows():
        x_obj = _safe_torch_load(row[x_column])
        x = _tensor_to_3d_numpy(_safe_torch_load(row[x_column]))

        g_bar = compute_g_bar_from_template_and_subject(
            template_volume=template_x,
            subject_volume=x,
            atlas_mgr=atlas_mgr,
            cfg=cfg,
        )

        save_path = output_dir / f"{row[subject_id_column]}_g_bar.pt"
        # torch.save(g_bar, save_path)
        save_plain_vector_pt(g_bar, save_path, key="g_bar")

        rows.append({
            "subject_id": row[subject_id_column],
            "x_path": row[x_column],
            "g_bar_path": str(save_path),
        })

    return pd.DataFrame(rows)

In [ ]:
import os
import glob
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader

# =====================================================================
# 1. LABEL ENCODING MAPPING
# =====================================================================
LABEL_MAP = {
    'CN': 0,
    'MCI': 1,
    'AD': 2
}

# =====================================================================
# 2. SOURCE DOMAIN DATASET (ADNI)
# =====================================================================
class SourceDomainDataset(Dataset):
    def __init__(self, csv_path, source_artifacts_dir):
        super().__init__()
        self.source_artifacts_dir = source_artifacts_dir
        
        df = pd.read_csv(csv_path)
        self.data = df[df['Label'].isin(LABEL_MAP.keys())].reset_index(drop=True)
        
        # [FIX] DYNAMIC PATH RESOLVER FOR ADNI
        # Instead of trusting the CSV path, we find exactly where the ADNI dataset 
        # is currently mounted in this specific Kaggle session to prevent FileNotFoundError.
        print("Locating ADNI source tensors dynamically...")
        self.mri_path_map = {}
        # Search Kaggle input for all ADNI .pt files
        adni_files = glob.glob("/kaggle/input/**/*.pt", recursive=True)
        for f in adni_files:
            # Only map the actual MRI tensors, ignore concepts/jacobians and oasis
            if "c_target" not in f and "g_jacobian" not in f and "oasis" not in f:
                basename = os.path.basename(f).replace('.pt', '')
                self.mri_path_map[basename] = f
                
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        sub_id = str(row['Subject_ID'])
        label_str = row['Label']
        
        # 1. Load Preprocessed MRI Tensor (X^S)
        # Look up the newly resolved path from our dictionary
        if sub_id in self.mri_path_map:
            mri_path = self.mri_path_map[sub_id]
        else:
            # Fallback if mapping failed (relies on standard Kaggle structure)
            mri_path = str(row['File_Path']).replace("/datasets/sanjayjoshy/", "/")
            
        # Safely load to CPU first (avoids CUDA initialization errors in workers)
        image_tensor = torch.load(mri_path, map_location='cpu', weights_only=False)
        
        if image_tensor.ndim == 3:
            image_tensor = image_tensor.unsqueeze(0)
            
        # 2. Extract Label (y^S)
        label_tensor = torch.tensor(LABEL_MAP[label_str], dtype=torch.long)
        
        # 3. Load Concept Target & Jacobian Prior (from the new mounted directory)
        c_target = torch.load(os.path.join(self.source_artifacts_dir, f"{sub_id}_c_target.pt"), map_location='cpu', weights_only=False)
        g_jacobian = torch.load(os.path.join(self.source_artifacts_dir, f"{sub_id}_g_jacobian.pt"), map_location='cpu', weights_only=False)
        
        return image_tensor, label_tensor, c_target, g_jacobian

# =====================================================================
# 3. TARGET DOMAIN DATASET (OASIS-1)
# =====================================================================
class TargetDomainDataset(Dataset):
    def __init__(self, csv_path, base_dir):
        super().__init__()
        self.base_dir = base_dir
        
        df = pd.read_csv(csv_path)
        self.data = df[df['Label'].isin(LABEL_MAP.keys())].reset_index(drop=True)
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        sub_id = str(row['Subject_ID'])
        label_str = row['Label']
        
        # [FIX] DYNAMIC TARGET PATH RECONSTRUCTION
        # Completely ignores the old /kaggle/working/ path in the CSV
        mri_path = os.path.join(self.base_dir, "target_oasis", label_str, f"{sub_id}_MRI.pt")
        
        image_tensor = torch.load(mri_path, map_location='cpu', weights_only=False)
        if image_tensor.ndim == 3:
            image_tensor = image_tensor.unsqueeze(0)
            
        label_tensor = torch.tensor(LABEL_MAP[label_str], dtype=torch.long)
        return image_tensor, label_tensor

# =====================================================================
# 4. DATALOADER INITIALIZATION WRAPPER
# =====================================================================
def get_domain_adaptation_dataloaders(
    base_dir="/kaggle/input/notebooks/alejopatio/preprocess-alzheimer/model_ready_data", 
    batch_size=4
):
    source_csv = os.path.join(base_dir, "source_labels.csv")
    target_csv = os.path.join(base_dir, "target_labels.csv")
    source_artifacts_dir = os.path.join(base_dir, "source_adni")
    
    source_dataset = SourceDomainDataset(csv_path=source_csv, source_artifacts_dir=source_artifacts_dir)
    target_dataset = TargetDomainDataset(csv_path=target_csv, base_dir=base_dir)
    
    print(f"Loaded Source Domain (ADNI): {len(source_dataset)} subjects.")
    print(f"Loaded Target Domain (OASIS): {len(target_dataset)} subjects.")
    
    # [FIX] Automatically disables pin_memory if no GPU is found
    use_pin_memory = torch.cuda.is_available()
    
    source_loader = DataLoader(
        source_dataset, batch_size=batch_size, shuffle=True, drop_last=True, 
        num_workers=2, pin_memory=use_pin_memory
    )
    
    target_loader = DataLoader(
        target_dataset, batch_size=batch_size, shuffle=True, drop_last=True, 
        num_workers=2, pin_memory=use_pin_memory
    )
    
    return source_loader, target_loader


In [ ]:
"""
model.py
========
MRI-only, source-target domain adaptation model for Alzheimer diagnosis.

Architecture pipeline (Section D → Q of the M&M):

  X̄  ──► E_θ ──► F          3D hierarchical encoder          (Sec. D)
          │
          ▼
          ROI Tokenizer       masked pooling + projection       (Sec. E)
          │
          ▼
  T  ──► Ψ ──► U             contextual ROI encoder (Transformer) (Sec. F)
                │
                ▼
          Attention Aggregation ──► z                           (Sec. G)
                │
          ┌─────┴───────┐
          ▼             ▼
   Concept Head       Class Head
   c ∈ ℝᴷ           p ∈ Δᶜ                                   (Sec. H, J)
          │
          ▼
   CBM Classifier
   p̃ ∈ Δᶜ                                                    (Sec. J)
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional


# ---------------------------------------------------------------------------
# A.  3D Hierarchical CNN Encoder  (Section D)
# ---------------------------------------------------------------------------

class ResBlock3D(nn.Module):
    """Basic 3D residual block with GroupNorm."""

    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        groups = min(8, out_ch)
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.gn1   = nn.GroupNorm(groups, out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False)
        self.gn2   = nn.GroupNorm(groups, out_ch)

        self.skip = (
            nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.GroupNorm(groups, out_ch),
            )
            if (in_ch != out_ch or stride != 1)
            else nn.Identity()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.relu(self.gn1(self.conv1(x)), inplace=True)
        out = self.gn2(self.conv2(out))
        return F.relu(out + self.skip(x), inplace=True)


class Encoder3D(nn.Module):
    """
    Hierarchical 3D CNN encoder (Section D).

    Input  : X̄ ∈ ℝ^{H×W×D}  (single-channel MRI, add batch + channel dims)
    Output : F ∈ ℝ^{h×w×d×C_f}

    Default channel progression: 1 → 32 → 64 → 128 → C_f
    with stride-2 downsampling at each stage (so h=H/8, w=W/8, d=D/8).
    """

    def __init__(self, C_f: int = 256, base_ch: int = 32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv3d(1, base_ch, kernel_size=7, stride=2, padding=3, bias=False),
            nn.GroupNorm(min(8, base_ch), base_ch),
            nn.ReLU(inplace=True),
        )

        self.layer1 = self._make_layer(base_ch,      base_ch * 2,  stride=2)
        self.layer2 = self._make_layer(base_ch * 2,  base_ch * 4,  stride=2)
        self.layer3 = self._make_layer(base_ch * 4,  C_f,          stride=1)
        self.C_f = C_f

    @staticmethod
    def _make_layer(in_ch: int, out_ch: int, stride: int = 1) -> nn.Sequential:
        return nn.Sequential(
            ResBlock3D(in_ch, out_ch, stride=stride),
            ResBlock3D(out_ch, out_ch),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : (B, 1, H, W, D)
        Returns:
            F : (B, C_f, h, w, d)
        """
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return x                          # (B, C_f, h, w, d)


# ---------------------------------------------------------------------------
# B.  ROI Tokenizer  (Section E)
# ---------------------------------------------------------------------------

class ROITokenizer(nn.Module):
    """
    Converts an encoded feature map F into a token matrix T (Section E).

    For each ROI R_k, computes a masked average-pool of F, then projects it:
        r_{n,k}  = mean_{u ∈ R_k}  F_n(u)          ∈ ℝ^{C_f}
        t_{n,k}  = W_r · r_{n,k} + b_r + e_k       ∈ ℝ^{C_t}

    Args:
        K    : number of ROIs
        C_f  : feature channels from encoder
        C_t  : token dimension
    """

    def __init__(self, K: int, C_f: int, C_t: int):
        super().__init__()
        self.K   = K
        self.C_f = C_f
        self.C_t = C_t

        self.proj    = nn.Linear(C_f, C_t)           # W_r, b_r
        self.roi_emb = nn.Embedding(K, C_t)          # e_k

    def forward(
        self,
        F: torch.Tensor,
        roi_masks: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            F         : (B, C_f, h, w, d)
            roi_masks : (K, h, w, d)  binary float masks, already resampled to
                        feature-map resolution and normalised so each mask sums
                        to 1 over its support (i.e. divide by |R_k| offline).

        Returns:
            T : (B, K, C_t)
        """
        B = F.shape[0]
        # F_flat : (B, C_f, h*w*d)
        F_flat = F.view(B, self.C_f, -1)

        # roi_masks_flat : (K, h*w*d)
        m_flat = roi_masks.view(self.K, -1)           # (K, N_vox)

        # r_{n,k} = F_flat @ m_flat^T  →  (B, K, C_f)
        R = torch.einsum("bcv,kv->bkc", F_flat, m_flat)

        # Project to token space
        T = self.proj(R)                              # (B, K, C_t)

        # Add learnable ROI position embeddings
        k_idx = torch.arange(self.K, device=F.device)
        T = T + self.roi_emb(k_idx).unsqueeze(0)     # broadcast over B

        return T                                      # (B, K, C_t)


# ---------------------------------------------------------------------------
# C.  Contextual ROI Encoder  Ψ  (Section F)
# ---------------------------------------------------------------------------

class ContextualROIEncoder(nn.Module):
    """
    Shallow Transformer encoder over ROI tokens (Section F).

    Models inter-regional dependencies (hippocampus ↔ entorhinal, etc.)
    via multi-head self-attention with residual connections.

    Args:
        C_t      : token dimension
        n_heads  : number of attention heads
        n_layers : number of Transformer layers
        ffn_mult : hidden-dim multiplier for the FFN
        dropout  : dropout probability
    """

    def __init__(
        self,
        C_t: int,
        n_heads: int = 4,
        n_layers: int = 2,
        ffn_mult: int = 4,
        dropout: float = 0.1,
    ):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=C_t,
            nhead=n_heads,
            dim_feedforward=C_t * ffn_mult,
            dropout=dropout,
            batch_first=True,
            norm_first=True,          # pre-LN for stability
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def forward(self, T: torch.Tensor) -> torch.Tensor:
        """
        Args:
            T : (B, K, C_t)
        Returns:
            U : (B, K, C_t)
        """
        return self.transformer(T)    # (B, K, C_t)


# ---------------------------------------------------------------------------
# D.  Attention-based Global Aggregation  (Section G)
# ---------------------------------------------------------------------------

class AttentionAggregator(nn.Module):
    """
    Computes a subject-level embedding z by soft attention over ROI tokens U.

        a_{n,k}   = v^T tanh(W_a u_{n,k} + b_a)
        α_{n,k}   = softmax_k(a_{n,k})
        z_n       = Σ_k α_{n,k} u_{n,k}       ∈ ℝ^{C_t}

    Args:
        C_t : token / embedding dimension
    """

    def __init__(self, C_t: int):
        super().__init__()
        self.W_a = nn.Linear(C_t, C_t, bias=True)
        self.v   = nn.Linear(C_t, 1,   bias=False)

    def forward(self, U: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            U : (B, K, C_t)
        Returns:
            z     : (B, C_t)
            alpha : (B, K)   attention weights (stored for interpretability)
        """
        a     = self.v(torch.tanh(self.W_a(U))).squeeze(-1)   # (B, K)
        alpha = torch.softmax(a, dim=-1)                        # (B, K)
        z     = (alpha.unsqueeze(-1) * U).sum(dim=1)           # (B, C_t)
        return z, alpha


# ---------------------------------------------------------------------------
# E.  Classification Head  (Section H)
# ---------------------------------------------------------------------------

class ClassificationHead(nn.Module):
    """
    Linear classifier on global embedding z (Section H).

        p_n = softmax(W_c z_n + b_c)

    Returns logits (softmax is applied in the loss).
    """

    def __init__(self, C_t: int, n_classes: int):
        super().__init__()
        self.fc = nn.Linear(C_t, n_classes)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """
        Args:
            z      : (B, C_t)
        Returns:
            logits : (B, C)
        """
        return self.fc(z)


# ---------------------------------------------------------------------------
# F.  Concept Bottleneck  (Sections J–K)
# ---------------------------------------------------------------------------

class ConceptBottleneck(nn.Module):
    """
    Per-ROI concept predictor and concept-based classifier (Section J).

    For each ROI k and subject n:
        c_{n,k} = σ(w_k^T u_{n,k} + b_k)     scalar ∈ [0,1]

    Then:
        p̃_n = softmax(W_cbm c_n + b_cbm)

    Args:
        K          : number of ROIs / concepts
        C_t        : token dimension
        n_classes  : number of diagnostic classes
    """

    # def __init__(self, K: int, C_t: int, n_classes: int):
    #     super().__init__()
    #     self.K = K
    #     self.C_t = C_t

    #     self.concept_weights = nn.Parameter(torch.empty(K, C_t))
    #     self.concept_bias = nn.Parameter(torch.zeros(K))
    #     nn.init.xavier_uniform_(self.concept_weights)

    #     self.cbm_head = nn.Linear(K, n_classes)

    # def forward(self, U: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    #     # U: (B, K, C_t)
    #     raw = torch.einsum("bkc,kc->bk", U, self.concept_weights) + self.concept_bias
    #     c = torch.sigmoid(raw)
    #     cbm_logits = self.cbm_head(c)
    #     return c, cbm_logits

    def __init__(self, K: int, C_t: int, n_classes: int, hidden: int = 64, p: float = 0.2):
        super().__init__()
        self.K = K
        self.concept_mlps = nn.ModuleList([
            nn.Sequential(
                nn.Linear(C_t, hidden),
                nn.GELU(),
                nn.Dropout(p),
                nn.Linear(hidden, 1)
            )
            for _ in range(K)
        ])
        self.cbm_head = nn.Linear(K, n_classes)

    def forward(self, U: torch.Tensor):
        c_list = []
        for k in range(self.K):
            ck = self.concept_mlps[k](U[:, k, :])   # (B,1)
            c_list.append(ck)
        raw = torch.cat(c_list, dim=1)              # (B,K)
        c = torch.sigmoid(raw)
        cbm_logits = self.cbm_head(c)
        return c, cbm_logits
    



# ---------------------------------------------------------------------------
# G.  Full Model  (Sections D–Q)
# ---------------------------------------------------------------------------


class AlzheimerDomainAdaptationModel(nn.Module):
    """
    Unified MRI-only model for Alzheimer diagnosis with domain adaptation.

    Forward pass returns a ModelOutput dataclass containing all intermediate
    tensors needed by the compound loss in losses.py.

    Args:
        K          : number of atlas ROIs
        C_f        : encoder output channels
        C_t        : token / embedding dimension
        n_classes  : number of diagnostic classes (C)
        n_heads    : Transformer attention heads
        n_layers   : Transformer layers
        base_ch    : base channel width in the 3D encoder stem
    """

    def __init__(
        self,
        K:          int = 84,
        C_f:        int = 256,
        C_t:        int = 128,
        n_classes:  int = 3,
        n_heads:    int = 4,
        n_layers:   int = 2,
        base_ch:    int = 32,
    ):
        super().__init__()
        self.K         = K
        self.C_f       = C_f
        self.C_t       = C_t
        self.n_classes = n_classes

        # --- modules (one per M&M section) ---
        self.encoder    = Encoder3D(C_f=C_f, base_ch=base_ch)
        self.tokenizer  = ROITokenizer(K=K, C_f=C_f, C_t=C_t)
        self.token_norm = nn.LayerNorm(C_t)
        self.token_mlp = nn.Sequential(
            nn.Linear(C_t, C_t),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(C_t, C_t),
        )
        self.token_dropout = nn.Dropout(0.2)
        self.ctx_enc    = ContextualROIEncoder(C_t=C_t, n_heads=n_heads, n_layers=n_layers)
        self.aggregator = AttentionAggregator(C_t=C_t)
        self.cls_head   = ClassificationHead(C_t=C_t, n_classes=n_classes)
        self.cbm        = ConceptBottleneck(K=K, C_t=C_t, n_classes=n_classes)

    # ------------------------------------------------------------------
    def forward(
        self,
        x: torch.Tensor,
        roi_masks: torch.Tensor,
    ) -> dict:
        """
        Full forward pass.

        Args:
            x         : (B, 1, H, W, D)   preprocessed MRI volume
            roi_masks : (K, h, w, d)       normalised binary ROI masks at
                                           feature-map resolution

        Returns dict with keys:
            F          : (B, C_f, h, w, d)  encoded feature map
            T          : (B, K, C_t)        ROI token matrix
            U          : (B, K, C_t)        contextualised tokens
            z          : (B, C_t)           global embedding
            alpha      : (B, K)             attention weights
            logits     : (B, C)             logits from z (direct classifier)
            c          : (B, K)             concept scores ∈ [0,1]
            cbm_logits : (B, C)             logits from concept bottleneck
        """
        F             = self.encoder(x)                      # (B, C_f, h, w, d)
        T             = self.tokenizer(F, roi_masks)         # (B, K, C_t)
        T             = self.token_norm(T)
        T             = T + self.token_mlp(T)
        T             = self.token_dropout(T)
        U             = self.ctx_enc(T)                     # (B, K, C_t)
        z, alpha      = self.aggregator(U)                   # (B, C_t), (B, K)
        logits        = self.cls_head(z)                     # (B, C)
        c, cbm_logits = self.cbm(U)                          # (B,K), (B,C)

        return {
            "F":          F,
            "T":          T,
            "U":          U,
            "z":          z,
            "alpha":      alpha,
            "logits":     logits,
            "c":          c,
            "cbm_logits": cbm_logits,
        }

    # ------------------------------------------------------------------
    @torch.no_grad()
    def predict(
        self,
        x: torch.Tensor,
        roi_masks: torch.Tensor,
    ) -> dict:
        """
        Inference entry-point (Section P).

        Returns:
            y_hat  : (B,)        predicted class index
            p_tilde: (B, C)      probability from concept bottleneck
            c      : (B, K)      concept vector for explanation
            alpha  : (B, K)      ROI attention weights
        """
        out      = self.forward(x, roi_masks)
        p_tilde  = torch.softmax(out["cbm_logits"], dim=-1)
        y_hat    = p_tilde.argmax(dim=-1)
        return {
            "y_hat":   y_hat,
            "p_tilde": p_tilde,
            "c":       out["c"],
            "alpha":   out["alpha"],
        }

class AlzheimerSupervisedMRIModel(AlzheimerDomainAdaptationModel):
    """
    MRI-only, anatomically structured, supervised model for Alzheimer diagnosis.

    Same architecture as the baseline model, but used in a purely supervised
    formulation without any domain adaptation terms.
    """
    pass

In [ ]:
"""
losses.py
=========
All training objectives from the M&M section (Sections H–N).

Loss map
--------
┌────────────────────────────┬──────────────────────────────────────────────┐
│ Symbol in M&M              │ Class / function here                        │
├────────────────────────────┼──────────────────────────────────────────────┤
│ L_cls                      │ ClassificationLoss           (Sec. H)        │
│ L_proto_align + L_proto_sep│ PrototypeLoss                (Sec. I)        │
│ L_pl                       │ PseudoLabelLoss              (Sec. M)        │
│ L_concept                  │ ConceptSupervisionLoss       (Sec. K)        │
│ L_anat                     │ AnatomicalConsistencyLoss    (Sec. L)        │
│ L_total                    │ TotalLoss                    (Sec. N)        │
└────────────────────────────┴──────────────────────────────────────────────┘

All loss modules are stateless (no internal buffers updated during forward);
prototype accumulators live in the trainer and are passed as arguments.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

class PredictionConsistencyLoss(nn.Module):
    """
    L_cons = KL( p_z || p_c )
    where p_z = softmax(logits_z) and p_c = softmax(logits_c).
    """

    def __init__(self):
        super().__init__()
        self.kl = nn.KLDivLoss(reduction="batchmean")

    def forward(self, logits_z: torch.Tensor, logits_c: torch.Tensor) -> torch.Tensor:
        log_p_z = F.log_softmax(logits_z, dim=-1)
        p_c = F.softmax(logits_c, dim=-1)
        return self.kl(log_p_z, p_c)


# ---------------------------------------------------------------------------
# 1.  Classification Loss  L_cls  (Section H)
# ---------------------------------------------------------------------------

class ClassificationLoss(nn.Module):
    """
    Supervised cross-entropy on labeled source mini-batch.

        L_cls = - Σ_{n ∈ B_S} Σ_c  y_{n,c} log p_{n,c}

    Uses logits + F.cross_entropy for numerical stability.

    Args:
        label_smoothing : float in [0, 1), optional label smoothing
    """

    def __init__(self, label_smoothing: float = 0.0):
        super().__init__()
        self.label_smoothing = label_smoothing

    def forward(
        self,
        logits: torch.Tensor,   # (B_S, C)
        labels: torch.Tensor,   # (B_S,)  integer class indices
    ) -> torch.Tensor:
        """
        Returns:
            scalar loss
        """
        return F.cross_entropy(
            logits, labels,
            label_smoothing=self.label_smoothing,
        )


# ---------------------------------------------------------------------------
# 2.  Prototype Loss  L_proto  (Section I)
# ---------------------------------------------------------------------------

class PrototypeLoss(nn.Module):
    """
    Class-conditional source–target prototype alignment + source separation.

        L_proto_align = Σ_c  ‖ μ_c^S − μ_c^T ‖²₂
        L_proto_sep   = Σ_{c≠c'} max(0, m − ‖ μ_c^S − μ_{c'}^S ‖₂)²
        L_proto       = L_proto_align + λ_sep · L_proto_sep

    Prototypes are computed **inside** this forward pass from the current
    mini-batch embeddings, using:
      - source labels  (hard)
      - target pseudo-labels only where confidence ≥ τ_p (Eq. I)

    Args:
        n_classes  : C
        tau_p      : confidence threshold for pseudo-labels
        margin     : separation margin m  (default 1.0)
        lambda_sep : weight of the separation term (default 0.1)
    """

    def __init__(
        self,
        n_classes: int,
        tau_p:      float = 0.9,
        margin:     float = 1.0,
        lambda_sep: float = 0.1,
    ):
        super().__init__()
        self.n_classes  = n_classes
        self.tau_p      = tau_p
        self.margin     = margin
        self.lambda_sep = lambda_sep

    # ------------------------------------------------------------------
    @staticmethod
    def _class_prototypes(
        z:       torch.Tensor,   # (B, C_t)
        labels:  torch.Tensor,   # (B,)  integer
        n_classes: int,
        eps: float = 1e-8,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Computes per-class mean embeddings.

        Returns:
            protos : (C, C_t)   — prototype per class (zero if class absent)
            valid  : (C,) bool  — True where prototype is non-trivial
        """
        C_t = z.shape[1]
        protos = z.new_zeros(n_classes, C_t)
        counts = z.new_zeros(n_classes)

        for c in range(n_classes):
            mask = labels == c
            if mask.any():
                protos[c] = z[mask].mean(dim=0)
                counts[c] = 1.0

        return protos, counts.bool()

    # ------------------------------------------------------------------
    def forward(
        self,
        z_src:      torch.Tensor,   # (B_S, C_t)  source embeddings
        y_src:      torch.Tensor,   # (B_S,)       source integer labels
        z_tgt:      torch.Tensor,   # (B_T, C_t)  target embeddings
        logits_tgt: torch.Tensor,   # (B_T, C)    target logits (for pseudo-label)
    ) -> tuple[torch.Tensor, dict]:
        """
        Returns:
            loss   : scalar
            info   : dict with 'align', 'sep', 'n_confident' for logging
        """
        # ---------- target pseudo-labels with confidence gate ----------
        probs_tgt = torch.softmax(logits_tgt, dim=-1)     # (B_T, C)
        conf, pseudo_labels = probs_tgt.max(dim=-1)       # (B_T,) each
        confident_mask = conf >= self.tau_p                # (B_T,)

        n_confident = int(confident_mask.sum().item())

        # ---------- source prototypes (always available) ---------------
        mu_src, valid_src = self._class_prototypes(
            z_src, y_src, self.n_classes
        )

        # ---------- alignment loss ------------------------------------
        align_loss = z_src.new_tensor(0.0)

        if n_confident > 0:
            z_conf  = z_tgt[confident_mask]
            pl_conf = pseudo_labels[confident_mask]

            mu_tgt, valid_tgt = self._class_prototypes(
                z_conf, pl_conf, self.n_classes
            )

            # align only for classes present in both source and target batch
            both_valid = valid_src & valid_tgt
            if both_valid.any():
                diff       = mu_src[both_valid] - mu_tgt[both_valid]  # (M, C_t)
                align_loss = (diff ** 2).sum(dim=-1).mean()

        # ---------- separation loss (source side only) ----------------
        sep_loss = z_src.new_tensor(0.0)
        valid_idx = valid_src.nonzero(as_tuple=True)[0]

        if len(valid_idx) >= 2:
            valid_protos = mu_src[valid_idx]                # (M, C_t)
            M = valid_protos.shape[0]
            sep_terms = []
            for i in range(M):
                for j in range(i + 1, M):
                    d = torch.norm(valid_protos[i] - valid_protos[j], p=2)
                    margin_violation = F.relu(self.margin - d) ** 2
                    sep_terms.append(margin_violation)

            if sep_terms:
                sep_loss = torch.stack(sep_terms).mean()

        # ---------- combined ------------------------------------------
        loss = align_loss + self.lambda_sep * sep_loss

        info = {
            "proto_align":   align_loss.item(),
            "proto_sep":     sep_loss.item(),
            "n_confident_T": n_confident,
        }
        return loss, info


# ---------------------------------------------------------------------------
# 3.  Pseudo-Label Self-Training Loss  L_pl  (Section M)
# ---------------------------------------------------------------------------

class PseudoLabelLoss(nn.Module):
    """
    Cross-entropy self-training on confident target samples.

        L_pl = - Σ_{m ∈ B^T_{τ_p}} Σ_c  ŷ_{m,c} log p_{m,c}

    where ŷ_{m,c} is the one-hot vector of the argmax pseudo-label.

    Args:
        tau_p : confidence threshold (same value as in PrototypeLoss)
    """

    def __init__(self, tau_p: float = 0.9):
        super().__init__()
        self.tau_p = tau_p

    def forward(
        self,
        logits_tgt: torch.Tensor,   # (B_T, C)
    ) -> tuple[torch.Tensor, int]:
        """
        Returns:
            loss        : scalar (0 if no confident sample)
            n_confident : int, for logging
        """
        probs      = torch.softmax(logits_tgt, dim=-1)
        conf, pseudo = probs.max(dim=-1)
        mask         = conf >= self.tau_p

        n_confident = int(mask.sum().item())

        if n_confident == 0:
            return logits_tgt.new_tensor(0.0), 0

        loss = F.cross_entropy(logits_tgt[mask], pseudo[mask])
        return loss, n_confident


# ---------------------------------------------------------------------------
# 4.  Concept Supervision Loss  L_concept  (Section K)
# ---------------------------------------------------------------------------

class ConceptSupervisionLoss(nn.Module):
    """
    Forces each concept score to match a pre-computed anatomical target.

        L_concept = Σ_{n ∈ B_S} Σ_k  | c_{n,k} − c̃_{n,k} |²

    where c̃_{n,k} is the normalised structural biomarker target (e.g.
    regional volume fraction, cortical thickness proxy).

    Targets are expected to be pre-normalised into [0, 1] to match the
    sigmoid activation of the concept head.
    """

    def __init__(self):
        super().__init__()

    def forward(
        self,
        c: torch.Tensor,              # (B_S, K)  predicted concepts
        c_target: torch.Tensor,       # (B_S, K)  normalised anatomical targets
    ) -> torch.Tensor:
        """
        Returns:
            scalar MSE loss averaged over subjects and ROIs
        """
        return F.mse_loss(c, c_target)


# ---------------------------------------------------------------------------
# 5.  Anatomical Consistency Loss  L_anat  (Section L)
# ---------------------------------------------------------------------------

class AnatomicalConsistencyLoss(nn.Module):
    """
    Encourages concept scores to remain compatible with the Jacobian-based
    deformation summary (Section L).

        L_anat = Σ_n Σ_k  ω_k · | c_{n,k} − ḡ_{n,k} |²

    where ḡ_{n,k} = normalised -log(J) regional mean, and ω_k is an
    ROI-specific weight (default: uniform).

    Note: This loss acts as a plausibility regulariser, NOT a supervision
    signal claiming Jacobian determinants are ground-truth pathology.

    Args:
        K       : number of ROIs
        roi_weights : optional (K,) tensor of ω_k; uniform if None
    """

    def __init__(
        self,
        K: int,
        roi_weights: Optional[torch.Tensor] = None,
    ):
        super().__init__()
        if roi_weights is None:
            roi_weights = torch.ones(K) / K
        # Register as buffer so it moves with .to(device)
        self.register_buffer("roi_weights", roi_weights)

    def forward(
        self,
        c: torch.Tensor,           # (B, K)  predicted concept scores
        g_bar: torch.Tensor,       # (B, K)  normalised deformation summaries
    ) -> torch.Tensor:
        """
        Returns:
            scalar weighted MSE loss
        """
        if g_bar.device != c.device:
            g_bar = g_bar.to(device=c.device, dtype=c.dtype, non_blocking=True)

        roi_weights = self.roi_weights
        if roi_weights.device != c.device or roi_weights.dtype != c.dtype:
            roi_weights = roi_weights.to(device=c.device, dtype=c.dtype, non_blocking=True)

        residuals = (c - g_bar) ** 2                      # (B, K)
        weighted  = residuals * roi_weights.unsqueeze(0) # (B, K)
        return weighted.mean()


# ---------------------------------------------------------------------------
# 6.  Combined Total Loss  L_total  (Section N)
# ---------------------------------------------------------------------------

# class SupervisedTotalLoss(nn.Module):
#     """
#     Supervised objective matching Sections I, J, K, L of the adjusted M&M.

#     Warm-up:
#         L_warm = lambda_z * L_cls^z

#     Full:
#         L_total = lambda_z    * L_cls^z
#                 + lambda_c    * L_cls^c
#                 + lambda_cons * L_cons
#                 + lambda_cbm  * L_concept
#                 + lambda_anat * L_anat
#     """

#     def __init__(
#         self,
#         n_classes: int,
#         K: int,
#         roi_weights: Optional[torch.Tensor] = None,
#         lambda_z: float = 1.0,
#         lambda_c: float = 1.0,
#         lambda_cons: float = 0.1,
#         lambda_cbm: float = 0.5,
#         lambda_anat: float = 0.2,
#         label_smoothing: float = 0.1,
#     ):
#         super().__init__()

#         self.lambda_z = float(lambda_z)
#         self.lambda_c = float(lambda_c)
#         self.lambda_cons = float(lambda_cons)
#         self.lambda_cbm = float(lambda_cbm)
#         self.lambda_anat = float(lambda_anat)

#         self.loss_cls_z = ClassificationLoss(label_smoothing=label_smoothing)
#         self.loss_cls_c = ClassificationLoss(label_smoothing=label_smoothing)
#         self.loss_cons = PredictionConsistencyLoss()
#         self.loss_concept = ConceptSupervisionLoss()
#         self.loss_anat = AnatomicalConsistencyLoss(K=K, roi_weights=roi_weights)

#     def forward_warm(
#         self,
#         logits_z: torch.Tensor,
#         labels: torch.Tensor,
#     ):
#         l_cls_z = self.loss_cls_z(logits_z, labels)

#         total = self.lambda_z * l_cls_z
#         info = {
#             "L_total": float(total.item()),
#             "L_cls_z": float(l_cls_z.item()),
#         }
#         return total, info

#     def forward_full(
#         self,
#         logits_z: torch.Tensor,
#         logits_c: torch.Tensor,
#         labels: torch.Tensor,
#         c: torch.Tensor,
#         c_target: torch.Tensor,
#         g_bar: torch.Tensor,
#     ):
#         l_cls_z = self.loss_cls_z(logits_z, labels)
#         l_cls_c = self.loss_cls_c(logits_c, labels)
#         l_cons = self.loss_cons(logits_z, logits_c)
#         l_concept = self.loss_concept(c, c_target)
#         l_anat = self.loss_anat(c, g_bar)

#         total = (
#             self.lambda_z    * l_cls_z
#             + self.lambda_c    * l_cls_c
#             + self.lambda_cons * l_cons
#             + self.lambda_cbm  * l_concept
#             + self.lambda_anat * l_anat
#         )

#         info = {
#             "L_total": float(total.item()),
#             "L_cls_z": float(l_cls_z.item()),
#             "L_cls_c": float(l_cls_c.item()),
#             "L_cons": float(l_cons.item()),
#             "L_concept": float(l_concept.item()),
#             "L_anat": float(l_anat.item()),
#         }
#         return total, info

from typing import Optional
import torch
import torch.nn as nn

class SupervisedTotalLoss(nn.Module):
    """
    Supervised objective with concept-first warm-up.

    Warm-up:
        L_warm = warm_lambda_c    * lambda_c    * L_cls^c
               + warm_lambda_cbm  * lambda_cbm  * L_concept
               + warm_lambda_anat * lambda_anat * L_anat
               + warm_lambda_z    * lambda_z    * L_cls^z

    Full:
        L_total = lambda_z    * L_cls^z
                + lambda_c    * L_cls^c
                + lambda_cons * L_cons
                + lambda_cbm  * L_concept
                + lambda_anat * L_anat
    """

    def __init__(
        self,
        n_classes: int,
        K: int,
        roi_weights: Optional[torch.Tensor] = None,
        lambda_z: float = 1.0,
        lambda_c: float = 1.0,
        lambda_cons: float = 0.1,
        lambda_cbm: float = 0.5,
        lambda_anat: float = 0.2,
        label_smoothing: float = 0.1,
        # warm-up coefficients
        warm_lambda_z: float = 0.1,   # epsilon * lambda_z
        warm_lambda_c: float = 1.0,
        warm_lambda_cbm: float = 1.0,
        warm_lambda_anat: float = 1.0,
    ):
        super().__init__()

        self.lambda_z = float(lambda_z)
        self.lambda_c = float(lambda_c)
        self.lambda_cons = float(lambda_cons)
        self.lambda_cbm = float(lambda_cbm)
        self.lambda_anat = float(lambda_anat)

        self.warm_lambda_z = float(warm_lambda_z)
        self.warm_lambda_c = float(warm_lambda_c)
        self.warm_lambda_cbm = float(warm_lambda_cbm)
        self.warm_lambda_anat = float(warm_lambda_anat)

        self.loss_cls_z = ClassificationLoss(label_smoothing=label_smoothing)
        self.loss_cls_c = ClassificationLoss(label_smoothing=label_smoothing)
        self.loss_cons = PredictionConsistencyLoss()
        self.loss_concept = ConceptSupervisionLoss()
        self.loss_anat = AnatomicalConsistencyLoss(K=K, roi_weights=roi_weights)

    def forward_warm(
        self,
        logits_z: torch.Tensor,
        logits_c: torch.Tensor,
        labels: torch.Tensor,
        c: torch.Tensor,
        c_target: torch.Tensor,
        g_bar: torch.Tensor,
    ):
        l_cls_z = self.loss_cls_z(logits_z, labels)
        l_cls_c = self.loss_cls_c(logits_c, labels)
        l_concept = self.loss_concept(c, c_target)
        l_anat = self.loss_anat(c, g_bar)

        total = (
            self.warm_lambda_c    * self.lambda_c    * l_cls_c
            + self.warm_lambda_cbm  * self.lambda_cbm  * l_concept
            + self.warm_lambda_anat * self.lambda_anat * l_anat
            + self.warm_lambda_z    * self.lambda_z    * l_cls_z
        )

        info = {
            "L_total": float(total.item()),
            "L_cls_z": float(l_cls_z.item()),
            "L_cls_c": float(l_cls_c.item()),
            "L_concept": float(l_concept.item()),
            "L_anat": float(l_anat.item()),
        }
        return total, info

    def forward_full(
        self,
        logits_z: torch.Tensor,
        logits_c: torch.Tensor,
        labels: torch.Tensor,
        c: torch.Tensor,
        c_target: torch.Tensor,
        g_bar: torch.Tensor,
    ):
        l_cls_z = self.loss_cls_z(logits_z, labels)
        l_cls_c = self.loss_cls_c(logits_c, labels)
        l_cons = self.loss_cons(logits_z, logits_c)
        l_concept = self.loss_concept(c, c_target)
        l_anat = self.loss_anat(c, g_bar)

        total = (
            self.lambda_z    * l_cls_z
            + self.lambda_c    * l_cls_c
            + self.lambda_cons * l_cons
            + self.lambda_cbm  * l_concept
            + self.lambda_anat * l_anat
        )

        info = {
            "L_total": float(total.item()),
            "L_cls_z": float(l_cls_z.item()),
            "L_cls_c": float(l_cls_c.item()),
            "L_cons": float(l_cons.item()),
            "L_concept": float(l_concept.item()),
            "L_anat": float(l_anat.item()),
        }
        return total, info


In [ ]:

"""
preprocessing.py
=================
Section B of the Materials and Methods.

This module standardizes a structural MRI volume into the common tensor space
expected by the model:
    X -> X_tilde -> X_bar

It is intentionally conservative. If the Kaggle derivatives are already
preprocessed, this module behaves as a consistency operator plus resampling
and intensity normalization.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Sequence, Tuple, Union

import nibabel as nib
import numpy as np
import torch
import torch.nn.functional as F


ArrayLikePath = Union[str, Path]


@dataclass
class PreprocessConfig:
    target_shape: Tuple[int, int, int] = (128, 128, 128)
    eps: float = 1e-6
    brain_mask_threshold: float = 0.0
    clip_percentiles: Tuple[float, float] = (0.5, 99.5)
    enforce_canonical: bool = True


def load_nifti_canonical(path: ArrayLikePath, enforce_canonical: bool = True) -> tuple[np.ndarray, np.ndarray]:
    img = nib.load(str(path))
    if enforce_canonical:
        img = nib.as_closest_canonical(img)
    vol = img.get_fdata(dtype=np.float32)
    if vol.ndim == 4:
        vol = vol[..., 0]
    vol = np.nan_to_num(vol, nan=0.0, posinf=0.0, neginf=0.0)
    return vol.astype(np.float32), img.affine.astype(np.float32)


def make_brain_mask(volume: np.ndarray, threshold: float = 0.0) -> np.ndarray:
    mask = np.isfinite(volume) & (volume > threshold)
    return mask.astype(np.float32)


def robust_clip_inside_mask(
    volume: np.ndarray,
    mask: np.ndarray,
    clip_percentiles: Tuple[float, float] = (0.5, 99.5),
) -> np.ndarray:
    vox = volume[mask > 0]
    if vox.size == 0:
        return volume.astype(np.float32)
    lo, hi = np.percentile(vox, clip_percentiles)
    return np.clip(volume, lo, hi).astype(np.float32)


def zscore_inside_mask(volume: np.ndarray, mask: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    vox = volume[mask > 0]
    if vox.size == 0:
        return volume.astype(np.float32)
    mu = float(vox.mean())
    sigma = float(vox.std())
    out = (volume - mu) / (sigma + eps)
    out[mask <= 0] = 0.0
    return out.astype(np.float32)


def resize_volume_torch(volume: np.ndarray, target_shape: Sequence[int]) -> np.ndarray:
    x = torch.from_numpy(volume).unsqueeze(0).unsqueeze(0)  # (1,1,H,W,D)
    x = F.interpolate(x, size=tuple(target_shape), mode="trilinear", align_corners=False)
    return x.squeeze(0).squeeze(0).cpu().numpy().astype(np.float32)


def preprocess_volume_array(volume: np.ndarray, cfg: PreprocessConfig) -> tuple[np.ndarray, np.ndarray]:
    mask = make_brain_mask(volume, threshold=cfg.brain_mask_threshold)
    volume = robust_clip_inside_mask(volume, mask, cfg.clip_percentiles)
    volume = resize_volume_torch(volume, cfg.target_shape)
    mask = resize_volume_torch(mask.astype(np.float32), cfg.target_shape)
    mask = (mask > 0.5).astype(np.float32)
    volume = zscore_inside_mask(volume, mask, eps=cfg.eps)
    return volume.astype(np.float32), mask.astype(np.float32)


def preprocess_nifti(
    path: ArrayLikePath,
    cfg: Optional[PreprocessConfig] = None,
    save_pt_path: Optional[ArrayLikePath] = None,
) -> dict:
    cfg = cfg or PreprocessConfig()
    volume, affine = load_nifti_canonical(path, enforce_canonical=cfg.enforce_canonical)
    x_bar, brain_mask = preprocess_volume_array(volume, cfg)

    tensor = torch.from_numpy(x_bar).unsqueeze(0)      # (1,H,W,D)
    mask_t = torch.from_numpy(brain_mask).unsqueeze(0) # (1,H,W,D)

    out = {
        "x": tensor.to(torch.float32),
        "brain_mask": mask_t.to(torch.float32),
        "affine": affine,
        "source_path": str(path),
    }

    if save_pt_path is not None:
        save_pt_path = Path(save_pt_path)
        save_pt_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(out, save_pt_path)

    return out


def validate_tensor_contract(sample: dict, expected_shape: Sequence[int] = (1, 128, 128, 128)) -> None:
    if "x" not in sample:
        raise KeyError("Missing key 'x' in preprocessed sample.")
    x = sample["x"]
    if not torch.is_tensor(x):
        raise TypeError("'x' must be a torch.Tensor.")
    if tuple(x.shape) != tuple(expected_shape):
        raise ValueError(f"Expected x shape {tuple(expected_shape)}, got {tuple(x.shape)}.")


In [ ]:

"""
dataset_contract.py
===================
Minimal dataset contract for the already-created dataloaders.

Required keys for Stage I source batches:
    x        : (B,1,H,W,D)
    y        : (B,)
    c_target : (B,K)
    g_bar    : (B,K)

Required keys for Stage II target batches:
    x        : (B,1,H,W,D)
    g_bar    : (B,K)

Optional metadata:
    subject_id, source_path, label_name
"""

from __future__ import annotations

from typing import Mapping

import torch


def validate_source_batch(batch: Mapping, K: int) -> None:
    for key in ["x", "y", "c_target", "g_bar"]:
        if key not in batch:
            raise KeyError(f"Missing key '{key}' in source batch.")
    if batch["x"].ndim != 5:
        raise ValueError(f"Expected x shape (B,1,H,W,D), got {tuple(batch['x'].shape)}.")
    if batch["c_target"].shape[-1] != K or batch["g_bar"].shape[-1] != K:
        raise ValueError("Concept targets or Jacobian summaries do not match K.")

def validate_target_batch(batch: Mapping, K: int) -> None:
    for key in ["x", "g_bar"]:
        if key not in batch:
            raise KeyError(f"Missing key '{key}' in target batch.")
    if batch["x"].ndim != 5:
        raise ValueError(f"Expected x shape (B,1,H,W,D), got {tuple(batch['x'].shape)}.")
    if batch["g_bar"].shape[-1] != K:
        raise ValueError("Jacobian summaries do not match K.")

In [ ]:
# ============================================================
# CEREBRA (.mnc) -> atlas discreto NIfTI listo para usar
# usando nibabel en lugar de SimpleITK
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import nibabel as nib
from nibabel.processing import resample_from_to

RAW_CEREBRA_PATH = "/kaggle/input/datasets/alejopatio/cerebra/mni_icbm152_CerebrA_tal_nlin_sym_09c.mnc"
OUT_DIR = "/kaggle/working/cerebra_prepared"
os.makedirs(OUT_DIR, exist_ok=True)

DISCRETE_ATLAS_PATH = os.path.join(OUT_DIR, "CerebrA_discrete_ready.nii.gz")
RESAMPLED_ATLAS_PATH = os.path.join(OUT_DIR, "CerebrA_discrete_resampled_to_reference.nii.gz")
LUT_CSV_PATH = os.path.join(OUT_DIR, "CerebrA_label_lut.csv")
META_JSON_PATH = os.path.join(OUT_DIR, "CerebrA_prepare_meta.json")


# ------------------------------------------------------------
# 1) UTILIDADES
# ------------------------------------------------------------
def read_image_any_nib(path: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"La ruta no existe: {path}")
    if not os.path.isfile(path):
        raise FileNotFoundError(f"La ruta existe pero no es un archivo: {path}")
    if os.path.getsize(path) == 0:
        raise RuntimeError(f"El archivo está vacío: {path}")

    try:
        img = nib.load(path)
        return img
    except Exception as e:
        raise RuntimeError(
            f"Nibabel no pudo abrir el archivo:\n{path}\n\n"
            f"Error original:\n{e}"
        )


def image_info_nib(img, path=""):
    hdr = img.header
    zooms = hdr.get_zooms()[:3] if len(hdr.get_zooms()) >= 3 else hdr.get_zooms()
    return {
        "path": path,
        "shape_xyz": tuple(int(v) for v in img.shape[:3]),
        "zooms_xyz": tuple(float(v) for v in zooms),
        "dtype": str(img.get_data_dtype()),
        "affine": np.asarray(img.affine).tolist(),
        "class": img.__class__.__name__,
    }


def unique_summary_nib(img, max_show=25):
    arr = np.asarray(img.dataobj)
    uniq = np.unique(arr)
    return {
        "shape": tuple(int(v) for v in arr.shape),
        "dtype": str(arr.dtype),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "n_unique": int(len(uniq)),
        "unique_head": uniq[:max_show].tolist(),
        "looks_integer": bool(np.allclose(uniq, np.round(uniq))),
    }


# ------------------------------------------------------------
# 2) PREPARAR CEREBRA COMO ATLAS DISCRETO
# ------------------------------------------------------------
def prepare_cerebra_discrete_atlas_nib(
    raw_atlas_path: str,
    out_atlas_path: str,
    lut_csv_path: str,
    meta_json_path: str,
    background_label: int = 0,
):
    img = read_image_any_nib(raw_atlas_path)
    img = nib.as_closest_canonical(img)

    info_before = image_info_nib(img, raw_atlas_path)
    uniq_before = unique_summary_nib(img)

    arr = np.asarray(img.dataobj)

    if arr.ndim != 3:
        raise ValueError(f"Se esperaba un atlas 3D, pero llegó shape {arr.shape}")

    # Discretización conservadora
    arr_disc = np.rint(arr).astype(np.int32)

    # Reindexar labels a 0,1,2,...,K
    unique_vals = np.unique(arr_disc)
    unique_vals = [int(v) for v in unique_vals if np.isfinite(v)]

    if background_label not in unique_vals:
        unique_vals = [background_label] + unique_vals

    roi_vals = sorted([v for v in unique_vals if v != background_label])

    old_to_new = {background_label: 0}
    for new_id, old_id in enumerate(roi_vals, start=1):
        old_to_new[old_id] = new_id

    remapped = np.zeros_like(arr_disc, dtype=np.int16)
    for old_id, new_id in old_to_new.items():
        remapped[arr_disc == old_id] = new_id

    out_img = nib.Nifti1Image(remapped, img.affine)
    out_img = nib.as_closest_canonical(out_img)
    nib.save(out_img, out_atlas_path)

    lut_df = pd.DataFrame({
        "old_label": list(old_to_new.keys()),
        "new_label": list(old_to_new.values()),
        "is_background": [int(k == background_label) for k in old_to_new.keys()]
    }).sort_values("new_label").reset_index(drop=True)
    lut_df.to_csv(lut_csv_path, index=False)

    info_after = image_info_nib(out_img, out_atlas_path)
    uniq_after = unique_summary_nib(out_img)

    meta = {
        "raw_atlas_path": raw_atlas_path,
        "discrete_atlas_path": out_atlas_path,
        "lut_csv_path": lut_csv_path,
        "background_label_old": int(background_label),
        "n_rois_excluding_background": int(lut_df["new_label"].max()),
        "image_info_before": info_before,
        "image_info_after": info_after,
        "unique_summary_before": uniq_before,
        "unique_summary_after": uniq_after,
    }

    with open(meta_json_path, "w") as f:
        json.dump(meta, f, indent=2)

    print("\n--- ATLAS ORIGINAL ---")
    print(info_before)
    print("\nResumen de labels originales:")
    print(uniq_before)

    print("\n--- ATLAS DISCRETO ---")
    print(info_after)
    print("\nResumen de labels discretos:")
    print(uniq_after)

    print(f"\n[OK] Atlas discreto guardado en: {out_atlas_path}")
    print(f"[OK] LUT guardada en: {lut_csv_path}")
    print(f"[OK] Número de ROIs (sin fondo): {meta['n_rois_excluding_background']}")

    return meta

# ============================================================
# VERIFICACIÓN GEOMÉTRICA CON MRI DE REFERENCIA
# ============================================================

import glob
import numpy as np
import nibabel as nib
from nibabel.processing import resample_from_to

def find_first_oasis_nifti(search_root="/kaggle/input"):
    candidates = sorted(
        glob.glob(os.path.join(search_root, "**", "*.nii"), recursive=True) +
        glob.glob(os.path.join(search_root, "**", "*.nii.gz"), recursive=True)
    )
    return candidates[0] if candidates else None


def compare_geometry_nib(img_a, img_b, atol_affine=1e-3, atol_zooms=1e-4):
    zooms_a = np.array(img_a.header.get_zooms()[:3], dtype=float)
    zooms_b = np.array(img_b.header.get_zooms()[:3], dtype=float)

    report = {
        "shape_a": tuple(int(v) for v in img_a.shape[:3]),
        "shape_b": tuple(int(v) for v in img_b.shape[:3]),
        "same_shape": bool(tuple(img_a.shape[:3]) == tuple(img_b.shape[:3])),
        "zooms_a": tuple(float(v) for v in zooms_a),
        "zooms_b": tuple(float(v) for v in zooms_b),
        "same_zooms": bool(np.allclose(zooms_a, zooms_b, atol=atol_zooms)),
        "same_affine": bool(np.allclose(img_a.affine, img_b.affine, atol=atol_affine)),
    }
    report["same_grid_exact"] = bool(
        report["same_shape"] and report["same_zooms"] and report["same_affine"]
    )
    return report


def resample_label_atlas_to_reference_nib(atlas_path, reference_mri_path, out_path):
    atlas_img = nib.load(atlas_path)
    ref_img = nib.load(reference_mri_path)

    # order=0 -> nearest neighbor para preservar labels
    atlas_resampled = resample_from_to(atlas_img, ref_img, order=0)
    atlas_resampled = nib.Nifti1Image(
        np.asarray(atlas_resampled.dataobj).astype(np.int16),
        atlas_resampled.affine
    )
    nib.save(atlas_resampled, out_path)
    return out_path

ATLAS_PATH = '/kaggle/input/datasets/alejopatio/cerebra/mni_icbm152_CerebrA_tal_nlin_sym_09c.mnc'

oasis_ref_path = find_first_oasis_nifti("/kaggle/input")

if oasis_ref_path is not None:
    print("MRI de referencia encontrado:")
    print(oasis_ref_path)

    atlas_img = nib.load(ATLAS_PATH)
    mri_img = nib.load(oasis_ref_path)

    report = compare_geometry_nib(atlas_img, mri_img)

    print("\n--- COMPARACIÓN GEOMÉTRICA ---")
    for k, v in report.items():
        print(f"{k}: {v}")

    if not report["same_grid_exact"]:
        print("\n[WARN] Atlas y MRI no comparten el mismo grid exacto.")
        print("       Voy a remuestrear el atlas al grid del MRI.")
        ATLAS_PATH = resample_label_atlas_to_reference_nib(
            atlas_path=ATLAS_PATH,
            reference_mri_path=oasis_ref_path,
            out_path=RESAMPLED_ATLAS_PATH
        )
        print("\nNuevo ATLAS_PATH:")
        print(ATLAS_PATH)
    else:
        print("\n[OK] Atlas y MRI comparten el mismo grid exacto.")
else:
    print("\n[WARN] No encontré un MRI NIfTI para verificación.")
    print("       Si solo tienes .pt, no puedes verificar affine/spacing retrospectivamente.")


meta = prepare_cerebra_discrete_atlas_nib(
    raw_atlas_path=RAW_CEREBRA_PATH,
    out_atlas_path=DISCRETE_ATLAS_PATH,
    lut_csv_path=LUT_CSV_PATH,
    meta_json_path=META_JSON_PATH,
    background_label=0,
)

ATLAS_PATH = DISCRETE_ATLAS_PATH
print("\nATLAS_PATH final:")
print(ATLAS_PATH)

In [ ]:
"""
trainer.py
==========
Section O of the Materials and Methods.

Two-stage optimization:
  - Stage I : source pretraining
  - Stage II: source-target adaptation

Design choice:
  The concept head p_tilde is used as the operative classifier during
  training and inference. This is the cleaner interpretation-preserving
  option described in Section J of the M&M.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Optional

import torch
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score,
)

# from atlas_utils import AtlasROIManager


@dataclass
class SupervisedTrainConfig:
    n_epochs_warm: int = 20
    n_epochs_full: int = 30
    lr: float = 3e-4
    weight_decay: float = 1e-4
    grad_clip_norm: float = 5.0
    device: str = "cpu"
    log_every: int = 10
    use_amp: bool = False


def _move_batch(batch: dict, device: torch.device) -> dict:
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            out[k] = v.to(device, non_blocking=True)
        else:
            out[k] = v
    return out


def _require_keys(batch: dict, keys: Iterable[str]) -> None:
    missing = [k for k in keys if k not in batch]
    if missing:
        raise KeyError(f"Batch is missing required keys: {missing}")


def _safe_macro_ovr_auc(y_true: np.ndarray, y_prob: np.ndarray, n_classes: int) -> float:
    """
    AUC macro one-vs-rest robusta frente a clases ausentes en y_true.
    Si una clase no tiene positivos o no tiene negativos, se omite del promedio.
    """
    aucs = []
    for c in range(n_classes):
        y_bin = (y_true == c).astype(np.int32)
        if y_bin.min() == y_bin.max():
            # no hay positivos o no hay negativos
            continue
        try:
            auc_c = roc_auc_score(y_bin, y_prob[:, c])
            aucs.append(float(auc_c))
        except Exception:
            continue

    if len(aucs) == 0:
        return float("nan")
    return float(np.mean(aucs))


def _classification_metrics_from_outputs(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_prob: np.ndarray,
    n_classes: int,
) -> dict:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "auc_macro_ovr": _safe_macro_ovr_auc(y_true, y_prob, n_classes=n_classes),
    }

@dataclass
class SupervisedTrainConfig:
    n_epochs_warm: int = 20
    n_epochs_full: int = 30
    lr: float = 3e-4
    weight_decay: float = 1e-4
    grad_clip_norm: float = 5.0
    num_workers: int = 0
    device: str = "cpu"
    log_every: int = 10
    use_amp: bool = False


class SupervisedMRITrainer:
    def __init__(
        self,
        model,
        loss_fn,
        atlas_mgr,
        input_shape=(128, 128, 128),
        cfg: Optional[SupervisedTrainConfig] = None,
        optimizer=None,
    ):
        self.model = model
        self.loss_fn = loss_fn
        self.atlas_mgr = atlas_mgr
        self.cfg = cfg or SupervisedTrainConfig()

        self.device = torch.device(self.cfg.device)
        self.model = self.model.to(self.device)
        self.loss_fn = self.loss_fn.to(self.device)

        if optimizer is None:
            self.optimizer = torch.optim.AdamW(
                self.model.parameters(),
                lr=self.cfg.lr,
                weight_decay=self.cfg.weight_decay,
            )
        else:
            self.optimizer = optimizer

        self.scaler = torch.amp.GradScaler(
            "cuda",
            enabled=self.cfg.use_amp and self.device.type == "cuda"
        )

        self.roi_masks = self._build_feature_space_roi_masks(input_shape)

    def _build_feature_space_roi_masks(self, input_shape):
        with torch.no_grad():
            dummy = torch.zeros(1, 1, *input_shape, device=self.device)
            feat = self.model.encoder(dummy)
            feature_shape = feat.shape[-3:]

        return self.atlas_mgr.get_masks(
            target_shape=feature_shape,
            normalize=True,
            device=self.device,
            dtype=torch.float32,
        )

    def _forward(self, x):
        return self.model(x, self.roi_masks)

    # @torch.no_grad()
    # def evaluate(self, loader, prefix="", stage="full"):
    #     """
    #     stage:
    #         - 'warm' : evalúa solo L_cls^z
    #         - 'full' : evalúa L_cls^z, L_cls^c, L_cons, L_concept, L_anat y L_total
    #     """
    #     self.model.eval()
    
    #     loss_sums = {}
    #     n_batches = 0
    
    #     y_true = []
    #     y_pred = []
    #     y_prob = []
    
    #     for batch in loader:
    #         batch = _move_batch(batch, self.device)
    #         out = self._forward(batch["x"])
    
    #         # ---------------------------------------------------------
    #         # métricas de clasificación: usamos la cabeza conceptual
    #         # ---------------------------------------------------------
    #         probs = torch.softmax(out["cbm_logits"], dim=-1)
    #         pred = probs.argmax(dim=-1)
    
    #         y_true.append(batch["y"].detach().cpu())
    #         y_pred.append(pred.detach().cpu())
    #         y_prob.append(probs.detach().cpu())
    
    #         # ---------------------------------------------------------
    #         # pérdidas
    #         # ---------------------------------------------------------
    #         if stage == "warm":
    #             loss, info = self.loss_fn.forward_warm(
    #                 logits_z=out["logits"],
    #                 labels=batch["y"],
    #             )
    #         elif stage == "full":
    #             _require_keys(batch, ["c_target", "g_bar"])
    #             loss, info = self.loss_fn.forward_full(
    #                 logits_z=out["logits"],
    #                 logits_c=out["cbm_logits"],
    #                 labels=batch["y"],
    #                 c=out["c"],
    #                 c_target=batch["c_target"],
    #                 g_bar=batch["g_bar"],
    #             )
    #         else:
    #             raise ValueError(f"stage must be 'warm' or 'full', got {stage!r}")
    
    #         # acumular componentes del loss
    #         for k, v in info.items():
    #             loss_sums[k] = loss_sums.get(k, 0.0) + float(v)
    
    #         n_batches += 1

    #     # -------------------------------------------------------------
    #     # promedio de losses
    #     # -------------------------------------------------------------
    #     mean_losses = {k: v / max(n_batches, 1) for k, v in loss_sums.items()}
    
    #     # -------------------------------------------------------------
    #     # métricas de clasificación
    #     # -------------------------------------------------------------
    #     y_true = torch.cat(y_true).numpy()
    #     y_pred = torch.cat(y_pred).numpy()
    #     y_prob = torch.cat(y_prob).numpy()
    
    #     metrics = _classification_metrics_from_outputs(
    #         y_true=y_true,
    #         y_pred=y_pred,
    #         y_prob=y_prob,
    #         n_classes=self.model.n_classes if hasattr(self.model, "n_classes") else y_prob.shape[1],
    #     )
    
    #     out_dict = {**mean_losses, **metrics}
    
    #     if prefix:
    #         out_dict = {f"{prefix}_{k}": v for k, v in out_dict.items()}
    
    #     return out_dict

    @torch.no_grad()
    def evaluate(self, loader, prefix="", stage="full"):
        """
        stage:
            - 'warm' : evalúa el warm-up concept-first y usa la cabeza conceptual
            - 'full' : evalúa la pérdida completa y usa la cabeza conceptual
        """
        self.model.eval()
    
        loss_sums = {}
        n_batches = 0
    
        y_true = []
        y_pred = []
        y_prob = []
    
        for batch in loader:
            batch = _move_batch(batch, self.device)
            out = self._forward(batch["x"])
    
            if stage == "warm":
                eval_logits = out["cbm_logits"]
            elif stage == "full":
                eval_logits = out["cbm_logits"]
            else:
                raise ValueError(f"stage must be 'warm' or 'full', got {stage!r}")
    
            probs = torch.softmax(eval_logits, dim=-1)
            pred = probs.argmax(dim=-1)

            y_true.append(batch["y"].detach().cpu())
            y_pred.append(pred.detach().cpu())
            y_prob.append(probs.detach().cpu())
    
            if stage == "warm":
                _require_keys(batch, ["c_target", "g_bar"])
                loss, info = self.loss_fn.forward_warm(
                    logits_z=out["logits"],
                    logits_c=out["cbm_logits"],
                    labels=batch["y"],
                    c=out["c"],
                    c_target=batch["c_target"],
                    g_bar=batch["g_bar"],
                )
            elif stage == "full":
                _require_keys(batch, ["c_target", "g_bar"])
                loss, info = self.loss_fn.forward_full(
                    logits_z=out["logits"],
                    logits_c=out["cbm_logits"],
                    labels=batch["y"],
                    c=out["c"],
                    c_target=batch["c_target"],
                    g_bar=batch["g_bar"],
                )
            else:
                raise ValueError(f"stage must be 'warm' or 'full', got {stage!r}")
    
            for k, v in info.items():
                loss_sums[k] = loss_sums.get(k, 0.0) + float(v)
    
            n_batches += 1
    
        mean_losses = {k: v / max(n_batches, 1) for k, v in loss_sums.items()}
    
        y_true = torch.cat(y_true).numpy()
        y_pred = torch.cat(y_pred).numpy()
        y_prob = torch.cat(y_prob).numpy()
    
        metrics = _classification_metrics_from_outputs(
            y_true=y_true,
            y_pred=y_pred,
            y_prob=y_prob,
            n_classes=self.model.n_classes if hasattr(self.model, "n_classes") else y_prob.shape[1],
        )
    
        out_dict = {**mean_losses, **metrics}
    
        if prefix:
            out_dict = {f"{prefix}_{k}": v for k, v in out_dict.items()}
    
        return out_dict

    def train_warm_epoch(self, train_loader, epoch):
        self.model.train()
        meter = {"loss": 0.0, "n": 0}
    
        for step, batch in enumerate(train_loader):
            _require_keys(batch, ["x", "y", "c_target", "g_bar"])
            batch = _move_batch(batch, self.device)
    
            self.optimizer.zero_grad(set_to_none=True)
    
            with torch.autocast(
                device_type=self.device.type,
                enabled=self.cfg.use_amp and self.device.type == "cuda"
            ):
                out = self._forward(batch["x"])
                loss, info = self.loss_fn.forward_warm(
                    logits_z=out["logits"],
                    logits_c=out["cbm_logits"],
                    labels=batch["y"],
                    c=out["c"],
                    c_target=batch["c_target"],
                    g_bar=batch["g_bar"],
                )
    
            if self.scaler.is_enabled():
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                self.optimizer.step()
    
            meter["loss"] += float(loss.item())
            meter["n"] += 1
    
        meter["loss"] /= max(meter["n"], 1)
        return meter

    def train_full_epoch(self, train_loader, epoch):
        self.model.train()
        meter = {"loss": 0.0, "n": 0}

        for step, batch in enumerate(train_loader):
            _require_keys(batch, ["x", "y", "c_target", "g_bar"])
            batch = _move_batch(batch, self.device)

            self.optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=self.device.type,
                enabled=self.cfg.use_amp and self.device.type == "cuda"
            ):
                out = self._forward(batch["x"])
                loss, info = self.loss_fn.forward_full(
                    logits_z=out["logits"],
                    logits_c=out["cbm_logits"],
                    labels=batch["y"],
                    c=out["c"],
                    c_target=batch["c_target"],
                    g_bar=batch["g_bar"],
                )

            if self.scaler.is_enabled():
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                self.optimizer.step()

            meter["loss"] += float(loss.item())
            meter["n"] += 1

        meter["loss"] /= max(meter["n"], 1)
        return meter

    # def fit(self, train_loader, source_eval_loader=None, target_eval_loader=None):
    #     history = {"warm": [], "full": []}
    
    #     for epoch in range(1, self.cfg.n_epochs_warm + 1):
    #         train_info = self.train_warm_epoch(train_loader, epoch)
    
    #         src_eval = (
    #             self.evaluate(source_eval_loader, prefix="source", stage="warm")
    #             if source_eval_loader is not None else {}
    #         )
    #         tgt_eval = (
    #             self.evaluate(target_eval_loader, prefix="target", stage="warm")
    #             if target_eval_loader is not None else {}
    #         )
    
    #         epoch_info = {**train_info, **src_eval, **tgt_eval}
    #         history["warm"].append(epoch_info)
    
    #         print(
    #             f"[Warm][Epoch {epoch:03d}] "
    #             f"train_loss={epoch_info['loss']:.4f} | "
    #             f"VAL acc={epoch_info.get('source_accuracy', float('nan')):.4f} "
    #             f"f1={epoch_info.get('source_f1_macro', float('nan')):.4f} "
    #             f"rec={epoch_info.get('source_recall_macro', float('nan')):.4f} "
    #             f"prec={epoch_info.get('source_precision_macro', float('nan')):.4f} "
    #             f"auc={epoch_info.get('source_auc_macro_ovr', float('nan')):.4f} | "
    #             # f"EXT acc={epoch_info.get('target_accuracy', float('nan')):.4f} "
    #             # f"f1={epoch_info.get('target_f1_macro', float('nan')):.4f} "
    #             # f"rec={epoch_info.get('target_recall_macro', float('nan')):.4f} "
    #             # f"prec={epoch_info.get('target_precision_macro', float('nan')):.4f} "
    #             # f"auc={epoch_info.get('target_auc_macro_ovr', float('nan')):.4f}"
    #         )

    #     for epoch in range(1, self.cfg.n_epochs_full + 1):
    #         train_info = self.train_full_epoch(train_loader, epoch)
    
    #         src_eval = (
    #             self.evaluate(source_eval_loader, prefix="source", stage="full")
    #             if source_eval_loader is not None else {}
    #         )
    #         tgt_eval = (
    #             self.evaluate(target_eval_loader, prefix="target", stage="full")
    #             if target_eval_loader is not None else {}
    #         )
    
    #         epoch_info = {**train_info, **src_eval, **tgt_eval}
    #         history["full"].append(epoch_info)
    
    #         print(
    #             f"[Full][Epoch {epoch:03d}] "
    #             f"train_loss={epoch_info['loss']:.4f} | "
    #             f"VAL acc={epoch_info.get('source_accuracy', float('nan')):.4f} "
    #             f"f1={epoch_info.get('source_f1_macro', float('nan')):.4f} "
    #             f"rec={epoch_info.get('source_recall_macro', float('nan')):.4f} "
    #             f"prec={epoch_info.get('source_precision_macro', float('nan')):.4f} "
    #             f"auc={epoch_info.get('source_auc_macro_ovr', float('nan')):.4f} | "
    #             # f"EXT acc={epoch_info.get('target_accuracy', float('nan')):.4f} "
    #             # f"f1={epoch_info.get('target_f1_macro', float('nan')):.4f} "
    #             # f"rec={epoch_info.get('target_recall_macro', float('nan')):.4f} "
    #             # f"prec={epoch_info.get('target_precision_macro', float('nan')):.4f} "
    #             # f"auc={epoch_info.get('target_auc_macro_ovr', float('nan')):.4f}"
    #         )
    
    #     return history
    def fit(self, train_loader, train_eval_loader=None, val_loader=None, external_loader=None):
        history = {"warm": [], "full": []}
    
        for epoch in range(1, self.cfg.n_epochs_warm + 1):
            train_info = self.train_warm_epoch(train_loader, epoch)
    
            train_eval = (
                self.evaluate(train_eval_loader, prefix="traincohort", stage="warm")
                if train_eval_loader is not None else {}
            )
            val_eval = (
                self.evaluate(val_loader, prefix="val", stage="warm")
                if val_loader is not None else {}
            )
            ext_eval = (
                self.evaluate(external_loader, prefix="external", stage="warm")
                if external_loader is not None else {}
            )
    
            epoch_info = {**train_info, **train_eval, **val_eval, **ext_eval}
            history["warm"].append(epoch_info)
    
            print(
                f"[Warm][Epoch {epoch:03d}] "
                f"train_loss={epoch_info['loss']:.4f} | "
                f"TRAIN acc={epoch_info.get('traincohort_accuracy', float('nan')):.4f} "
                f"f1={epoch_info.get('traincohort_f1_macro', float('nan')):.4f} | "
                f"VAL acc={epoch_info.get('val_accuracy', float('nan')):.4f} "
                f"f1={epoch_info.get('val_f1_macro', float('nan')):.4f}"
                f"rec={epoch_info.get('val_recall_macro', float('nan')):.4f} "
                f"prec={epoch_info.get('val_precision_macro', float('nan')):.4f} "
                f"auc={epoch_info.get('val_auc_macro_ovr', float('nan')):.4f} | "
            )
    
        for epoch in range(1, self.cfg.n_epochs_full + 1):
            train_info = self.train_full_epoch(train_loader, epoch)
    
            train_eval = (
                self.evaluate(train_eval_loader, prefix="traincohort", stage="full")
                if train_eval_loader is not None else {}
            )
            val_eval = (
                self.evaluate(val_loader, prefix="val", stage="full")
                if val_loader is not None else {}
            )
            ext_eval = (
                self.evaluate(external_loader, prefix="external", stage="full")
                if external_loader is not None else {}
            )
    
            epoch_info = {**train_info, **train_eval, **val_eval, **ext_eval}
            history["full"].append(epoch_info)
    
            print(
                f"[Full][Epoch {epoch:03d}] "
                f"train_loss={epoch_info['loss']:.4f} | "
                f"TRAIN acc={epoch_info.get('traincohort_accuracy', float('nan')):.4f} "
                f"f1={epoch_info.get('traincohort_f1_macro', float('nan')):.4f} | "
                f"VAL acc={epoch_info.get('val_accuracy', float('nan')):.4f} "
                f"f1={epoch_info.get('val_f1_macro', float('nan')):.4f}"
                f"rec={epoch_info.get('val_recall_macro', float('nan')):.4f} "
                f"prec={epoch_info.get('val_precision_macro', float('nan')):.4f} "
                f"auc={epoch_info.get('val_auc_macro_ovr', float('nan')):.4f} | "
            )
    
        return history


In [ ]:
"""
wire_and_train.py
=================
Wires together:
  - existing model.py / losses.py
  - the missing modules already created:
        atlas_utils.py
        concept_targets.py
        jacobian_utils.py
        trainer.py
        model_patch_concept.py
  - the user's Kaggle-style data layout

This script is based on the user's working dataloader pattern, but upgrades it so
that batches satisfy the mathematical contract required by Stage I and Stage II:

    Source batch: {x, y, c_target, g_bar, subject_id, label_name}
    Target batch: {x, y, g_bar, subject_id, label_name}

Key fixes relative to the base example:
  1) target batches now include g_bar, which Stage II needs.
  2) both c_target and g_bar are cached if missing.
  3) K is taken from the atlas, not from hard-coded defaults.
  4) the concept head is patched so it matches c_{n,k}=sigmoid(w_k^T u_{n,k}+b_k).
  5) the model is trained through the CBM logits, matching the more interpretable
     option stated in the Materials and Methods.

Expected environment:
  - Kaggle-style mounted data.
  - model.py and losses.py available in the working directory or a project folder.
  - SimpleITK installed if Jacobian priors must be computed from scratch.
"""

from __future__ import annotations

import os
import sys
import glob
import json
from dataclasses import asdict
from pathlib import Path
from typing import Dict, Optional, Tuple

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, roc_auc_score

# ---------------------------------------------------------------------
# 0) LABEL MAP
# ---------------------------------------------------------------------
LABEL_MAP = {
    "CN": 0,
    "MCI": 1,
    "AD": 2,
}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}


# ---------------------------------------------------------------------
# 1) PATH BOOTSTRAP
# ---------------------------------------------------------------------
def add_module_dir_to_path(module_dir: str | os.PathLike) -> None:
    module_dir = str(module_dir)
    if module_dir not in sys.path:
        sys.path.insert(0, module_dir)


def resolve_single_path(candidates) -> str:
    candidates = [str(p) for p in candidates if p and os.path.exists(str(p))]
    if not candidates:
        raise FileNotFoundError("No valid path found among candidates.")
    return candidates[0]


def discover_project_file(filename: str, search_roots: list[str]) -> str:
    hits = []
    for root in search_roots:
        if not root or not os.path.exists(root):
            continue
        hits.extend(glob.glob(os.path.join(root, "**", filename), recursive=True))
    hits = sorted(set(hits))
    if not hits:
        raise FileNotFoundError(f"Could not find {filename!r} in the provided roots.")
    return hits[0]


# ---------------------------------------------------------------------
# 2) ROBUST .pt LOADING
# ---------------------------------------------------------------------
def load_tensor_like(obj_path: str) -> torch.Tensor:
    obj = torch.load(obj_path, map_location="cpu", weights_only=False)
    if isinstance(obj, dict):
        for key in ["x", "image", "mri", "tensor", "volume"]:
            if key in obj and torch.is_tensor(obj[key]):
                x = obj[key]
                break
        else:
            raise KeyError(f"Could not find tensor-like key in dict loaded from {obj_path}")
    elif torch.is_tensor(obj):
        x = obj
    else:
        raise TypeError(f"Unsupported object type loaded from {obj_path}: {type(obj)}")

    if x.ndim == 3:
        x = x.unsqueeze(0)  # (1,H,W,D)
    if x.ndim != 4:
        raise ValueError(f"Expected MRI tensor with shape (1,H,W,D), got {tuple(x.shape)} from {obj_path}")
    return x.to(torch.float32)


# ---------------------------------------------------------------------
# 3) METADATA RESOLUTION FROM THE USER'S BASE LAYOUT
# ---------------------------------------------------------------------
def build_source_path_map() -> Dict[str, str]:
    """
    Mirrors the user's base example: dynamically searches /kaggle/input for ADNI .pt
    tensors while excluding cached concept/Jacobian artifacts and OASIS files.
    """
    path_map = {}
    for f in glob.glob("/kaggle/input/**/*.pt", recursive=True):
        fl = f.lower()
        if any(tok in fl for tok in ["c_target", "g_bar", "g_jacobian", "target_oasis", "oasis"]):
            continue
        basename = os.path.basename(f).replace(".pt", "")
        path_map[basename] = f
    return path_map


def resolve_source_x_path(row: pd.Series, source_path_map: Dict[str, str]) -> str:
    sub_id = str(row["Subject_ID"])
    if sub_id in source_path_map:
        return source_path_map[sub_id]

    # fallback to whichever path-like columns may exist in the CSV
    for col in ["File_Path", "Raw_File_Path", "Processed_File_Path", "x_path"]:
        if col in row and pd.notna(row[col]) and os.path.exists(str(row[col])):
            return str(row[col])

    raise FileNotFoundError(f"Could not resolve source MRI path for Subject_ID={sub_id}")


def resolve_target_x_path(row: pd.Series, base_dir: str) -> str:
    sub_id = str(row["Subject_ID"])
    label = str(row["Label"])

    # First try the normalized storage layout produced in preprocessing.
    candidate = os.path.join(base_dir, "target_oasis", label, f"{sub_id}_MRI.pt")
    if os.path.exists(candidate):
        return candidate

    # Then trust any explicit path in the CSV if present.
    for col in ["Processed_File_Path", "File_Path", "Raw_File_Path", "x_path"]:
        if col in row and pd.notna(row[col]) and os.path.exists(str(row[col])):
            return str(row[col])

    raise FileNotFoundError(f"Could not resolve target MRI path for Subject_ID={sub_id}")


def build_inventory_dataframes(base_dir: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    source_csv = os.path.join(base_dir, "source_labels.csv")
    target_csv = os.path.join(base_dir, "target_labels.csv")

    if not os.path.exists(source_csv):
        raise FileNotFoundError(f"Missing source CSV: {source_csv}")
    if not os.path.exists(target_csv):
        raise FileNotFoundError(f"Missing target CSV: {target_csv}")

    df_source = pd.read_csv(source_csv)
    df_target = pd.read_csv(target_csv)

    df_source = df_source[df_source["Label"].isin(LABEL_MAP)].reset_index(drop=True)
    df_target = df_target[df_target["Label"].isin(LABEL_MAP)].reset_index(drop=True)

    source_path_map = build_source_path_map()
    df_source = df_source.copy()
    df_target = df_target.copy()

    df_source["subject_id"] = df_source["Subject_ID"].astype(str)
    df_source["label"] = df_source["Label"].astype(str)
    df_source["x_path"] = df_source.apply(lambda r: resolve_source_x_path(r, source_path_map), axis=1)

    df_target["subject_id"] = df_target["Subject_ID"].astype(str)
    df_target["label"] = df_target["Label"].astype(str)
    df_target["x_path"] = df_target.apply(lambda r: resolve_target_x_path(r, base_dir), axis=1)

    return df_source[["subject_id", "label", "x_path"]], df_target[["subject_id", "label", "x_path"]]


# ---------------------------------------------------------------------
# 4) ARTIFACT CACHE CREATION (K and L)
# ---------------------------------------------------------------------
def find_existing_atlas_path(explicit_atlas_path: Optional[str] = None) -> str:
    if explicit_atlas_path is not None and os.path.exists(explicit_atlas_path):
        return explicit_atlas_path

    patterns = [
        "/kaggle/input/**/*atlas*.nii*",
        "/kaggle/input/**/*aal*.nii*",
        "/kaggle/input/**/*harvard*oxford*.nii*",
        "/kaggle/input/**/*label*.nii*",
        "/kaggle/working/**/*atlas*.nii*",
    ]
    hits = []
    for pat in patterns:
        hits.extend(glob.glob(pat, recursive=True))
    hits = sorted(set(hits))

    if not hits:
        raise FileNotFoundError(
            "Atlas file not found automatically. Please pass atlas_path explicitly."
        )
    return hits[0]


def ensure_simpleitk_or_raise():
    try:
        import SimpleITK  # noqa: F401
    except Exception as e:
        raise ImportError(
            "SimpleITK is required to compute g_bar Jacobian priors. "
            "Install it in Kaggle before running artifact generation."
        ) from e


def choose_template_x_path(df_source: pd.DataFrame) -> str:
    cn = df_source[df_source["label"] == "CN"].reset_index(drop=True)
    if len(cn) == 0:
        # fallback: first source sample
        return str(df_source.iloc[0]["x_path"])
    return str(cn.iloc[0]["x_path"])


def ensure_artifact_cache(
    base_dir: str,
    module_dir: str,
    atlas_path: str,
    recompute: bool = False,
) -> dict:
    add_module_dir_to_path(module_dir)

    # from atlas_utils import AtlasROIManager
    # from concept_targets import (
    #     ConceptTargetConfig,
    #     precompute_concept_targets_from_dataframe,
    # )
    # from jacobian_utils import (
    #     JacobianConfig,
    #     precompute_jacobians_from_dataframe,
    # )

    df_source, df_target = build_inventory_dataframes(base_dir)

    artifacts_dir = os.path.join(module_dir, "derived_artifacts")
    src_concepts_dir = os.path.join(artifacts_dir, "source_concepts")
    src_jac_dir = os.path.join(artifacts_dir, "source_jacobians")
    tgt_jac_dir = os.path.join(artifacts_dir, "target_jacobians")
    os.makedirs(artifacts_dir, exist_ok=True)

    atlas_mgr = AtlasROIManager(atlas_path)
    template_x_path = choose_template_x_path(df_source)

    source_concepts_csv = os.path.join(artifacts_dir, "source_concepts_index.csv")
    source_jac_csv = os.path.join(artifacts_dir, "source_jacobians_index.csv")
    target_jac_csv = os.path.join(artifacts_dir, "target_jacobians_index.csv")

    if recompute or not os.path.exists(source_concepts_csv):
        _, df_concepts = precompute_concept_targets_from_dataframe(
            df=df_source,
            atlas_mgr=atlas_mgr,
            x_column="x_path",
            label_column="label",
            subject_id_column="subject_id",
            output_dir=src_concepts_dir,
            cfg=ConceptTargetConfig(normal_class_name="CN"),
        )
        df_concepts.to_csv(source_concepts_csv, index=False)
    else:
        df_concepts = pd.read_csv(source_concepts_csv)

    if recompute or not os.path.exists(source_jac_csv):
        ensure_simpleitk_or_raise()
        df_src_jac = precompute_jacobians_from_dataframe(
            df=df_source,
            atlas_mgr=atlas_mgr,
            template_x_path=template_x_path,
            x_column="x_path",
            subject_id_column="subject_id",
            output_dir=src_jac_dir,
            cfg=JacobianConfig(),
        )
        df_src_jac.to_csv(source_jac_csv, index=False)
    else:
        df_src_jac = pd.read_csv(source_jac_csv)

    if recompute or not os.path.exists(target_jac_csv):
        ensure_simpleitk_or_raise()
        df_tgt_jac = precompute_jacobians_from_dataframe(
            df=df_target,
            atlas_mgr=atlas_mgr,
            template_x_path=template_x_path,
            x_column="x_path",
            subject_id_column="subject_id",
            output_dir=tgt_jac_dir,
            cfg=JacobianConfig(),
        )
        df_tgt_jac.to_csv(target_jac_csv, index=False)
    else:
        df_tgt_jac = pd.read_csv(target_jac_csv)

    return {
        "atlas_path": atlas_path,
        "template_x_path": template_x_path,
        "K": atlas_mgr.K,
        "df_source": df_source,
        "df_target": df_target,
        "df_concepts": df_concepts,
        "df_src_jac": df_src_jac,
        "df_tgt_jac": df_tgt_jac,
        "source_concepts_csv": source_concepts_csv,
        "source_jac_csv": source_jac_csv,
        "target_jac_csv": target_jac_csv,
    }


# ---------------------------------------------------------------------
# 5) DATASETS AND DATALOADERS (adapted from the user's base example)
# ---------------------------------------------------------------------
def _index_by_subject(df: pd.DataFrame, path_col: str) -> Dict[str, str]:
    out = {}
    for _, row in df.iterrows():
        out[str(row["subject_id"])] = str(row[path_col])
    return out


def _load_vector(path: str, expected_last_dim: int) -> torch.Tensor:
    obj = torch.load(path, map_location="cpu", weights_only=False)

    if torch.is_tensor(obj):
        v = obj

    elif isinstance(obj, dict):
        v = None
        for key in ["c_target", "g_bar", "x", "tensor", "vector"]:
            if key in obj:
                v = obj[key]
                break
        if v is None:
            raise KeyError(
                f"No se encontró ningún vector válido en {path}. "
                f"Keys disponibles: {list(obj.keys())}"
            )
    else:
        raise TypeError(f"Unsupported object at {path}: {type(obj)}")

    if not torch.is_tensor(v):
        v = torch.as_tensor(v)

    v = v.detach().to(torch.float32).view(-1)

    if v.numel() != expected_last_dim:
        raise ValueError(
            f"Expected vector with K={expected_last_dim} at {path}, got shape {tuple(v.shape)}"
        )
    # romper cualquier posible dependencia residual con MetaTensor
    v = torch.tensor(v.cpu().numpy(), dtype=torch.float32)
    return v

class SourceDomainDatasetWired(Dataset):
    def __init__(self, df_source, df_concepts, df_src_jac, K: int):
        super().__init__()
        self.data = df_source.reset_index(drop=True)
        self.K = int(K)
        self.c_map = _index_by_subject(df_concepts, "concept_target_path")
        self.g_map = _index_by_subject(df_src_jac, "g_bar_path")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        sub_id = str(row["subject_id"])
        label_str = str(row["label"])

        x = load_tensor_like(str(row["x_path"]))
        y = torch.tensor(LABEL_MAP[label_str], dtype=torch.long)
        c_target = _load_vector(self.c_map[sub_id], expected_last_dim=self.K)
        g_bar = _load_vector(self.g_map[sub_id], expected_last_dim=self.K)

        return {
            "x": x,
            "y": y,
            "c_target": c_target,
            "g_bar": g_bar,
            "subject_id": sub_id,
            "label_name": label_str,
        }


class TargetDomainDatasetWired(Dataset):
    def __init__(self, df_target, df_tgt_jac, K: int):
        super().__init__()
        self.data = df_target.reset_index(drop=True)
        self.K = int(K)
        self.g_map = _index_by_subject(df_tgt_jac, "g_bar_path")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        sub_id = str(row["subject_id"])
        label_str = str(row["label"])

        x = load_tensor_like(str(row["x_path"]))
        y = torch.tensor(LABEL_MAP[label_str], dtype=torch.long)
        g_bar = _load_vector(self.g_map[sub_id], expected_last_dim=self.K)

        return {
            "x": x,
            "y": y,
            "g_bar": g_bar,
            "subject_id": sub_id,
            "label_name": label_str,
        }


def get_domain_adaptation_dataloaders_wired(
    base_dir: str,
    module_dir: str,
    atlas_path: str,
    batch_size: int = 2,
    num_workers: int = 2,
    recompute_artifacts: bool = False,
):
    cache = ensure_artifact_cache(
        base_dir=base_dir,
        module_dir=module_dir,
        atlas_path=atlas_path,
        recompute=recompute_artifacts,
    )

    K = int(cache["K"])
    source_dataset = SourceDomainDatasetWired(
        df_source=cache["df_source"],
        df_concepts=cache["df_concepts"],
        df_src_jac=cache["df_src_jac"],
        K=K,
    )
    target_dataset = TargetDomainDatasetWired(
        df_target=cache["df_target"],
        df_tgt_jac=cache["df_tgt_jac"],
        K=K,
    )

    use_pin_memory = torch.cuda.is_available()

    source_loader = DataLoader(
        source_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )
    target_loader = DataLoader(
        target_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    return source_loader, target_loader, cache


# ---------------------------------------------------------------------
# 6) MODEL FACTORY WITH CONCEPT-HEAD PATCH
# ---------------------------------------------------------------------
def build_patched_model(
    project_root: str,
    K: int,
    n_classes: int = 3,
    C_f: int = 256,
    C_t: int = 128,
    n_heads: int = 4,
    n_layers: int = 2,
    base_ch: int = 32,
):
    add_module_dir_to_path(project_root)
    # from model import AlzheimerDomainAdaptationModel
    # from model_patch_concept import ConceptBottleneck as PatchedConceptBottleneck

    model = AlzheimerDomainAdaptationModel(
        K=K,
        C_f=C_f,
        C_t=C_t,
        n_classes=n_classes,
        n_heads=n_heads,
        n_layers=n_layers,
        base_ch=base_ch,
    )

    # Replace only the CBM module so the math matches Section J exactly.
    model.cbm = PatchedConceptBottleneck(K=K, C_t=C_t, n_classes=n_classes)
    model.K = K
    return model

import torch
import torch.nn as nn

def pick_safe_device(verbose: bool = True) -> torch.device:
    """
    Selecciona CUDA solo si realmente es usable en esta sesión.
    Si CUDA existe pero la build actual de PyTorch no soporta la GPU asignada,
    hace fallback a CPU.
    """
    if not torch.cuda.is_available():
        if verbose:
            print("[Device] CUDA no disponible. Se usará CPU.")
        return torch.device("cpu")

    try:
        # prueba mínima de CUDA
        x = torch.zeros(1, device="cuda")
        _ = x + 1

        # prueba mínima de kernel real
        conv = nn.Conv3d(1, 2, kernel_size=3, padding=1).to("cuda")
        y = conv(torch.zeros(1, 1, 8, 8, 8, device="cuda"))
        _ = y.sum().item()

        if verbose:
            print(f"[Device] CUDA usable: {torch.cuda.get_device_name(0)}")
        return torch.device("cuda")

    except Exception as e:
        if verbose:
            print("[Device] CUDA detectada pero no usable con la build actual de PyTorch.")
            print(f"[Device] Fallback a CPU. Motivo: {type(e).__name__}: {e}")
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        return torch.device("cpu")
# ---------------------------------------------------------------------
# 7) TRAINING ENTRY POINT
# ---------------------------------------------------------------------

# ============================================================
# analysis_cbm.py
# Instrumentación para análisis anatómico e interpretabilidad
# Compatible con tu SupervisedMRITrainer actual
# ============================================================

from __future__ import annotations

import json
import math
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import f1_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


# ------------------------------------------------------------
# Utilidades
# ------------------------------------------------------------
def _to_numpy(x):
    if x is None:
        return None
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def _safe_nanmean(x):
    x = np.asarray(x, dtype=np.float64)
    if x.size == 0:
        return float("nan")
    return float(np.nanmean(x))


def _safe_corr_1d(x, y, eps: float = 1e-12):
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    y = np.asarray(y, dtype=np.float64).reshape(-1)

    if x.size != y.size or x.size < 2:
        return float("nan")

    x_std = x.std()
    y_std = y.std()
    if x_std < eps or y_std < eps:
        return float("nan")

    return float(np.corrcoef(x, y)[0, 1])


def _mean_roi_corr(pred, target):
    pred = np.asarray(pred, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)

    if pred.ndim != 2 or target.ndim != 2 or pred.shape != target.shape:
        return float("nan")

    corrs = []
    for k in range(pred.shape[1]):
        corrs.append(_safe_corr_1d(pred[:, k], target[:, k]))
    return _safe_nanmean(corrs)


def _alpha_entropy(alpha, eps: float = 1e-8):
    alpha = np.asarray(alpha, dtype=np.float64)
    if alpha.ndim != 2:
        return float("nan")
    ent = -(alpha * np.log(alpha + eps)).sum(axis=1)
    return _safe_nanmean(ent)


def _batch_size_from_dict(batch: dict) -> int:
    for v in batch.values():
        if torch.is_tensor(v):
            return int(v.shape[0])
        if isinstance(v, (list, tuple)):
            return len(v)
    raise ValueError("No fue posible inferir batch size.")


def _slice_batch(batch: dict, n_keep: int) -> dict:
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            out[k] = v[:n_keep]
        elif isinstance(v, (list, tuple)):
            out[k] = list(v[:n_keep])
        else:
            out[k] = v
    return out


def _save_npz_dict(path: str, data: dict):
    to_save = {}
    for k, v in data.items():
        arr = v
        if isinstance(arr, list):
            arr = np.asarray(arr, dtype=object)
        elif torch.is_tensor(arr):
            arr = arr.detach().cpu().numpy()
        elif not isinstance(arr, np.ndarray):
            arr = np.asarray(arr)
        to_save[k] = arr
    np.savez_compressed(path, **to_save)


def _score_regression_r2(y_true, y_pred, eps: float = 1e-12):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)

    ss_res = float(((y_true - y_pred) ** 2).sum())
    ss_tot = float(((y_true - y_true.mean()) ** 2).sum())
    if ss_tot < eps:
        return float("nan")
    return 1.0 - ss_res / ss_tot


def _safe_stratified_splits(y, desired_splits: int):
    y = np.asarray(y)
    classes, counts = np.unique(y, return_counts=True)
    if len(classes) < 2:
        return 0
    min_count = int(counts.min())
    return max(0, min(desired_splits, min_count))


@dataclass
class AnalysisConfig:
    save_dir: str = "/kaggle/working/analysis_cbm"
    save_best_checkpoints: bool = True
    save_epoch_representations: bool = False
    checkpoint_split: str = "val"         # "traincohort", "val", "external"
    ablation_stage: str = "U"             # "T", "U", "c"
    ablation_baseline: float = 0.0
    ablation_max_subjects: Optional[int] = 64
    probes_cv_splits: int = 3
    random_state: int = 42


# ------------------------------------------------------------
# Trainer instrumentado
# ------------------------------------------------------------
# class AnalysisAwareSupervisedMRITrainer(SupervisedMRITrainer):
#     def __init__(
#         self,
#         model,
#         loss_fn,
#         atlas_mgr,
#         input_shape=(128, 128, 128),
#         cfg: Optional[SupervisedTrainConfig] = None,
#         optimizer=None,
#         analysis_cfg: Optional[AnalysisConfig] = None,
#     ):
#         super().__init__(
#             model=model,
#             loss_fn=loss_fn,
#             atlas_mgr=atlas_mgr,
#             input_shape=input_shape,
#             cfg=cfg,
#             optimizer=optimizer,
#         )
#         self.analysis_cfg = analysis_cfg or AnalysisConfig()
#         self.analysis_dir = Path(self.analysis_cfg.save_dir)
#         self.analysis_dir.mkdir(parents=True, exist_ok=True)

#         self.best_scores = {
#             "z": -np.inf,
#             "cbm": -np.inf,
#             "anat": -np.inf,
#         }
#         self.best_paths = {
#             "z": None,
#             "cbm": None,
#             "anat": None,
#         }

#     def _checkpoint_payload(self, epoch: int, stage: str, epoch_info: dict) -> dict:
#         return {
#             "epoch": int(epoch),
#             "stage": str(stage),
#             "model_state_dict": self.model.state_dict(),
#             "optimizer_state_dict": self.optimizer.state_dict(),
#             "epoch_info": epoch_info,
#             "roi_masks_shape": tuple(self.roi_masks.shape),
#             "analysis_cfg": self.analysis_cfg.__dict__,
#         }

#     def _save_checkpoint(self, tag: str, epoch: int, stage: str, epoch_info: dict):
#         path = self.analysis_dir / f"best_{tag}.pt"
#         torch.save(self._checkpoint_payload(epoch, stage, epoch_info), path)
#         self.best_paths[tag] = str(path)

#     def _safe_metric(self, epoch_info: dict, key: str, default: float = float("-inf")):
#         val = epoch_info.get(key, default)
#         if val is None:
#             return default
#         try:
#             val = float(val)
#         except Exception:
#             return default
#         if math.isnan(val):
#             return default
#         return val

#     def _score_z(self, epoch_info: dict, split: str):
#         return (
#             self._safe_metric(epoch_info, f"{split}_z_f1_macro", -np.inf)
#             + self._safe_metric(epoch_info, f"{split}_z_auc_macro_ovr", -np.inf)
#         )

#     def _score_cbm(self, epoch_info: dict, split: str):
#         return (
#             self._safe_metric(epoch_info, f"{split}_cbm_f1_macro", -np.inf)
#             + self._safe_metric(epoch_info, f"{split}_cbm_auc_macro_ovr", -np.inf)
#         )

#     def _score_anat(self, epoch_info: dict, split: str):
#         corr_c = self._safe_metric(epoch_info, f"{split}_concept_corr_mean", -np.inf)
#         corr_g = self._safe_metric(epoch_info, f"{split}_jac_corr_mean", -np.inf)
#         mse_c = self._safe_metric(epoch_info, f"{split}_concept_mse", np.inf)
#         mse_g = self._safe_metric(epoch_info, f"{split}_jac_mse", np.inf)

#         if np.isinf(mse_c):
#             mse_c = 1e6
#         if np.isinf(mse_g):
#             mse_g = 1e6

#         return 0.5 * (corr_c + corr_g) - 0.25 * (mse_c + mse_g)

#     def _maybe_save_best(self, epoch: int, stage: str, epoch_info: dict):
#         if not self.analysis_cfg.save_best_checkpoints:
#             return

#         split = self.analysis_cfg.checkpoint_split

#         z_score = self._score_z(epoch_info, split)
#         cbm_score = self._score_cbm(epoch_info, split)
#         anat_score = self._score_anat(epoch_info, split)

#         if z_score > self.best_scores["z"]:
#             self.best_scores["z"] = z_score
#             self._save_checkpoint("z", epoch, stage, epoch_info)

#         if cbm_score > self.best_scores["cbm"]:
#             self.best_scores["cbm"] = cbm_score
#             self._save_checkpoint("cbm", epoch, stage, epoch_info)

#         if anat_score > self.best_scores["anat"]:
#             self.best_scores["anat"] = anat_score
#             self._save_checkpoint("anat", epoch, stage, epoch_info)

#     @torch.no_grad()
#     def extract_bundle(self, loader, max_subjects: Optional[int] = None):
#         self.model.eval()

#         acc = {
#             "subject_id": [],
#             "label_name": [],
#             "y": [],
#             "c_target": [],
#             "g_bar": [],
#             "T": [],
#             "U": [],
#             "z": [],
#             "alpha": [],
#             "c": [],
#             "logits_z": [],
#             "logits_c": [],
#             "prob_z": [],
#             "prob_c": [],
#         }

#         seen = 0

#         for batch in loader:
#             batch_size = _batch_size_from_dict(batch)

#             if max_subjects is not None:
#                 remaining = max_subjects - seen
#                 if remaining <= 0:
#                     break
#                 if batch_size > remaining:
#                     batch = _slice_batch(batch, remaining)
#                     batch_size = remaining

#             batch = _move_batch(batch, self.device)
#             out = self._forward(batch["x"])

#             prob_z = torch.softmax(out["logits"], dim=-1)
#             prob_c = torch.softmax(out["cbm_logits"], dim=-1)

#             if "subject_id" in batch:
#                 acc["subject_id"].extend(list(batch["subject_id"]))
#             else:
#                 acc["subject_id"].extend([f"sample_{seen+i}" for i in range(batch_size)])

#             if "label_name" in batch:
#                 acc["label_name"].extend(list(batch["label_name"]))
#             else:
#                 acc["label_name"].extend([""] * batch_size)

#             acc["y"].append(batch["y"].detach().cpu())
#             if "c_target" in batch:
#                 acc["c_target"].append(batch["c_target"].detach().cpu())
#             if "g_bar" in batch:
#                 acc["g_bar"].append(batch["g_bar"].detach().cpu())

#             acc["T"].append(out["T"].detach().cpu())
#             acc["U"].append(out["U"].detach().cpu())
#             acc["z"].append(out["z"].detach().cpu())
#             acc["alpha"].append(out["alpha"].detach().cpu())
#             acc["c"].append(out["c"].detach().cpu())
#             acc["logits_z"].append(out["logits"].detach().cpu())
#             acc["logits_c"].append(out["cbm_logits"].detach().cpu())
#             acc["prob_z"].append(prob_z.detach().cpu())
#             acc["prob_c"].append(prob_c.detach().cpu())

#             seen += batch_size

#         out_bundle = {}
#         for k, v in acc.items():
#             if len(v) == 0:
#                 out_bundle[k] = np.array([])
#             elif isinstance(v[0], torch.Tensor):
#                 out_bundle[k] = torch.cat(v, dim=0).numpy()
#             else:
#                 out_bundle[k] = np.asarray(v, dtype=object)

#         return out_bundle

#     @torch.no_grad()
#     def evaluate(self, loader, prefix="", stage="full"):
#         """
#         stage:
#             - warm: mantiene compatibilidad con la cabeza z
#             - full: mantiene compatibilidad con la cabeza cbm

#         Además reporta:
#             - métricas de ambas cabezas
#             - MSE y correlación anatómica
#             - entropía media de atención
#         """
#         self.model.eval()

#         loss_sums = {}
#         n_batches = 0

#         y_true = []
#         y_pred_z = []
#         y_prob_z = []

#         y_pred_c = []
#         y_prob_c = []

#         c_all = []
#         c_target_all = []
#         g_bar_all = []
#         alpha_all = []

#         for batch in loader:
#             batch = _move_batch(batch, self.device)
#             out = self._forward(batch["x"])

#             prob_z = torch.softmax(out["logits"], dim=-1)
#             pred_z = prob_z.argmax(dim=-1)

#             prob_c = torch.softmax(out["cbm_logits"], dim=-1)
#             pred_c = prob_c.argmax(dim=-1)

#             y_true.append(batch["y"].detach().cpu())
#             y_pred_z.append(pred_z.detach().cpu())
#             y_prob_z.append(prob_z.detach().cpu())

#             y_pred_c.append(pred_c.detach().cpu())
#             y_prob_c.append(prob_c.detach().cpu())

#             alpha_all.append(out["alpha"].detach().cpu())
#             c_all.append(out["c"].detach().cpu())

#             if "c_target" in batch:
#                 c_target_all.append(batch["c_target"].detach().cpu())
#             if "g_bar" in batch:
#                 g_bar_all.append(batch["g_bar"].detach().cpu())

#             if stage == "warm":
#                 _, info = self.loss_fn.forward_warm(
#                     logits_z=out["logits"],
#                     labels=batch["y"],
#                 )
#             elif stage == "full":
#                 _require_keys(batch, ["c_target", "g_bar"])
#                 _, info = self.loss_fn.forward_full(
#                     logits_z=out["logits"],
#                     logits_c=out["cbm_logits"],
#                     labels=batch["y"],
#                     c=out["c"],
#                     c_target=batch["c_target"],
#                     g_bar=batch["g_bar"],
#                 )
#             else:
#                 raise ValueError(f"stage must be 'warm' or 'full', got {stage!r}")

#             for k, v in info.items():
#                 loss_sums[k] = loss_sums.get(k, 0.0) + float(v)

#             n_batches += 1

#         mean_losses = {k: v / max(n_batches, 1) for k, v in loss_sums.items()}

#         y_true = torch.cat(y_true).numpy()

#         y_pred_z = torch.cat(y_pred_z).numpy()
#         y_prob_z = torch.cat(y_prob_z).numpy()
#         z_metrics = _classification_metrics_from_outputs(
#             y_true=y_true,
#             y_pred=y_pred_z,
#             y_prob=y_prob_z,
#             n_classes=self.model.n_classes,
#         )

#         y_pred_c = torch.cat(y_pred_c).numpy()
#         y_prob_c = torch.cat(y_prob_c).numpy()
#         cbm_metrics = _classification_metrics_from_outputs(
#             y_true=y_true,
#             y_pred=y_pred_c,
#             y_prob=y_prob_c,
#             n_classes=self.model.n_classes,
#         )

#         alpha_np = torch.cat(alpha_all).numpy()
#         c_np = torch.cat(c_all).numpy()

#         concept_mse = float("nan")
#         jac_mse = float("nan")
#         concept_corr_mean = float("nan")
#         jac_corr_mean = float("nan")

#         if len(c_target_all) > 0:
#             c_target_np = torch.cat(c_target_all).numpy()
#             concept_mse = float(np.mean((c_np - c_target_np) ** 2))
#             concept_corr_mean = _mean_roi_corr(c_np, c_target_np)

#         if len(g_bar_all) > 0:
#             g_bar_np = torch.cat(g_bar_all).numpy()
#             jac_mse = float(np.mean((c_np - g_bar_np) ** 2))
#             jac_corr_mean = _mean_roi_corr(c_np, g_bar_np)

#         out_dict = {
#             **mean_losses,
#             "z_accuracy": z_metrics["accuracy"],
#             "z_f1_macro": z_metrics["f1_macro"],
#             "z_recall_macro": z_metrics["recall_macro"],
#             "z_precision_macro": z_metrics["precision_macro"],
#             "z_auc_macro_ovr": z_metrics["auc_macro_ovr"],
#             "cbm_accuracy": cbm_metrics["accuracy"],
#             "cbm_f1_macro": cbm_metrics["f1_macro"],
#             "cbm_recall_macro": cbm_metrics["recall_macro"],
#             "cbm_precision_macro": cbm_metrics["precision_macro"],
#             "cbm_auc_macro_ovr": cbm_metrics["auc_macro_ovr"],
#             "concept_mse": concept_mse,
#             "jac_mse": jac_mse,
#             "concept_corr_mean": concept_corr_mean,
#             "jac_corr_mean": jac_corr_mean,
#             "alpha_entropy": _alpha_entropy(alpha_np),
#         }

#         # Compatibilidad con el historial previo:
#         # warm -> usa cabeza z como cabeza "activa"
#         # full -> usa cabeza cbm como cabeza "activa"
#         active_head = "z" if stage == "warm" else "cbm"
#         for metric_name in ["accuracy", "f1_macro", "recall_macro", "precision_macro", "auc_macro_ovr"]:
#             out_dict[metric_name] = out_dict[f"{active_head}_{metric_name}"]

#         if prefix:
#             out_dict = {f"{prefix}_{k}": v for k, v in out_dict.items()}

#         return out_dict

#     def fit(self, train_loader, train_eval_loader=None, val_loader=None, external_loader=None):
#         history = {"warm": [], "full": []}

#         for epoch in range(1, self.cfg.n_epochs_warm + 1):
#             train_info = self.train_warm_epoch(train_loader, epoch)

#             train_eval = (
#                 self.evaluate(train_eval_loader, prefix="traincohort", stage="warm")
#                 if train_eval_loader is not None else {}
#             )
#             val_eval = (
#                 self.evaluate(val_loader, prefix="val", stage="warm")
#                 if val_loader is not None else {}
#             )
#             ext_eval = (
#                 self.evaluate(external_loader, prefix="external", stage="warm")
#                 if external_loader is not None else {}
#             )

#             epoch_info = {**train_info, **train_eval, **val_eval, **ext_eval}
#             history["warm"].append(epoch_info)
#             self._maybe_save_best(epoch=epoch, stage="warm", epoch_info=epoch_info)

#             if self.analysis_cfg.save_epoch_representations and val_loader is not None:
#                 bundle = self.extract_bundle(val_loader, max_subjects=self.analysis_cfg.ablation_max_subjects)
#                 _save_npz_dict(
#                     str(self.analysis_dir / f"warm_epoch_{epoch:03d}_val_bundle.npz"),
#                     bundle,
#                 )

#             print(
#                 f"[Warm][Epoch {epoch:03d}] "
#                 f"train_loss={epoch_info['loss']:.4f} | "
#                 f"TRAIN z_f1={epoch_info.get('traincohort_z_f1_macro', float('nan')):.4f} "
#                 f"cbm_f1={epoch_info.get('traincohort_cbm_f1_macro', float('nan')):.4f} | "
#                 f"VAL z_f1={epoch_info.get('val_z_f1_macro', float('nan')):.4f} "
#                 f"cbm_f1={epoch_info.get('val_cbm_f1_macro', float('nan')):.4f} "
#                 f"anat_corr={epoch_info.get('val_concept_corr_mean', float('nan')):.4f} "
#                 f"jac_corr={epoch_info.get('val_jac_corr_mean', float('nan')):.4f}"
#             )

#         for epoch in range(1, self.cfg.n_epochs_full + 1):
#             train_info = self.train_full_epoch(train_loader, epoch)

#             train_eval = (
#                 self.evaluate(train_eval_loader, prefix="traincohort", stage="full")
#                 if train_eval_loader is not None else {}
#             )
#             val_eval = (
#                 self.evaluate(val_loader, prefix="val", stage="full")
#                 if val_loader is not None else {}
#             )
#             ext_eval = (
#                 self.evaluate(external_loader, prefix="external", stage="full")
#                 if external_loader is not None else {}
#             )

#             epoch_info = {**train_info, **train_eval, **val_eval, **ext_eval}
#             history["full"].append(epoch_info)
#             self._maybe_save_best(epoch=epoch, stage="full", epoch_info=epoch_info)

#             if self.analysis_cfg.save_epoch_representations and val_loader is not None:
#                 bundle = self.extract_bundle(val_loader, max_subjects=self.analysis_cfg.ablation_max_subjects)
#                 _save_npz_dict(
#                     str(self.analysis_dir / f"full_epoch_{epoch:03d}_val_bundle.npz"),
#                     bundle,
#                 )

#             print(
#                 f"[Full][Epoch {epoch:03d}] "
#                 f"train_loss={epoch_info['loss']:.4f} | "
#                 f"TRAIN z_f1={epoch_info.get('traincohort_z_f1_macro', float('nan')):.4f} "
#                 f"cbm_f1={epoch_info.get('traincohort_cbm_f1_macro', float('nan')):.4f} | "
#                 f"VAL z_f1={epoch_info.get('val_z_f1_macro', float('nan')):.4f} "
#                 f"cbm_f1={epoch_info.get('val_cbm_f1_macro', float('nan')):.4f} "
#                 f"anat_corr={epoch_info.get('val_concept_corr_mean', float('nan')):.4f} "
#                 f"jac_corr={epoch_info.get('val_jac_corr_mean', float('nan')):.4f}"
#             )

#         with open(self.analysis_dir / "history_analysis.json", "w", encoding="utf-8") as f:
#             json.dump(history, f, indent=2)

#         with open(self.analysis_dir / "best_paths.json", "w", encoding="utf-8") as f:
#             json.dump(self.best_paths, f, indent=2)

#         return history

from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Optional

import numpy as np
import torch


class AnalysisAwareSupervisedMRITrainer(SupervisedMRITrainer):
    def __init__(
        self,
        model,
        loss_fn,
        atlas_mgr,
        input_shape=(128, 128, 128),
        cfg: Optional[SupervisedTrainConfig] = None,
        optimizer=None,
        analysis_cfg: Optional[AnalysisConfig] = None,
    ):
        super().__init__(
            model=model,
            loss_fn=loss_fn,
            atlas_mgr=atlas_mgr,
            input_shape=input_shape,
            cfg=cfg,
            optimizer=optimizer,
        )
        self.analysis_cfg = analysis_cfg or AnalysisConfig()
        self.analysis_dir = Path(self.analysis_cfg.save_dir)
        self.analysis_dir.mkdir(parents=True, exist_ok=True)

        self.best_scores = {
            "z": -np.inf,
            "cbm": -np.inf,
            "anat": -np.inf,
        }
        self.best_paths = {
            "z": None,
            "cbm": None,
            "anat": None,
        }

    def _checkpoint_payload(self, epoch: int, stage: str, epoch_info: dict) -> dict:
        return {
            "epoch": int(epoch),
            "stage": str(stage),
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "epoch_info": epoch_info,
            "roi_masks_shape": tuple(self.roi_masks.shape),
            "analysis_cfg": self.analysis_cfg.__dict__,
        }

    def _save_checkpoint(self, tag: str, epoch: int, stage: str, epoch_info: dict):
        path = self.analysis_dir / f"best_{tag}.pt"
        torch.save(self._checkpoint_payload(epoch, stage, epoch_info), path)
        self.best_paths[tag] = str(path)

    def _safe_metric(self, epoch_info: dict, key: str, default: float = float("-inf")):
        val = epoch_info.get(key, default)
        if val is None:
            return default
        try:
            val = float(val)
        except Exception:
            return default
        if math.isnan(val):
            return default
        return val

    def _score_z(self, epoch_info: dict, split: str):
        return (
            self._safe_metric(epoch_info, f"{split}_z_f1_macro", -np.inf)
            + self._safe_metric(epoch_info, f"{split}_z_auc_macro_ovr", -np.inf)
        )

    def _score_cbm(self, epoch_info: dict, split: str):
        return (
            self._safe_metric(epoch_info, f"{split}_cbm_f1_macro", -np.inf)
            + self._safe_metric(epoch_info, f"{split}_cbm_auc_macro_ovr", -np.inf)
        )

    def _score_anat(self, epoch_info: dict, split: str):
        corr_c = self._safe_metric(epoch_info, f"{split}_concept_corr_mean", -np.inf)
        corr_g = self._safe_metric(epoch_info, f"{split}_jac_corr_mean", -np.inf)
        mse_c = self._safe_metric(epoch_info, f"{split}_concept_mse", np.inf)
        mse_g = self._safe_metric(epoch_info, f"{split}_jac_mse", np.inf)

        if np.isinf(mse_c):
            mse_c = 1e6
        if np.isinf(mse_g):
            mse_g = 1e6

        return 0.5 * (corr_c + corr_g) - 0.25 * (mse_c + mse_g)

    def _maybe_save_best(self, epoch: int, stage: str, epoch_info: dict):
        if not self.analysis_cfg.save_best_checkpoints:
            return

        split = self.analysis_cfg.checkpoint_split

        z_score = self._score_z(epoch_info, split)
        cbm_score = self._score_cbm(epoch_info, split)
        anat_score = self._score_anat(epoch_info, split)

        if z_score > self.best_scores["z"]:
            self.best_scores["z"] = z_score
            self._save_checkpoint("z", epoch, stage, epoch_info)

        if cbm_score > self.best_scores["cbm"]:
            self.best_scores["cbm"] = cbm_score
            self._save_checkpoint("cbm", epoch, stage, epoch_info)

        if anat_score > self.best_scores["anat"]:
            self.best_scores["anat"] = anat_score
            self._save_checkpoint("anat", epoch, stage, epoch_info)

    @torch.no_grad()
    def extract_bundle(self, loader, max_subjects: Optional[int] = None):
        self.model.eval()

        acc = {
            "subject_id": [],
            "label_name": [],
            "y": [],
            "c_target": [],
            "g_bar": [],
            "T": [],
            "U": [],
            "z": [],
            "alpha": [],
            "c": [],
            "logits_z": [],
            "logits_c": [],
            "prob_z": [],
            "prob_c": [],
        }

        seen = 0

        for batch in loader:
            batch_size = _batch_size_from_dict(batch)

            if max_subjects is not None:
                remaining = max_subjects - seen
                if remaining <= 0:
                    break
                if batch_size > remaining:
                    batch = _slice_batch(batch, remaining)
                    batch_size = remaining

            batch = _move_batch(batch, self.device)
            out = self._forward(batch["x"])

            prob_z = torch.softmax(out["logits"], dim=-1)
            prob_c = torch.softmax(out["cbm_logits"], dim=-1)

            if "subject_id" in batch:
                acc["subject_id"].extend(list(batch["subject_id"]))
            else:
                acc["subject_id"].extend([f"sample_{seen+i}" for i in range(batch_size)])

            if "label_name" in batch:
                acc["label_name"].extend(list(batch["label_name"]))
            else:
                acc["label_name"].extend([""] * batch_size)

            acc["y"].append(batch["y"].detach().cpu())
            if "c_target" in batch:
                acc["c_target"].append(batch["c_target"].detach().cpu())
            if "g_bar" in batch:
                acc["g_bar"].append(batch["g_bar"].detach().cpu())

            acc["T"].append(out["T"].detach().cpu())
            acc["U"].append(out["U"].detach().cpu())
            acc["z"].append(out["z"].detach().cpu())
            acc["alpha"].append(out["alpha"].detach().cpu())
            acc["c"].append(out["c"].detach().cpu())
            acc["logits_z"].append(out["logits"].detach().cpu())
            acc["logits_c"].append(out["cbm_logits"].detach().cpu())
            acc["prob_z"].append(prob_z.detach().cpu())
            acc["prob_c"].append(prob_c.detach().cpu())

            seen += batch_size

        out_bundle = {}
        for k, v in acc.items():
            if len(v) == 0:
                out_bundle[k] = np.array([])
            elif isinstance(v[0], torch.Tensor):
                out_bundle[k] = torch.cat(v, dim=0).numpy()
            else:
                out_bundle[k] = np.asarray(v, dtype=object)

        return out_bundle

    @torch.no_grad()
    def evaluate(self, loader, prefix="", stage="full"):
        """
        stage:
            - warm: evalúa el warm-up concept-first y usa la cabeza conceptual
            - full: evalúa la pérdida completa y usa la cabeza conceptual

        Además reporta:
            - métricas de ambas cabezas
            - MSE y correlación anatómica
            - entropía media de atención
        """
        self.model.eval()

        loss_sums = {}
        n_batches = 0

        y_true = []
        y_pred_z = []
        y_prob_z = []

        y_pred_c = []
        y_prob_c = []

        c_all = []
        c_target_all = []
        g_bar_all = []
        alpha_all = []

        for batch in loader:
            batch = _move_batch(batch, self.device)
            out = self._forward(batch["x"])

            prob_z = torch.softmax(out["logits"], dim=-1)
            pred_z = prob_z.argmax(dim=-1)

            prob_c = torch.softmax(out["cbm_logits"], dim=-1)
            pred_c = prob_c.argmax(dim=-1)

            y_true.append(batch["y"].detach().cpu())
            y_pred_z.append(pred_z.detach().cpu())
            y_prob_z.append(prob_z.detach().cpu())

            y_pred_c.append(pred_c.detach().cpu())
            y_prob_c.append(prob_c.detach().cpu())

            alpha_all.append(out["alpha"].detach().cpu())
            c_all.append(out["c"].detach().cpu())

            if "c_target" in batch:
                c_target_all.append(batch["c_target"].detach().cpu())
            if "g_bar" in batch:
                g_bar_all.append(batch["g_bar"].detach().cpu())

            if stage == "warm":
                _require_keys(batch, ["c_target", "g_bar"])
                _, info = self.loss_fn.forward_warm(
                    logits_z=out["logits"],
                    logits_c=out["cbm_logits"],
                    labels=batch["y"],
                    c=out["c"],
                    c_target=batch["c_target"],
                    g_bar=batch["g_bar"],
                )
            elif stage == "full":
                _require_keys(batch, ["c_target", "g_bar"])
                _, info = self.loss_fn.forward_full(
                    logits_z=out["logits"],
                    logits_c=out["cbm_logits"],
                    labels=batch["y"],
                    c=out["c"],
                    c_target=batch["c_target"],
                    g_bar=batch["g_bar"],
                )
            else:
                raise ValueError(f"stage must be 'warm' or 'full', got {stage!r}")

            for k, v in info.items():
                loss_sums[k] = loss_sums.get(k, 0.0) + float(v)

            n_batches += 1

        mean_losses = {k: v / max(n_batches, 1) for k, v in loss_sums.items()}

        y_true = torch.cat(y_true).numpy()

        y_pred_z = torch.cat(y_pred_z).numpy()
        y_prob_z = torch.cat(y_prob_z).numpy()
        z_metrics = _classification_metrics_from_outputs(
            y_true=y_true,
            y_pred=y_pred_z,
            y_prob=y_prob_z,
            n_classes=self.model.n_classes,
        )

        y_pred_c = torch.cat(y_pred_c).numpy()
        y_prob_c = torch.cat(y_prob_c).numpy()
        cbm_metrics = _classification_metrics_from_outputs(
            y_true=y_true,
            y_pred=y_pred_c,
            y_prob=y_prob_c,
            n_classes=self.model.n_classes,
        )

        alpha_np = torch.cat(alpha_all).numpy()
        c_np = torch.cat(c_all).numpy()

        concept_mse = float("nan")
        jac_mse = float("nan")
        concept_corr_mean = float("nan")
        jac_corr_mean = float("nan")

        if len(c_target_all) > 0:
            c_target_np = torch.cat(c_target_all).numpy()
            concept_mse = float(np.mean((c_np - c_target_np) ** 2))
            concept_corr_mean = _mean_roi_corr(c_np, c_target_np)

        if len(g_bar_all) > 0:
            g_bar_np = torch.cat(g_bar_all).numpy()
            jac_mse = float(np.mean((c_np - g_bar_np) ** 2))
            jac_corr_mean = _mean_roi_corr(c_np, g_bar_np)

        out_dict = {
            **mean_losses,
            "z_accuracy": z_metrics["accuracy"],
            "z_f1_macro": z_metrics["f1_macro"],
            "z_recall_macro": z_metrics["recall_macro"],
            "z_precision_macro": z_metrics["precision_macro"],
            "z_auc_macro_ovr": z_metrics["auc_macro_ovr"],
            "cbm_accuracy": cbm_metrics["accuracy"],
            "cbm_f1_macro": cbm_metrics["f1_macro"],
            "cbm_recall_macro": cbm_metrics["recall_macro"],
            "cbm_precision_macro": cbm_metrics["precision_macro"],
            "cbm_auc_macro_ovr": cbm_metrics["auc_macro_ovr"],
            "concept_mse": concept_mse,
            "jac_mse": jac_mse,
            "concept_corr_mean": concept_corr_mean,
            "jac_corr_mean": jac_corr_mean,
            "alpha_entropy": _alpha_entropy(alpha_np),
        }

        # En el nuevo warm-up, la cabeza activa también es la conceptual
        active_head = "cbm"
        for metric_name in ["accuracy", "f1_macro", "recall_macro", "precision_macro", "auc_macro_ovr"]:
            out_dict[metric_name] = out_dict[f"{active_head}_{metric_name}"]

        if prefix:
            out_dict = {f"{prefix}_{k}": v for k, v in out_dict.items()}

        return out_dict

    def train_warm_epoch(self, train_loader, epoch):
        self.model.train()
        meter = {"loss": 0.0, "n": 0}

        for step, batch in enumerate(train_loader):
            _require_keys(batch, ["x", "y", "c_target", "g_bar"])
            batch = _move_batch(batch, self.device)

            self.optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=self.device.type,
                enabled=self.cfg.use_amp and self.device.type == "cuda"
            ):
                out = self._forward(batch["x"])
                loss, info = self.loss_fn.forward_warm(
                    logits_z=out["logits"],
                    logits_c=out["cbm_logits"],
                    labels=batch["y"],
                    c=out["c"],
                    c_target=batch["c_target"],
                    g_bar=batch["g_bar"],
                )

            if self.scaler.is_enabled():
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                self.optimizer.step()

            meter["loss"] += float(loss.item())
            meter["n"] += 1

        meter["loss"] /= max(meter["n"], 1)
        return meter

    def fit(self, train_loader, train_eval_loader=None, val_loader=None, external_loader=None):
        history = {"warm": [], "full": []}

        for epoch in range(1, self.cfg.n_epochs_warm + 1):
            train_info = self.train_warm_epoch(train_loader, epoch)

            train_eval = (
                self.evaluate(train_eval_loader, prefix="traincohort", stage="warm")
                if train_eval_loader is not None else {}
            )
            val_eval = (
                self.evaluate(val_loader, prefix="val", stage="warm")
                if val_loader is not None else {}
            )
            ext_eval = (
                self.evaluate(external_loader, prefix="external", stage="warm")
                if external_loader is not None else {}
            )

            epoch_info = {**train_info, **train_eval, **val_eval, **ext_eval}
            history["warm"].append(epoch_info)
            self._maybe_save_best(epoch=epoch, stage="warm", epoch_info=epoch_info)

            if self.analysis_cfg.save_epoch_representations and val_loader is not None:
                bundle = self.extract_bundle(
                    val_loader,
                    max_subjects=self.analysis_cfg.ablation_max_subjects
                )
                _save_npz_dict(
                    str(self.analysis_dir / f"warm_epoch_{epoch:03d}_val_bundle.npz"),
                    bundle,
                )

            print(
                f"[Warm][Epoch {epoch:03d}] "
                f"train_loss={epoch_info['loss']:.4f} | "
                f"TRAIN cbm_f1={epoch_info.get('traincohort_cbm_f1_macro', float('nan')):.4f} | "
                f"VAL cbm_f1={epoch_info.get('val_cbm_f1_macro', float('nan')):.4f} "
                f"anat_corr={epoch_info.get('val_concept_corr_mean', float('nan')):.4f} "
                f"jac_corr={epoch_info.get('val_jac_corr_mean', float('nan')):.4f}"
            )

        for epoch in range(1, self.cfg.n_epochs_full + 1):
            train_info = self.train_full_epoch(train_loader, epoch)

            train_eval = (
                self.evaluate(train_eval_loader, prefix="traincohort", stage="full")
                if train_eval_loader is not None else {}
            )
            val_eval = (
                self.evaluate(val_loader, prefix="val", stage="full")
                if val_loader is not None else {}
            )
            ext_eval = (
                self.evaluate(external_loader, prefix="external", stage="full")
                if external_loader is not None else {}
            )

            epoch_info = {**train_info, **train_eval, **val_eval, **ext_eval}
            history["full"].append(epoch_info)
            self._maybe_save_best(epoch=epoch, stage="full", epoch_info=epoch_info)

            if self.analysis_cfg.save_epoch_representations and val_loader is not None:
                bundle = self.extract_bundle(
                    val_loader,
                    max_subjects=self.analysis_cfg.ablation_max_subjects
                )
                _save_npz_dict(
                    str(self.analysis_dir / f"full_epoch_{epoch:03d}_val_bundle.npz"),
                    bundle,
                )

            print(
                f"[Full][Epoch {epoch:03d}] "
                f"train_loss={epoch_info['loss']:.4f} | "
                f"TRAIN z_f1={epoch_info.get('traincohort_z_f1_macro', float('nan')):.4f} "
                f"cbm_f1={epoch_info.get('traincohort_cbm_f1_macro', float('nan')):.4f} | "
                f"VAL z_f1={epoch_info.get('val_z_f1_macro', float('nan')):.4f} "
                f"cbm_f1={epoch_info.get('val_cbm_f1_macro', float('nan')):.4f} "
                f"anat_corr={epoch_info.get('val_concept_corr_mean', float('nan')):.4f} "
                f"jac_corr={epoch_info.get('val_jac_corr_mean', float('nan')):.4f}"
            )

        with open(self.analysis_dir / "history_analysis.json", "w", encoding="utf-8") as f:
            json.dump(history, f, indent=2)

        with open(self.analysis_dir / "best_paths.json", "w", encoding="utf-8") as f:
            json.dump(self.best_paths, f, indent=2)

        return history


# ------------------------------------------------------------
# Carga de checkpoints
# ------------------------------------------------------------
def load_checkpoint_into_trainer(trainer: AnalysisAwareSupervisedMRITrainer, ckpt_path: str):
    payload = torch.load(ckpt_path, map_location=trainer.device, weights_only=False)
    trainer.model.load_state_dict(payload["model_state_dict"])
    return payload


# ------------------------------------------------------------
# Probes lineales ROI a ROI
# ------------------------------------------------------------
def run_roi_probes(bundle: dict, roi_names=None, n_splits: int = 3, random_state: int = 42):
    """
    Probes:
      - T_k -> c_target_k (R2)
      - U_k -> c_target_k (R2)
      - T_k -> y          (macro-F1)
      - U_k -> y          (macro-F1)
    """
    T = np.asarray(bundle["T"], dtype=np.float32)   # (N, K, Ct)
    U = np.asarray(bundle["U"], dtype=np.float32)   # (N, K, Ct)
    y = np.asarray(bundle["y"]).astype(int).reshape(-1)
    c_target = np.asarray(bundle["c_target"], dtype=np.float32)

    N, K, Ct = T.shape
    if roi_names is None:
        roi_names = [f"ROI_{k:03d}" for k in range(K)]

    rows = []

    # splits para clasificación
    cls_splits = _safe_stratified_splits(y, n_splits)

    for k in range(K):
        row = {
            "roi_idx": int(k),
            "roi_name": roi_names[k],
            "T_to_ctarget_r2": np.nan,
            "U_to_ctarget_r2": np.nan,
            "T_to_y_f1_macro": np.nan,
            "U_to_y_f1_macro": np.nan,
            "delta_U_minus_T_ctarget_r2": np.nan,
            "delta_U_minus_T_y_f1_macro": np.nan,
        }

        y_reg = c_target[:, k]

        # ---------------------------
        # regresión anatómica
        # ---------------------------
        reg_scores_T = []
        reg_scores_U = []

        reg_k = min(n_splits, N)
        if reg_k >= 2 and np.nanstd(y_reg) > 1e-8:
            kf = KFold(n_splits=reg_k, shuffle=True, random_state=random_state)

            for tr, te in kf.split(T[:, k, :]):
                model_T = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
                model_U = make_pipeline(StandardScaler(), Ridge(alpha=1.0))

                model_T.fit(T[tr, k, :], y_reg[tr])
                model_U.fit(U[tr, k, :], y_reg[tr])

                pred_T = model_T.predict(T[te, k, :])
                pred_U = model_U.predict(U[te, k, :])

                reg_scores_T.append(_score_regression_r2(y_reg[te], pred_T))
                reg_scores_U.append(_score_regression_r2(y_reg[te], pred_U))

            row["T_to_ctarget_r2"] = _safe_nanmean(reg_scores_T)
            row["U_to_ctarget_r2"] = _safe_nanmean(reg_scores_U)
            row["delta_U_minus_T_ctarget_r2"] = row["U_to_ctarget_r2"] - row["T_to_ctarget_r2"]

        # ---------------------------
        # clasificación diagnóstica
        # ---------------------------
        cls_scores_T = []
        cls_scores_U = []

        if cls_splits >= 2:
            skf = StratifiedKFold(n_splits=cls_splits, shuffle=True, random_state=random_state)

            for tr, te in skf.split(T[:, k, :], y):
                model_T = make_pipeline(
                    StandardScaler(),
                    LogisticRegression(
                        max_iter=1000,
                        class_weight="balanced",
                        multi_class="auto",
                    ),
                )
                model_U = make_pipeline(
                    StandardScaler(),
                    LogisticRegression(
                        max_iter=1000,
                        class_weight="balanced",
                        multi_class="auto",
                    ),
                )

                model_T.fit(T[tr, k, :], y[tr])
                model_U.fit(U[tr, k, :], y[tr])

                pred_T = model_T.predict(T[te, k, :])
                pred_U = model_U.predict(U[te, k, :])

                cls_scores_T.append(f1_score(y[te], pred_T, average="macro", zero_division=0))
                cls_scores_U.append(f1_score(y[te], pred_U, average="macro", zero_division=0))

            row["T_to_y_f1_macro"] = _safe_nanmean(cls_scores_T)
            row["U_to_y_f1_macro"] = _safe_nanmean(cls_scores_U)
            row["delta_U_minus_T_y_f1_macro"] = row["U_to_y_f1_macro"] - row["T_to_y_f1_macro"]

        rows.append(row)

    df = pd.DataFrame(rows)
    return df.sort_values("delta_U_minus_T_ctarget_r2", ascending=False).reset_index(drop=True)


# ------------------------------------------------------------
# Ablación causal por ROI
# ------------------------------------------------------------
@torch.no_grad()
def run_roi_ablation(
    trainer: AnalysisAwareSupervisedMRITrainer,
    loader,
    stage: str = "U",
    max_subjects: Optional[int] = 64,
    baseline_value: float = 0.0,
    roi_names=None,
):
    """
    stage:
      - "T": anula un token antes del Transformer
      - "U": anula un token contextualizado
      - "c": anula un concepto antes de cbm_head

    Métrica:
      caída media en p(y_true | x) usando la cabeza conceptual.
    """
    trainer.model.eval()
    model = trainer.model
    device = trainer.device

    K = int(model.K)
    if roi_names is None:
        roi_names = [f"ROI_{k:03d}" for k in range(K)]

    sum_drop = np.zeros(K, dtype=np.float64)
    sum_sq_drop = np.zeros(K, dtype=np.float64)
    n_subjects = 0

    for batch in loader:
        batch_size = _batch_size_from_dict(batch)

        if max_subjects is not None:
            remaining = max_subjects - n_subjects
            if remaining <= 0:
                break
            if batch_size > remaining:
                batch = _slice_batch(batch, remaining)
                batch_size = remaining

        batch = _move_batch(batch, device)
        out = trainer._forward(batch["x"])

        y = batch["y"]
        B = int(y.shape[0])
        idx = torch.arange(B, device=device)

        base_prob = torch.softmax(out["cbm_logits"], dim=-1)
        base_true_prob = base_prob[idx, y]

        T0 = out["T"]
        U0 = out["U"]
        c0 = out["c"]

        for k in range(K):
            stage_upper = stage.upper()

            if stage_upper == "T":
                T_ab = T0.clone()
                T_ab[:, k, :] = 0.0

                U_ab = model.ctx_enc(T_ab)
                c_ab, cbm_logits_ab = model.cbm(U_ab)

            elif stage_upper == "U":
                U_ab = U0.clone()
                U_ab[:, k, :] = 0.0

                c_ab, cbm_logits_ab = model.cbm(U_ab)

            elif stage_lower := stage.lower() == "c":
                c_ab = c0.clone()
                c_ab[:, k] = baseline_value
                cbm_logits_ab = model.cbm.cbm_head(c_ab)

            else:
                raise ValueError("stage debe ser 'T', 'U' o 'c'.")

            ab_prob = torch.softmax(cbm_logits_ab, dim=-1)
            ab_true_prob = ab_prob[idx, y]
            drop = (base_true_prob - ab_true_prob).detach().cpu().numpy()

            sum_drop[k] += float(drop.sum())
            sum_sq_drop[k] += float((drop ** 2).sum())

        n_subjects += B

    if n_subjects == 0:
        raise ValueError("No se procesaron sujetos en run_roi_ablation().")

    mean_drop = sum_drop / n_subjects
    var_drop = np.maximum(sum_sq_drop / n_subjects - mean_drop**2, 0.0)
    std_drop = np.sqrt(var_drop)

    df = pd.DataFrame({
        "roi_idx": np.arange(K, dtype=int),
        "roi_name": roi_names,
        "mean_delta_true_prob": mean_drop,
        "std_delta_true_prob": std_drop,
        "n_subjects": n_subjects,
        "ablation_stage": stage,
    })

    return df.sort_values("mean_delta_true_prob", ascending=False).reset_index(drop=True)


# ------------------------------------------------------------
# Resumen post-hoc integral
# ------------------------------------------------------------
def summarize_bundle(bundle: dict):
    y = np.asarray(bundle["y"]).astype(int).reshape(-1)
    prob_z = np.asarray(bundle["prob_z"], dtype=np.float32)
    prob_c = np.asarray(bundle["prob_c"], dtype=np.float32)

    pred_z = prob_z.argmax(axis=1)
    pred_c = prob_c.argmax(axis=1)

    z_metrics = _classification_metrics_from_outputs(
        y_true=y,
        y_pred=pred_z,
        y_prob=prob_z,
        n_classes=prob_z.shape[1],
    )

    cbm_metrics = _classification_metrics_from_outputs(
        y_true=y,
        y_pred=pred_c,
        y_prob=prob_c,
        n_classes=prob_c.shape[1],
    )

    c = np.asarray(bundle["c"], dtype=np.float32)
    c_target = np.asarray(bundle["c_target"], dtype=np.float32)
    g_bar = np.asarray(bundle["g_bar"], dtype=np.float32)
    alpha = np.asarray(bundle["alpha"], dtype=np.float32)

    summary = {
        "n_subjects": int(len(y)),
        "z_accuracy": z_metrics["accuracy"],
        "z_f1_macro": z_metrics["f1_macro"],
        "z_recall_macro": z_metrics["recall_macro"],
        "z_precision_macro": z_metrics["precision_macro"],
        "z_auc_macro_ovr": z_metrics["auc_macro_ovr"],
        "cbm_accuracy": cbm_metrics["accuracy"],
        "cbm_f1_macro": cbm_metrics["f1_macro"],
        "cbm_recall_macro": cbm_metrics["recall_macro"],
        "cbm_precision_macro": cbm_metrics["precision_macro"],
        "cbm_auc_macro_ovr": cbm_metrics["auc_macro_ovr"],
        "concept_mse": float(np.mean((c - c_target) ** 2)),
        "jac_mse": float(np.mean((c - g_bar) ** 2)),
        "concept_corr_mean": _mean_roi_corr(c, c_target),
        "jac_corr_mean": _mean_roi_corr(c, g_bar),
        "alpha_entropy": _alpha_entropy(alpha),
    }
    return summary


def run_full_posthoc_analysis(
    trainer: AnalysisAwareSupervisedMRITrainer,
    loader,
    split_name: str = "val",
    roi_names=None,
    max_subjects: Optional[int] = 64,
):
    analysis_dir = Path(trainer.analysis_dir)
    analysis_dir.mkdir(parents=True, exist_ok=True)

    bundle = trainer.extract_bundle(loader, max_subjects=max_subjects)
    _save_npz_dict(str(analysis_dir / f"{split_name}_bundle.npz"), bundle)

    summary = summarize_bundle(bundle)
    with open(analysis_dir / f"{split_name}_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    probes_df = run_roi_probes(
        bundle,
        roi_names=roi_names,
        n_splits=trainer.analysis_cfg.probes_cv_splits,
        random_state=trainer.analysis_cfg.random_state,
    )
    probes_df.to_csv(analysis_dir / f"{split_name}_roi_probes.csv", index=False)

    ablation_df = run_roi_ablation(
        trainer=trainer,
        loader=loader,
        stage=trainer.analysis_cfg.ablation_stage,
        max_subjects=max_subjects,
        baseline_value=trainer.analysis_cfg.ablation_baseline,
        roi_names=roi_names,
    )
    ablation_df.to_csv(analysis_dir / f"{split_name}_roi_ablation.csv", index=False)

    return {
        "summary": summary,
        "probes_df": probes_df,
        "ablation_df": ablation_df,
        "bundle_path": str(analysis_dir / f"{split_name}_bundle.npz"),
    }
    

def train_model_with_wiring(
    base_dir: str,
    project_root: str,
    module_dir: str,
    atlas_path: Optional[str] = None,
    batch_size: int = 2,
    num_workers: int = 2,
    n_epochs_stage1: int = 5,
    n_epochs_stage2: int = 10,
    lr: float = 1e-4,
    weight_decay: float = 1e-4,
    recompute_artifacts: bool = False,
    save_dir: Optional[str] = None,
):
    atlas_path = find_existing_atlas_path(atlas_path)
    add_module_dir_to_path(module_dir)
    add_module_dir_to_path(project_root)

    source_loader, target_loader, cache = get_domain_adaptation_dataloaders_wired(
        base_dir=base_dir,
        module_dir=module_dir,
        atlas_path=atlas_path,
        batch_size=batch_size,
        num_workers=num_workers,
        recompute_artifacts=recompute_artifacts,
    )

    K = int(cache["K"])

    # from atlas_utils import AtlasROIManager
    # from trainer import DomainAdaptationTrainer, TrainConfig
    # from losses import TotalLoss

    ATLAS_PATH = "/kaggle/working/cerebra_prepared/CerebrA_discrete_resampled_to_reference.nii.gz"

    atlas_mgr = AtlasROIManager(
        atlas_path=ATLAS_PATH,
        config=AtlasConfig(
            label_values=None,         # inferir automáticamente
            drop_background=True,      # quitar label 0
            eps=1e-8,
            min_voxels_per_roi=1,
        )
    )
    
    print(atlas_mgr.summary())
    print("K =", atlas_mgr.K)                 # debe dar 102
    print("atlas_tensor shape =", atlas_mgr.atlas_tensor.shape)  # (102, H, W, D)
    roi_weights = atlas_mgr.roi_weights_from_volume(power=0.0).to(torch.float32).cpu()

    model = build_patched_model(
        project_root=project_root,
        K=K,
        n_classes=len(LABEL_MAP),
    )

    loss_fn = TotalLoss(
        n_classes=len(LABEL_MAP),
        K=K,
        roi_weights=roi_weights,
        tau_p=0.95,
        margin=1.0,
        lambda_cls=0.8,
        lambda_proto=0.95,
        lambda_pl=0.3,
        lambda_cbm=0.5,
        lambda_anat=0.2,
        lambda_sep=0.1,
        label_smoothing=0.1,
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"
    cfg = TrainConfig(
        n_epochs_stage1=n_epochs_stage1,
        n_epochs_stage2=n_epochs_stage2,
        lr=lr,
        weight_decay=weight_decay,
        # num_workers=num_workers,
        device=device,
        log_every=10,
        use_amp=torch.cuda.is_available(),
    )

    trainer = DomainAdaptationTrainer(
        model=model,
        loss_fn=loss_fn,
        atlas_mgr=atlas_mgr,
        input_shape=(128, 128, 128),
        cfg=cfg,
    )
    print("trainer.device =", trainer.device)
    print("model device =", next(trainer.model.parameters()).device)
    print("loss_anat.roi_weights device =", trainer.loss_fn.loss_anat.roi_weights.device)

    history = trainer.fit(
        source_loader=source_loader,
        target_loader=target_loader,
        val_loader=None,
    )

    payload = {
        "K": K,
        "atlas_path": atlas_path,
        "template_x_path": cache["template_x_path"],
        "history": history,
        "train_cfg": asdict(cfg),
    }

    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        ckpt_path = os.path.join(save_dir, "alzheimer_da_cbm.pt")
        hist_path = os.path.join(save_dir, "history.json")
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "K": K,
                "atlas_path": atlas_path,
                "train_cfg": asdict(cfg),
                "history": history,
            },
            ckpt_path,
        )
        with open(hist_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)
        payload["checkpoint_path"] = ckpt_path
        payload["history_path"] = hist_path

    return payload

def train_supervised_cv_fold(
    base_dir: str,
    project_root: str,
    module_dir: str,
    atlas_path: str,
    precomputed_artifacts_dir: str,
    cohort: str = "target",
    n_splits: int = 5,
    fold_idx: int = 0,
    batch_size: int = 2,
    num_workers: int = 0,
    n_epochs_warm: int = 10,
    n_epochs_full: int = 20,
    lr: float = 1e-4,
    weight_decay: float = 1e-4,
):
    atlas_path = find_existing_atlas_path(atlas_path)
    add_module_dir_to_path(module_dir)
    add_module_dir_to_path(project_root)

    cache = load_precomputed_artifacts(
        base_dir=base_dir,
        module_dir=module_dir,
        atlas_path=atlas_path,
        precomputed_artifacts_dir=precomputed_artifacts_dir,
    )

    dataset, labels, df_inventory = build_supervised_dataset_for_cohort(cache, cohort=cohort)

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42,
    )

    splits = list(skf.split(np.arange(len(labels)), labels))
    if fold_idx < 0 or fold_idx >= len(splits):
        raise ValueError(f"fold_idx must be in [0, {len(splits)-1}], got {fold_idx}")

    train_idx, val_idx = splits[fold_idx]

    train_loader = _make_loader_from_subset(
        dataset, train_idx, batch_size=batch_size, shuffle=True, num_workers=num_workers
    )
    train_eval_loader = _make_loader_from_subset(
        dataset, train_idx, batch_size=batch_size, shuffle=False, num_workers=num_workers
    )
    val_loader = _make_loader_from_subset(
        dataset, val_idx, batch_size=batch_size, shuffle=False, num_workers=num_workers
    )

    # evaluación externa opcional: la otra cohorte completa
    external_loader = None
    if cohort == "target":
        external_dataset = SupervisedMRIDatasetWired(
            df_inventory=cache["df_source"],
            df_concepts=cache["df_concepts"],
            df_jac=cache["df_src_jac"],
            K=int(cache["K"]),
        )
        external_loader = DataLoader(
            external_dataset,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
        )
    elif cohort == "source" and cache.get("df_tgt_concepts", None) is not None:
        external_dataset = SupervisedMRIDatasetWired(
            df_inventory=cache["df_target"],
            df_concepts=cache["df_tgt_concepts"],
            df_jac=cache["df_tgt_jac"],
            K=int(cache["K"]),
        )
        external_loader = DataLoader(
            external_dataset,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
        )

    atlas_mgr = AtlasROIManager(atlas_path)
    roi_weights = atlas_mgr.roi_weights_from_volume(power=0.0).to(torch.float32).cpu()

    model = build_patched_model(
        project_root=project_root,
        K=int(cache["K"]),
        n_classes=len(LABEL_MAP),
    )

    loss_fn = SupervisedTotalLoss(
        n_classes=len(LABEL_MAP),
        K=int(cache["K"]),
        roi_weights=roi_weights,
        lambda_z=1.0,
        lambda_c=1.0,
        lambda_cons=0.1,
        lambda_cbm=0.5,
        lambda_anat=0.2,
        label_smoothing=0.1,
    )

    safe_device = pick_safe_device(verbose=True)

    cfg = SupervisedTrainConfig(
        n_epochs_warm=n_epochs_warm,
        n_epochs_full=n_epochs_full,
        lr=lr,
        weight_decay=weight_decay,
        num_workers=num_workers,
        device=safe_device.type,
        log_every=10,
        use_amp=(safe_device.type == "cuda"),
    )

    # trainer = SupervisedMRITrainer(
    #     model=model,
    #     loss_fn=loss_fn,
    #     atlas_mgr=atlas_mgr,
    #     input_shape=(128, 128, 128),
    #     cfg=cfg,
    # )

    analysis_dir = os.path.join(
        "/kaggle/working/analysis_cbm",
        f"{cohort}_fold_{fold_idx}"
    )

    analysis_cfg = AnalysisConfig(
        save_dir=analysis_dir,
        save_best_checkpoints=True,
        save_epoch_representations=False,   # True si quieres guardar bundles por época
        checkpoint_split="val",
        ablation_stage="c",                 # "T", "U" o "c"
        ablation_baseline=0.0,
        ablation_max_subjects=64,           # bájalo a 32 si el tiempo aprieta
        probes_cv_splits=3,
        random_state=42,
    )

    trainer = AnalysisAwareSupervisedMRITrainer(
        model=model,
        loss_fn=loss_fn,
        atlas_mgr=atlas_mgr,
        input_shape=(128, 128, 128),
        cfg=cfg,
        analysis_cfg=analysis_cfg,
    )

    history = trainer.fit(
        train_loader=train_loader,
        train_eval_loader=train_eval_loader,
        val_loader=val_loader,
        external_loader=external_loader,
    )

    # --------------------------------------------------------
    # Cargar el mejor checkpoint conceptual y correr análisis
    # --------------------------------------------------------
    if trainer.best_paths["cbm"] is not None:
        load_checkpoint_into_trainer(trainer, trainer.best_paths["cbm"])

    roi_names = [f"ROI_{k:03d}" for k in range(int(cache["K"]))]

    posthoc_val = run_full_posthoc_analysis(
        trainer=trainer,
        loader=val_loader,
        split_name="val",
        roi_names=roi_names,
        max_subjects=analysis_cfg.ablation_max_subjects,
    )

    posthoc_external = None
    if external_loader is not None:
        posthoc_external = run_full_posthoc_analysis(
            trainer=trainer,
            loader=external_loader,
            split_name="external",
            roi_names=roi_names,
            max_subjects=analysis_cfg.ablation_max_subjects,
        )

    return {
        "history": history,
        "fold_idx": fold_idx,
        "cohort": cohort,
        "n_train": len(train_idx),
        "n_val": len(val_idx),
        "analysis_dir": analysis_dir,
        "best_paths": trainer.best_paths,
        "best_scores": trainer.best_scores,
        "posthoc_val_summary": posthoc_val["summary"],
        "posthoc_external_summary": None if posthoc_external is None else posthoc_external["summary"],
    }

    # history = trainer.fit(
    #     train_loader=train_loader,
    #     train_eval_loader=train_eval_loader,
    #     val_loader=val_loader,
    #     external_loader=external_loader,
        
    # )

    # return {
    #     "history": history,
    #     "fold_idx": fold_idx,
    #     "cohort": cohort,
    #     "n_train": len(train_idx),
    #     "n_val": len(val_idx),
    # }

def run_supervised_cv_for_cohort(
    base_dir: str,
    project_root: str,
    module_dir: str,
    atlas_path: str,
    precomputed_artifacts_dir: str,
    cohort: str = "target",
    n_splits: int = 5,
    batch_size: int = 2,
    num_workers: int = 0,
    n_epochs_warm: int = 10,
    n_epochs_full: int = 20,
    lr: float = 1e-4,
    weight_decay: float = 1e-4,
):
    all_results = []

    for fold_idx in range(n_splits):
        print("\n" + "=" * 80)
        print(f"Running CV fold {fold_idx+1}/{n_splits} on cohort={cohort}")
        print("=" * 80)

        fold_result = train_supervised_cv_fold(
            base_dir=base_dir,
            project_root=project_root,
            module_dir=module_dir,
            atlas_path=atlas_path,
            precomputed_artifacts_dir=precomputed_artifacts_dir,
            cohort=cohort,
            n_splits=n_splits,
            fold_idx=fold_idx,
            batch_size=batch_size,
            num_workers=num_workers,
            n_epochs_warm=n_epochs_warm,
            n_epochs_full=n_epochs_full,
            lr=lr,
            weight_decay=weight_decay,
        )
        all_results.append(fold_result)

    return all_results

In [ ]:
"""
wire_and_train_precomputed.py
=============================
Patched wiring script for two-stage usage:

1) Artifact notebook:
   - generate source concept targets
   - generate source Jacobian priors
   - generate target Jacobian priors
   - save CSV indices in a writable directory

2) Training notebook:
   - reuse an existing precomputed_artifacts_dir
   - skip all expensive artifact generation
   - only read MRI tensors + precomputed vectors

Relative to the original wiring file, the key new capability is:
    precomputed_artifacts_dir=...   -> load cached artifacts directly

This file remains compatible with the original on-the-fly mode through:
    recompute_artifacts=False/True
    cache_dir=...
"""

from __future__ import annotations

import glob
import hashlib
import json
import os
import sys
from dataclasses import asdict
from pathlib import Path
from typing import Dict, Optional

import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Subset
import pandas as pd
import numpy as np
import os


LABEL_MAP = {"CN": 0, "MCI": 1, "AD": 2}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}


def add_module_dir_to_path(module_dir: str | os.PathLike) -> None:
    module_dir = str(module_dir)
    if module_dir not in sys.path:
        sys.path.insert(0, module_dir)


def load_tensor_like(obj_path: str, expected_ndim=None, prefer_keys=None) -> torch.Tensor:
    obj = torch.load(obj_path, map_location="cpu", weights_only=False)

    if prefer_keys is None:
        prefer_keys = ["x", "g_bar", "c_target", "image", "mri", "tensor", "volume"]

    if torch.is_tensor(obj):
        x = obj
    elif isinstance(obj, dict):
        x = None
        for key in prefer_keys:
            if key in obj:
                x = obj[key]
                break
        if x is None:
            raise KeyError(f"No se encontró tensor válido en {obj_path}")
    else:
        x = torch.as_tensor(obj)

    if not torch.is_tensor(x):
        x = torch.as_tensor(x)

    x = x.detach().to(torch.float32)
    x = torch.tensor(x.cpu().numpy(), dtype=torch.float32).contiguous()

    if expected_ndim is not None and x.ndim != expected_ndim:
        raise ValueError(
            f"Se esperaba tensor con ndim={expected_ndim}, llegó shape={tuple(x.shape)} desde {obj_path}"
        )

    return x


def build_source_path_map() -> Dict[str, str]:
    path_map = {}
    for f in glob.glob("/kaggle/input/**/*.pt", recursive=True):
        fl = f.lower()
        if any(tok in fl for tok in ["c_target", "g_bar", "g_jacobian", "target_oasis", "oasis"]):
            continue
        basename = os.path.basename(f).replace(".pt", "")
        path_map[basename] = f
    return path_map


def resolve_source_x_path(row: pd.Series, source_path_map: Dict[str, str]) -> str:
    sub_id = str(row["Subject_ID"])
    if sub_id in source_path_map:
        return source_path_map[sub_id]
    for col in ["File_Path", "Raw_File_Path", "Processed_File_Path", "x_path"]:
        if col in row and pd.notna(row[col]) and os.path.exists(str(row[col])):
            return str(row[col])
    raise FileNotFoundError(f"Could not resolve source MRI path for Subject_ID={sub_id}")


def resolve_target_x_path(row: pd.Series, base_dir: str) -> str:
    sub_id = str(row["Subject_ID"])
    label = str(row["Label"])

    candidate = os.path.join(base_dir, "target_oasis", label, f"{sub_id}_MRI.pt")
    if os.path.exists(candidate):
        return candidate

    for col in ["Processed_File_Path", "File_Path", "Raw_File_Path", "x_path"]:
        if col in row and pd.notna(row[col]) and os.path.exists(str(row[col])):
            return str(row[col])
    raise FileNotFoundError(f"Could not resolve target MRI path for Subject_ID={sub_id}")


def build_inventory_dataframes(base_dir: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    source_csv = os.path.join(base_dir, "source_labels.csv")
    target_csv = os.path.join(base_dir, "target_labels.csv")

    if not os.path.exists(source_csv):
        raise FileNotFoundError(f"Missing source CSV: {source_csv}")
    if not os.path.exists(target_csv):
        raise FileNotFoundError(f"Missing target CSV: {target_csv}")

    df_source = pd.read_csv(source_csv)
    df_target = pd.read_csv(target_csv)

    df_source = df_source[df_source["Label"].isin(LABEL_MAP)].reset_index(drop=True)
    df_target = df_target[df_target["Label"].isin(LABEL_MAP)].reset_index(drop=True)

    source_path_map = build_source_path_map()

    df_source = df_source.copy()
    df_source["subject_id"] = df_source["Subject_ID"].astype(str)
    df_source["label"] = df_source["Label"].astype(str)
    df_source["x_path"] = df_source.apply(lambda r: resolve_source_x_path(r, source_path_map), axis=1)

    df_target = df_target.copy()
    df_target["subject_id"] = df_target["Subject_ID"].astype(str)
    df_target["label"] = df_target["Label"].astype(str)
    df_target["x_path"] = df_target.apply(lambda r: resolve_target_x_path(r, base_dir), axis=1)

    return df_source[["subject_id", "label", "x_path"]], df_target[["subject_id", "label", "x_path"]]


def find_existing_atlas_path(explicit_atlas_path: Optional[str] = None) -> str:
    if explicit_atlas_path is not None and os.path.exists(explicit_atlas_path):
        return explicit_atlas_path

    patterns = [
        "/kaggle/input/**/*atlas*.nii*",
        "/kaggle/input/**/*aal*.nii*",
        "/kaggle/input/**/*harvard*oxford*.nii*",
        "/kaggle/input/**/*label*.nii*",
        "/kaggle/working/**/*atlas*.nii*",
    ]
    hits = []
    for pat in patterns:
        hits.extend(glob.glob(pat, recursive=True))
    hits = sorted(set(hits))
    if not hits:
        raise FileNotFoundError("Atlas file not found automatically. Please pass atlas_path explicitly.")
    return hits[0]


def ensure_simpleitk_or_raise() -> None:
    try:
        import SimpleITK  # noqa: F401
    except Exception as e:
        raise ImportError(
            "SimpleITK is required to compute g_bar Jacobian priors. "
            "Install it in Kaggle before running artifact generation."
        ) from e


def choose_template_x_path(df_source: pd.DataFrame) -> str:
    cn = df_source[df_source["label"] == "CN"].reset_index(drop=True)
    if len(cn) == 0:
        return str(df_source.iloc[0]["x_path"])
    return str(cn.iloc[0]["x_path"])


def _safe_name_from_path(path: str) -> str:
    base = os.path.basename(os.path.normpath(path)) or "dataset"
    digest = hashlib.md5(path.encode("utf-8")).hexdigest()[:8]
    return f"{base}_{digest}"


def _default_cache_dir(base_dir: str) -> str:
    return os.path.join("/kaggle/working", "derived_artifacts", _safe_name_from_path(base_dir))


def _validate_index_columns(df: pd.DataFrame, required: list[str], name: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")


def _validate_subject_coverage(df_inventory: pd.DataFrame, df_index: pd.DataFrame, name: str) -> None:
    inv = set(df_inventory["subject_id"].astype(str).tolist())
    idx = set(df_index["subject_id"].astype(str).tolist())
    missing = sorted(inv - idx)
    if missing:
        head = missing[:10]
        raise ValueError(
            f"{name} does not cover all inventory subjects. Missing={len(missing)}. "
            f"First missing: {head}"
        )


def ensure_artifact_cache(
    base_dir: str,
    module_dir: str,
    atlas_path: str,
    recompute: bool = False,
    cache_dir: Optional[str] = None,
) -> dict:
    add_module_dir_to_path(module_dir)

    from atlas_utils import AtlasROIManager
    from concept_targets import ConceptTargetConfig, precompute_concept_targets_from_dataframe
    from jacobian_utils import JacobianConfig, precompute_jacobians_from_dataframe

    df_source, df_target = build_inventory_dataframes(base_dir)

    if cache_dir is None:
        cache_dir = _default_cache_dir(base_dir)

    artifacts_dir = cache_dir
    src_concepts_dir = os.path.join(artifacts_dir, "source_concepts")
    src_jac_dir = os.path.join(artifacts_dir, "source_jacobians")
    tgt_jac_dir = os.path.join(artifacts_dir, "target_jacobians")
    os.makedirs(src_concepts_dir, exist_ok=True)
    os.makedirs(src_jac_dir, exist_ok=True)
    os.makedirs(tgt_jac_dir, exist_ok=True)

    atlas_mgr = AtlasROIManager(atlas_path)
    template_x_path = choose_template_x_path(df_source)

    source_inventory_csv = os.path.join(artifacts_dir, "source_inventory.csv")
    target_inventory_csv = os.path.join(artifacts_dir, "target_inventory.csv")
    source_concepts_csv = os.path.join(artifacts_dir, "source_concepts_index.csv")
    source_jac_csv = os.path.join(artifacts_dir, "source_jacobians_index.csv")
    target_jac_csv = os.path.join(artifacts_dir, "target_jacobians_index.csv")
    cache_meta_json = os.path.join(artifacts_dir, "cache_meta.json")

    df_source.to_csv(source_inventory_csv, index=False)
    df_target.to_csv(target_inventory_csv, index=False)

    if recompute or not os.path.exists(source_concepts_csv):
        _, df_concepts = precompute_concept_targets_from_dataframe(
            df=df_source,
            atlas_mgr=atlas_mgr,
            x_column="x_path",
            label_column="label",
            subject_id_column="subject_id",
            output_dir=src_concepts_dir,
            cfg=ConceptTargetConfig(normal_class_name="CN"),
        )
        df_concepts.to_csv(source_concepts_csv, index=False)
    else:
        df_concepts = pd.read_csv(source_concepts_csv)

    if recompute or not os.path.exists(source_jac_csv):
        ensure_simpleitk_or_raise()
        df_src_jac = precompute_jacobians_from_dataframe(
            df=df_source,
            atlas_mgr=atlas_mgr,
            template_x_path=template_x_path,
            x_column="x_path",
            subject_id_column="subject_id",
            output_dir=src_jac_dir,
            cfg=JacobianConfig(),
        )
        df_src_jac.to_csv(source_jac_csv, index=False)
    else:
        df_src_jac = pd.read_csv(source_jac_csv)

    if recompute or not os.path.exists(target_jac_csv):
        ensure_simpleitk_or_raise()
        df_tgt_jac = precompute_jacobians_from_dataframe(
            df=df_target,
            atlas_mgr=atlas_mgr,
            template_x_path=template_x_path,
            x_column="x_path",
            subject_id_column="subject_id",
            output_dir=tgt_jac_dir,
            cfg=JacobianConfig(),
        )
        df_tgt_jac.to_csv(target_jac_csv, index=False)
    else:
        df_tgt_jac = pd.read_csv(target_jac_csv)

    meta = {
        "base_dir": base_dir,
        "cache_dir": artifacts_dir,
        "atlas_path": atlas_path,
        "template_x_path": template_x_path,
        "K": int(atlas_mgr.K),
        "n_source": int(len(df_source)),
        "n_target": int(len(df_target)),
    }
    with open(cache_meta_json, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    return {
        "atlas_path": atlas_path,
        "template_x_path": template_x_path,
        "K": atlas_mgr.K,
        "df_source": df_source,
        "df_target": df_target,
        "df_concepts": df_concepts,
        "df_src_jac": df_src_jac,
        "df_tgt_jac": df_tgt_jac,
        "source_inventory_csv": source_inventory_csv,
        "target_inventory_csv": target_inventory_csv,
        "source_concepts_csv": source_concepts_csv,
        "source_jac_csv": source_jac_csv,
        "target_jac_csv": target_jac_csv,
        "cache_dir": artifacts_dir,
        "cache_meta_json": cache_meta_json,
    }

def _resolve_artifact_path(old_path: str, artifacts_dir: str, subdir: str) -> str:
    """
    Si el path serializado ya no existe (porque viene de /kaggle/working del notebook
    anterior), lo remapea al dataset montado actual dentro de precomputed_artifacts_dir.
    """
    old_path = str(old_path)

    if os.path.exists(old_path):
        return old_path

    fname = os.path.basename(old_path)

    candidates = [
        os.path.join(artifacts_dir, subdir, fname),
        os.path.join(artifacts_dir, fname),
    ]

    for c in candidates:
        if os.path.exists(c):
            return c

    matches = glob.glob(os.path.join(artifacts_dir, "**", fname), recursive=True)
    if matches:
        return matches[0]

    raise FileNotFoundError(
        f"Artifact file not found.\n"
        f"Original serialized path: {old_path}\n"
        f"Searched under: {artifacts_dir}"
    )


def _remap_artifact_dataframe_paths(df: pd.DataFrame, col: str, artifacts_dir: str, subdir: str) -> pd.DataFrame:
    df = df.copy()
    df[col] = df[col].apply(lambda p: _resolve_artifact_path(p, artifacts_dir, subdir))
    return df


import os
import glob
import pandas as pd

def _resolve_artifact_path(old_path: str, artifacts_dir: str, subdir: str) -> str:
    """
    Remapea rutas serializadas antiguas de /kaggle/working al dataset montado actual.
    """
    old_path = str(old_path)

    # caso ideal: la ruta serializada todavía existe
    if os.path.exists(old_path):
        return old_path

    fname = os.path.basename(old_path)

    # búsqueda determinista primero
    candidates = [
        os.path.join(artifacts_dir, subdir, fname),
        os.path.join(artifacts_dir, fname),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c

    # búsqueda recursiva final
    matches = glob.glob(os.path.join(artifacts_dir, "**", fname), recursive=True)
    if matches:
        return matches[0]

    raise FileNotFoundError(
        f"No se encontró el artefacto.\n"
        f"Ruta serializada original: {old_path}\n"
        f"Directorio de artefactos actual: {artifacts_dir}"
    )


def _remap_artifact_dataframe_paths(df: pd.DataFrame, col: str, artifacts_dir: str, subdir: str) -> pd.DataFrame:
    df = df.copy()
    df[col] = df[col].apply(lambda p: _resolve_artifact_path(p, artifacts_dir, subdir))
    return df

def load_precomputed_artifacts(
    base_dir: str,
    module_dir: str,
    atlas_path: str,
    precomputed_artifacts_dir: str,
) -> dict:
    add_module_dir_to_path(module_dir)

    artifacts_dir = str(precomputed_artifacts_dir)
    if not os.path.isdir(artifacts_dir):
        raise FileNotFoundError(f"precomputed_artifacts_dir no existe: {artifacts_dir}")

    source_concepts_csv = os.path.join(artifacts_dir, "source_concepts_index.csv")
    source_jac_csv      = os.path.join(artifacts_dir, "source_jacobians_index.csv")
    target_jac_csv      = os.path.join(artifacts_dir, "target_jacobians_index.csv")
    target_concepts_csv = os.path.join(artifacts_dir, "target_concepts_index.csv")

    source_inventory_csv = os.path.join(artifacts_dir, "source_inventory.csv")
    target_inventory_csv = os.path.join(artifacts_dir, "target_inventory.csv")

    for p in [source_concepts_csv, source_jac_csv, target_jac_csv]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Falta archivo requerido: {p}")

    if os.path.exists(source_inventory_csv) and os.path.exists(target_inventory_csv):
        df_source = pd.read_csv(source_inventory_csv)
        df_target = pd.read_csv(target_inventory_csv)
    else:
        df_source, df_target = build_inventory_dataframes(base_dir)

    df_concepts = pd.read_csv(source_concepts_csv)
    df_src_jac  = pd.read_csv(source_jac_csv)
    df_tgt_jac  = pd.read_csv(target_jac_csv)

    df_tgt_concepts = None
    if os.path.exists(target_concepts_csv):
        df_tgt_concepts = pd.read_csv(target_concepts_csv)

    # remapeo de rutas antiguas serializadas
    df_concepts = _remap_artifact_dataframe_paths(
        df_concepts, "concept_target_path", artifacts_dir, "source_concepts"
    )
    df_src_jac = _remap_artifact_dataframe_paths(
        df_src_jac, "g_bar_path", artifacts_dir, "source_jacobians"
    )
    df_tgt_jac = _remap_artifact_dataframe_paths(
        df_tgt_jac, "g_bar_path", artifacts_dir, "target_jacobians"
    )

    if df_tgt_concepts is not None:
        df_tgt_concepts = _remap_artifact_dataframe_paths(
            df_tgt_concepts, "concept_target_path", artifacts_dir, "target_concepts"
        )

    atlas_mgr = AtlasROIManager(atlas_path)
    template_x_path = choose_template_x_path(df_source)

    return {
        "atlas_path": atlas_path,
        "template_x_path": template_x_path,
        "K": int(atlas_mgr.K),
        "df_source": df_source,
        "df_target": df_target,
        "df_concepts": df_concepts,
        "df_src_jac": df_src_jac,
        "df_tgt_jac": df_tgt_jac,
        "df_tgt_concepts": df_tgt_concepts,
        "cache_dir": artifacts_dir,
    }

def _index_by_subject(df: pd.DataFrame, path_col: str) -> Dict[str, str]:
    out = {}
    for _, row in df.iterrows():
        out[str(row["subject_id"])] = str(row[path_col])
    return out


def _load_vector(path: str, expected_last_dim: int) -> torch.Tensor:
    obj = torch.load(path, map_location="cpu", weights_only=False)

    if torch.is_tensor(obj):
        v = obj

    elif isinstance(obj, dict):
        v = None
        for key in ["c_target", "g_bar", "x", "tensor", "vector"]:
            if key in obj:
                v = obj[key]
                break
        if v is None:
            raise KeyError(
                f"No se encontró ningún vector válido en {path}. "
                f"Keys disponibles: {list(obj.keys())}"
            )
    else:
        raise TypeError(f"Unsupported object at {path}: {type(obj)}")

    if not torch.is_tensor(v):
        v = torch.as_tensor(v)

    v = v.detach().to(torch.float32).view(-1)

    if v.numel() != expected_last_dim:
        raise ValueError(
            f"Expected vector with K={expected_last_dim} at {path}, got shape {tuple(v.shape)}"
        )
    # romper cualquier posible dependencia residual con MetaTensor
    v = torch.tensor(v.cpu().numpy(), dtype=torch.float32)
    return v


# class SourceDomainDatasetWired(Dataset):
#     def __init__(self, df_source, df_concepts, df_src_jac, K: int):
#         super().__init__()
#         self.data = df_source.reset_index(drop=True)
#         self.K = int(K)
#         self.c_map = _index_by_subject(df_concepts, "concept_target_path")
#         self.g_map = _index_by_subject(df_src_jac, "g_bar_path")

#     def __len__(self):
#         return len(self.data)

#     def __getitem__(self, idx):
#         row = self.data.iloc[idx]
#         sub_id = str(row["subject_id"])
#         label_str = str(row["label"])

#         x = load_tensor_like(str(row["x_path"]))
#         y = torch.tensor(LABEL_MAP[label_str], dtype=torch.long)
#         c_target = _load_vector(self.c_map[sub_id], expected_last_dim=self.K)
#         g_bar = _load_vector(self.g_map[sub_id], expected_last_dim=self.K)

#         return {
#             "x": x,
#             "y": y,
#             "c_target": c_target,
#             "g_bar": g_bar,
#             "subject_id": sub_id,
#             "label_name": label_str,
#         }


# class TargetDomainDatasetWired(Dataset):
#     def __init__(self, df_target, df_tgt_jac, K: int):
#         super().__init__()
#         self.data = df_target.reset_index(drop=True)
#         self.K = int(K)
#         self.g_map = _index_by_subject(df_tgt_jac, "g_bar_path")

#     def __len__(self):
#         return len(self.data)

#     def __getitem__(self, idx):
#         row = self.data.iloc[idx]
#         sub_id = str(row["subject_id"])
#         label_str = str(row["label"])

#         x = load_tensor_like(str(row["x_path"]))
#         y = torch.tensor(LABEL_MAP[label_str], dtype=torch.long)
#         g_bar = _load_vector(self.g_map[sub_id], expected_last_dim=self.K)

#         return {
#             "x": x,
#             "y": y,
#             "g_bar": g_bar,
#             "subject_id": sub_id,
#             "label_name": label_str,
#         }

class SupervisedMRIDatasetWired(Dataset):
    def __init__(self, df_inventory, df_concepts, df_jac, K: int):
        super().__init__()
        self.data = df_inventory.reset_index(drop=True)
        self.K = int(K)
        self.c_map = _index_by_subject(df_concepts, "concept_target_path")
        self.g_map = _index_by_subject(df_jac, "g_bar_path")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        sub_id = str(row["subject_id"])
        label_str = str(row["label"])

        x = load_tensor_like(str(row["x_path"]), expected_ndim=4, prefer_keys=["x"])
        y = torch.tensor(LABEL_MAP[label_str], dtype=torch.long)
        c_target = _load_vector(self.c_map[sub_id], expected_last_dim=self.K)
        g_bar = _load_vector(self.g_map[sub_id], expected_last_dim=self.K)

        return {
            "x": x,
            "y": y,
            "c_target": c_target,
            "g_bar": g_bar,
            "subject_id": sub_id,
            "label_name": label_str,
        }

def get_supervised_dataloaders_wired(
    base_dir: str,
    module_dir: str,
    atlas_path: str,
    batch_size: int = 2,
    num_workers: int = 2,
    precomputed_artifacts_dir: Optional[str] = None,
):
    cache = load_precomputed_artifacts(
        base_dir=base_dir,
        module_dir=module_dir,
        atlas_path=atlas_path,
        precomputed_artifacts_dir=precomputed_artifacts_dir,
    )

    K = int(cache["K"])

    source_dataset = SupervisedMRIDatasetWired(
        df_inventory=cache["df_source"],
        df_concepts=cache["df_concepts"],
        df_jac=cache["df_src_jac"],
        K=K,
    )

    target_dataset = SupervisedMRIDatasetWired(
        df_inventory=cache["df_target"],
        df_concepts=cache["df_tgt_concepts"],
        df_jac=cache["df_tgt_jac"],
        K=K,
    )

    use_pin_memory = torch.cuda.is_available()

    source_train_loader = DataLoader(
        source_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )
    source_eval_loader = DataLoader(
        source_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    target_train_loader = DataLoader(
        target_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )
    target_eval_loader = DataLoader(
        target_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    return {
        "source_train": source_train_loader,
        "source_eval": source_eval_loader,
        "target_train": target_train_loader,
        "target_eval": target_eval_loader,
        "cache": cache,
    }

def build_patched_model(
    project_root: str,
    K: int,
    n_classes: int = 3,
    C_f: int = 256,
    C_t: int = 128,
    n_heads: int = 4,
    n_layers: int = 2,
    base_ch: int = 32,
):
    add_module_dir_to_path(project_root)
    # from model import AlzheimerDomainAdaptationModel
    # from model_patch_concept import ConceptBottleneck as PatchedConceptBottleneck

    model = AlzheimerDomainAdaptationModel(
        K=K,
        C_f=C_f,
        C_t=C_t,
        n_classes=n_classes,
        n_heads=n_heads,
        n_layers=n_layers,
        base_ch=base_ch,
    )
    model.cbm = ConceptBottleneck(K=K, C_t=C_t, n_classes=n_classes)
    model.K = K
    return model


def train_supervised_model_with_wiring(
    base_dir: str,
    project_root: str,
    module_dir: str,
    atlas_path: Optional[str] = None,
    batch_size: int = 2,
    num_workers: int = 2,
    n_epochs_warm: int = 20,
    n_epochs_full: int = 30,
    lr: float = 3e-4,
    weight_decay: float = 1e-4,
    precomputed_artifacts_dir: Optional[str] = None,
    train_domain: str = "source",   # "source" o "target"
):
    atlas_path = find_existing_atlas_path(atlas_path)
    add_module_dir_to_path(module_dir)
    add_module_dir_to_path(project_root)

    loaders = get_supervised_dataloaders_wired(
        base_dir=base_dir,
        module_dir=module_dir,
        atlas_path=atlas_path,
        batch_size=batch_size,
        num_workers=num_workers,
        precomputed_artifacts_dir=precomputed_artifacts_dir,
    )

    cache = loaders["cache"]
    K = int(cache["K"])

    atlas_mgr = AtlasROIManager(atlas_path)
    roi_weights = atlas_mgr.roi_weights_from_volume(power=0.0).to(torch.float32).cpu()

    model = build_patched_model(
        project_root=project_root,
        K=K,
        n_classes=len(LABEL_MAP)
    )

    # loss_fn = SupervisedTotalLoss(
    #     n_classes=len(LABEL_MAP),
    #     K=K,
    #     roi_weights=roi_weights,
    #     lambda_z    = 1.0,
    #     lambda_c    = 1.0,
    #     lambda_cons = 0.05,
    #     lambda_cbm  = 0.1,
    #     lambda_anat = 0.052,
    #     label_smoothing=0.1,
    # )

    loss_fn = SupervisedTotalLoss(
        n_classes=n_classes,
        K=K,
        roi_weights=roi_weights,
        lambda_z=0.5,
        lambda_c=1.0,
        lambda_cons=0.1,
        lambda_cbm=0.5,
        lambda_anat=0.2,
        warm_lambda_z=0.1,
        warm_lambda_c=1.0,
        warm_lambda_cbm=1.0,
        warm_lambda_anat=1.0,
        label_smoothing=0.1,
    )

    try:
        if torch.cuda.is_available():
            _x = torch.zeros(1, device="cuda")
            _ = _x + 1
            safe_device = torch.device("cuda")
            print(f"[Device] CUDA usable: {torch.cuda.get_device_name(0)}")
        else:
            safe_device = torch.device("cpu")
            print("[Device] CUDA no disponible. Se usará CPU.")
    except Exception as e:
        safe_device = torch.device("cpu")
        print("[Device] CUDA detectada pero no usable con la build actual de PyTorch.")
        print(f"[Device] Fallback a CPU. Motivo: {type(e).__name__}: {e}")

    cfg = SupervisedTrainConfig(
        n_epochs_warm=n_epochs_warm,
        n_epochs_full=n_epochs_full,
        lr=lr,
        weight_decay=weight_decay,
        device=safe_device.type,
        log_every=10,
        use_amp=(safe_device.type == "cuda"),
    )

    trainer = SupervisedMRITrainer(
        model=model,
        loss_fn=loss_fn,
        atlas_mgr=atlas_mgr,
        input_shape=(128, 128, 128),
        cfg=cfg,
    )

    if train_domain == "source":
        train_loader = loaders["source_train"]
    elif train_domain == "target":
        train_loader = loaders["target_train"]
    else:
        raise ValueError(f"train_domain must be 'source' or 'target', got {train_domain!r}")

    history = trainer.fit(
        train_loader=train_loader,
        source_eval_loader=loaders["source_eval"],
        target_eval_loader=loaders["target_eval"],
    )

    return {
        "history": history,
        "train_domain": train_domain,
        "K": K,
        "atlas_path": atlas_path,
    }

def build_supervised_dataset_for_cohort(cache: dict, cohort: str):
    cohort = str(cohort).lower()

    if cohort == "source":
        df_inventory = cache["df_source"]
        df_concepts  = cache["df_concepts"]
        df_jac       = cache["df_src_jac"]

    elif cohort == "target":
        if cache.get("df_tgt_concepts", None) is None:
            raise ValueError(
                "No existe df_tgt_concepts en los artefactos precomputados. "
                "Para hacer CV supervisado completo sobre target debes generar "
                "también target_concepts_index.csv."
            )
        df_inventory = cache["df_target"]
        df_concepts  = cache["df_tgt_concepts"]
        df_jac       = cache["df_tgt_jac"]

    else:
        raise ValueError(f"cohort must be 'source' or 'target', got {cohort!r}")

    dataset = SupervisedMRIDatasetWired(
        df_inventory=df_inventory,
        df_concepts=df_concepts,
        df_jac=df_jac,
        K=int(cache["K"]),
    )

    labels = df_inventory["label"].map(LABEL_MAP).to_numpy()
    return dataset, labels, df_inventory


def _make_loader_from_subset(dataset, indices, batch_size, shuffle, num_workers):
    use_pin_memory = torch.cuda.is_available()
    subset = Subset(dataset, indices)
    return DataLoader(
        subset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=shuffle,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

In [ ]:
"""
build_precomputed_artifacts.py
==============================
Standalone notebook-friendly builder for the expensive anatomical artifacts:

  - source concept targets        c_target
  - source Jacobian priors        g_bar
  - target Jacobian priors        g_bar
  - CSV indices                   source_concepts_index.csv,
                                  source_jacobians_index.csv,
                                  target_jacobians_index.csv

Typical use:

    from build_precomputed_artifacts import build_all_precomputed_artifacts

    result = build_all_precomputed_artifacts(
        base_dir="/kaggle/input/notebooks/alejopatio/preprocess-alzheimer/model_ready_data",
        module_dir="/kaggle/working/mri_da_missing",
        atlas_path="/kaggle/working/cerebra_prepared/CerebrA_discrete_resampled_to_reference.nii.gz",
        output_dir="/kaggle/working/precomputed_artifacts_cerebra",
        recompute=False,
    )
    print(result)

Then, in a separate training notebook, call:

    train_model_with_wiring(..., precomputed_artifacts_dir=result["artifacts_dir"])
"""

from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Optional

import pandas as pd

def load_mri_tensor(filepath: str) -> torch.Tensor:
    img = nib.load(filepath)
    img = nib.as_closest_canonical(img)
    vol = img.get_fdata(dtype=np.float32)

    if vol.ndim == 4:
        vol = vol[..., 0]

    vol = np.nan_to_num(vol, nan=0.0, posinf=0.0, neginf=0.0)
    x = torch.from_numpy(vol).unsqueeze(0)   # (1, H, W, D)
    x = mri_transforms(x)                    # puede salir como MetaTensor
    x = to_plain_tensor(x, ensure_channel_dim=True)   # <- PARCHE CLAVE
    return x


def build_all_precomputed_artifacts(
    base_dir: str,
    module_dir: str,
    atlas_path: Optional[str] = None,
    output_dir: str = "/kaggle/working/precomputed_artifacts",
    recompute: bool = False,
    source_concepts_subdir: str = "source_concepts",
    source_jacobians_subdir: str = "source_jacobians",
    target_jacobians_subdir: str = "target_jacobians",
) -> dict:
    """
    Builds and stores all expensive precomputed artifacts in a writable directory.

    Parameters
    ----------
    base_dir:
        Read-only directory containing source_labels.csv, target_labels.csv,
        target_oasis/, etc.
    module_dir:
        Directory containing atlas_utils.py, concept_targets.py, jacobian_utils.py.
    atlas_path:
        Prepared discrete atlas path. If None, auto-discovery is attempted.
    output_dir:
        Writable directory where all precomputed artifacts and index CSVs are stored.
    recompute:
        If True, overwrite existing CSV indices and per-subject artifact files.
    """
    atlas_path = find_existing_atlas_path(atlas_path)
    add_module_dir_to_path(module_dir)

    # from atlas_utils import AtlasROIManager
    # from concept_targets import ConceptTargetConfig, precompute_concept_targets_from_dataframe
    # from jacobian_utils import JacobianConfig, precompute_jacobians_from_dataframe

    output_dir = str(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    src_concepts_dir = os.path.join(output_dir, source_concepts_subdir)
    src_jac_dir = os.path.join(output_dir, source_jacobians_subdir)
    tgt_jac_dir = os.path.join(output_dir, target_jacobians_subdir)
    os.makedirs(src_concepts_dir, exist_ok=True)
    os.makedirs(src_jac_dir, exist_ok=True)
    os.makedirs(tgt_jac_dir, exist_ok=True)

    df_source, df_target = build_inventory_dataframes(base_dir)
    atlas_mgr = AtlasROIManager(atlas_path)
    template_x_path = choose_template_x_path(df_source)

    source_inventory_csv = os.path.join(output_dir, "source_inventory.csv")
    target_inventory_csv = os.path.join(output_dir, "target_inventory.csv")
    source_concepts_csv = os.path.join(output_dir, "source_concepts_index.csv")
    source_jac_csv = os.path.join(output_dir, "source_jacobians_index.csv")
    target_jac_csv = os.path.join(output_dir, "target_jacobians_index.csv")
    meta_json = os.path.join(output_dir, "cache_meta.json")

    df_source.to_csv(source_inventory_csv, index=False)
    df_target.to_csv(target_inventory_csv, index=False)

    if recompute or not os.path.exists(source_concepts_csv):
        print("[1/3] Building source concept targets...")
        _, df_concepts = precompute_concept_targets_from_dataframe(
            df=df_source,
            atlas_mgr=atlas_mgr,
            x_column="x_path",
            label_column="label",
            subject_id_column="subject_id",
            output_dir=src_concepts_dir,
            cfg=ConceptTargetConfig(normal_class_name="CN"),
        )
        df_concepts.to_csv(source_concepts_csv, index=False)
    else:
        print("[1/3] Reusing source concept targets index...")
        df_concepts = pd.read_csv(source_concepts_csv)

    if recompute or not os.path.exists(source_jac_csv):
        print("[2/3] Building source Jacobian priors...")
        ensure_simpleitk_or_raise()
        df_src_jac = precompute_jacobians_from_dataframe(
            df=df_source,
            atlas_mgr=atlas_mgr,
            template_x_path=template_x_path,
            x_column="x_path",
            subject_id_column="subject_id",
            output_dir=src_jac_dir,
            cfg=JacobianConfig(),
        )
        df_src_jac.to_csv(source_jac_csv, index=False)
    else:
        print("[2/3] Reusing source Jacobian priors index...")
        df_src_jac = pd.read_csv(source_jac_csv)

    if recompute or not os.path.exists(target_jac_csv):
        print("[3/3] Building target Jacobian priors...")
        ensure_simpleitk_or_raise()
        df_tgt_jac = precompute_jacobians_from_dataframe(
            df=df_target,
            atlas_mgr=atlas_mgr,
            template_x_path=template_x_path,
            x_column="x_path",
            subject_id_column="subject_id",
            output_dir=tgt_jac_dir,
            cfg=JacobianConfig(),
        )
        df_tgt_jac.to_csv(target_jac_csv, index=False)
    else:
        print("[3/3] Reusing target Jacobian priors index...")
        df_tgt_jac = pd.read_csv(target_jac_csv)

    meta = {
        "base_dir": base_dir,
        "artifacts_dir": output_dir,
        "atlas_path": atlas_path,
        "template_x_path": template_x_path,
        "K": int(atlas_mgr.K),
        "n_source": int(len(df_source)),
        "n_target": int(len(df_target)),
        "source_inventory_csv": source_inventory_csv,
        "target_inventory_csv": target_inventory_csv,
        "source_concepts_csv": source_concepts_csv,
        "source_jac_csv": source_jac_csv,
        "target_jac_csv": target_jac_csv,
    }
    Path(meta_json).write_text(json.dumps(meta, indent=2), encoding="utf-8")

    print("\n[OK] Precomputed artifacts ready.")
    print(f"Artifacts dir: {output_dir}")
    print(f"K = {atlas_mgr.K}")
    print(f"Source subjects = {len(df_source)}")
    print(f"Target subjects = {len(df_target)}")

    return meta


In [ ]:
import numpy as np
import pandas as pd

# def summarize_cv_results(cv_results, stage="full"):
#     """
#     Convierte cv_results_target en un DataFrame resumen con mean ± std
#     para validation y external.

#     Parámetros
#     ----------
#     cv_results : list
#         Salida de run_supervised_cv_for_cohort(...)
#     stage : str
#         "warm" o "full"

#     Retorna
#     -------
#     df_summary : pd.DataFrame
#         DataFrame con mean, std y mean±std por métrica.
#     df_per_fold : pd.DataFrame
#         DataFrame con una fila por fold.
#     """

#     metric_names = [
#         "accuracy",
#         "f1_macro",
#         "recall_macro",
#         "precision_macro",
#         "auc_macro_ovr",
#     ]

#     rows = []

#     for fold_result in cv_results:
#         fold_idx = fold_result["fold_idx"]
#         cohort = fold_result["cohort"]
#         hist = fold_result["history"][stage]

#         # escoger la mejor época según val_f1_macro
#         best_epoch_idx = int(np.argmax([ep["val_f1_macro"] for ep in hist]))
#         best_epoch = hist[best_epoch_idx]

#         row = {
#             "fold_idx": fold_idx,
#             "cohort": cohort,
#             "best_epoch_idx": best_epoch_idx,
#         }

#         # métricas de validation
#         for m in metric_names:
#             row[f"val_{m}"] = best_epoch.get(f"val_{m}", np.nan)

#         # métricas externas
#         for m in metric_names:
#             row[f"external_{m}"] = best_epoch.get(f"external_{m}", np.nan)

#         rows.append(row)

#     df_per_fold = pd.DataFrame(rows)

#     summary_rows = []
#     for split in ["val", "external"]:
#         for m in metric_names:
#             col = f"{split}_{m}"
#             mean_val = df_per_fold[col].mean()
#             std_val = df_per_fold[col].std(ddof=1)

#             summary_rows.append({
#                 "split": split,
#                 "metric": m,
#                 "mean": mean_val,
#                 "std": std_val,
#                 "mean±std": f"{mean_val:.4f} ± {std_val:.4f}",
#             })

#     df_summary = pd.DataFrame(summary_rows)
#     return df_summary, df_per_fold

def summarize_cv_results_with_anatomy(cv_results, stage="full"):
    rows = []

    for fold_result in cv_results:
        hist = fold_result["history"][stage]
        best_epoch_idx = int(np.argmax([ep.get("val_cbm_f1_macro", np.nan) for ep in hist]))
        best_epoch = hist[best_epoch_idx]

        row = {
            "fold_idx": fold_result["fold_idx"],
            "cohort": fold_result["cohort"],
            "best_epoch_idx": best_epoch_idx,

            "val_z_f1_macro": best_epoch.get("val_z_f1_macro", np.nan),
            "val_cbm_f1_macro": best_epoch.get("val_cbm_f1_macro", np.nan),
            "val_z_auc_macro_ovr": best_epoch.get("val_z_auc_macro_ovr", np.nan),
            "val_cbm_auc_macro_ovr": best_epoch.get("val_cbm_auc_macro_ovr", np.nan),
            "val_concept_mse": best_epoch.get("val_concept_mse", np.nan),
            "val_jac_mse": best_epoch.get("val_jac_mse", np.nan),
            "val_concept_corr_mean": best_epoch.get("val_concept_corr_mean", np.nan),
            "val_jac_corr_mean": best_epoch.get("val_jac_corr_mean", np.nan),
            "val_alpha_entropy": best_epoch.get("val_alpha_entropy", np.nan),

            "external_z_f1_macro": best_epoch.get("external_z_f1_macro", np.nan),
            "external_cbm_f1_macro": best_epoch.get("external_cbm_f1_macro", np.nan),
            "external_z_auc_macro_ovr": best_epoch.get("external_z_auc_macro_ovr", np.nan),
            "external_cbm_auc_macro_ovr": best_epoch.get("external_cbm_auc_macro_ovr", np.nan),
            "external_concept_corr_mean": best_epoch.get("external_concept_corr_mean", np.nan),
            "external_jac_corr_mean": best_epoch.get("external_jac_corr_mean", np.nan),
        }

        if "posthoc_val_summary" in fold_result:
            row["posthoc_val_cbm_f1_macro"] = fold_result["posthoc_val_summary"].get("cbm_f1_macro", np.nan)
            row["posthoc_val_concept_corr_mean"] = fold_result["posthoc_val_summary"].get("concept_corr_mean", np.nan)
            row["posthoc_val_jac_corr_mean"] = fold_result["posthoc_val_summary"].get("jac_corr_mean", np.nan)

        rows.append(row)

    df_per_fold = pd.DataFrame(rows)

    summary_rows = []
    for col in df_per_fold.columns:
        if col in ["fold_idx", "cohort", "best_epoch_idx"]:
            continue
        if not np.issubdtype(df_per_fold[col].dtype, np.number):
            continue

        mean_val = df_per_fold[col].mean()
        std_val = df_per_fold[col].std(ddof=1)

        summary_rows.append({
            "metric": col,
            "mean": mean_val,
            "std": std_val,
            "mean±std": f"{mean_val:.4f} ± {std_val:.4f}",
        })

    df_summary = pd.DataFrame(summary_rows)
    return df_summary, df_per_fold

In [ ]:
from __future__ import annotations

"""
Baseline training suite adapted to the user's MRI-only Alzheimer pipeline.

Important
---------
This module is designed to run *after* the user's M&M notebook has already
been executed, because it reuses the same data inventory / precomputed artifact
logic and the same cross-validation protocol.

The linked GitHub repositories span different frameworks and input protocols
(Keras, 2D slice pipelines, longitudinal inputs, atlas-driven pipelines, etc.).
For reproducible benchmarking inside a single MRI-only 3D pipeline, this file
implements *architecture-style baselines* that preserve the core inductive bias
of each public model family while training them with the user's data loaders,
CV splits, and metrics.

Therefore:
  - CNN_design_for_AD, DenseNet-CNN, ViT, AAGN are close PyTorch reimplementations.
  - Joint-Transformer and LongFormer are adapted to 3D MRI tensors and the user's
    cross-validation/data contract.
  - FasterSNN is implemented as a spiking-style 3D surrogate; if spikingjelly is
    available it can be extended further, but the code runs without it.

All models output logits for 3-way CN/MCI/AD classification.
"""

import math
import os
import json
import random
import warnings
from dataclasses import dataclass, asdict
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset, Subset

# -----------------------------------------------------------------------------
# Namespace bridge to the already executed notebook
# -----------------------------------------------------------------------------

def _require(ns: Mapping[str, Any], name: str) -> Any:
    if name not in ns:
        raise KeyError(
            f"Required symbol '{name}' was not found. Execute the user's MRI notebook first, "
            f"then call this module with namespace=globals()."
        )
    return ns[name]


def _maybe(ns: Mapping[str, Any], name: str, default: Any = None) -> Any:
    return ns[name] if name in ns else default


# -----------------------------------------------------------------------------
# Generic utils
# -----------------------------------------------------------------------------

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@dataclass
class BaselineTrainConfig:
    n_epochs: int = 25
    lr: float = 1e-4
    weight_decay: float = 1e-4
    batch_size: int = 2
    num_workers: int = 0
    use_amp: bool = True
    grad_clip_norm: float = 1.0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    log_every: int = 10
    early_stopping_patience: int = 10
    scheduler: str = "cosine"
    label_smoothing: float = 0.0
    seed: int = 42


@dataclass
class BaselineModelConfig:
    input_shape: Tuple[int, int, int] = (128, 128, 128)
    n_classes: int = 3
    base_ch: int = 32
    embed_dim: int = 128
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.1
    patch_size: Tuple[int, int, int] = (16, 16, 16)
    n_slice_tokens: int = 24
    longformer_window: int = 32


# -----------------------------------------------------------------------------
# Metrics
# -----------------------------------------------------------------------------

def _safe_auc_macro_ovr(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    try:
        return float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro"))
    except Exception:
        return float("nan")


def classification_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "auc_macro_ovr": _safe_auc_macro_ovr(y_true, y_prob),
    }


# -----------------------------------------------------------------------------
# Common layers
# -----------------------------------------------------------------------------
class ConvNormAct3D(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, k: int = 3, s: int = 1, p: Optional[int] = None):
        super().__init__()
        if p is None:
            p = k // 2
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class ResidualBlock3D(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        self.conv1 = ConvNormAct3D(in_ch, out_ch, 3, stride)
        self.conv2 = nn.Sequential(
            nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
        )
        self.act = nn.GELU()
        self.skip = None
        if in_ch != out_ch or stride != 1:
            self.skip = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.InstanceNorm3d(out_ch, affine=True),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x if self.skip is None else self.skip(x)
        out = self.conv1(x)
        out = self.conv2(out)
        return self.act(out + identity)


class Small3DBackbone(nn.Module):
    def __init__(self, in_ch: int = 1, base_ch: int = 32, out_ch: int = 256):
        super().__init__()
        self.stem = ConvNormAct3D(in_ch, base_ch, 5, 2, 2)
        self.layer1 = nn.Sequential(
            ResidualBlock3D(base_ch, base_ch),
            ResidualBlock3D(base_ch, base_ch),
        )
        self.layer2 = nn.Sequential(
            ResidualBlock3D(base_ch, base_ch * 2, 2),
            ResidualBlock3D(base_ch * 2, base_ch * 2),
        )
        self.layer3 = nn.Sequential(
            ResidualBlock3D(base_ch * 2, base_ch * 4, 2),
            ResidualBlock3D(base_ch * 4, base_ch * 4),
        )
        self.layer4 = nn.Sequential(
            ResidualBlock3D(base_ch * 4, out_ch, 2),
            ResidualBlock3D(out_ch, out_ch),
        )
        self.out_ch = out_ch

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        feat = self.layer4(x)
        pooled = F.adaptive_avg_pool3d(feat, output_size=1).flatten(1)
        return feat, pooled


class MLP(nn.Module):
    def __init__(self, dim: int, hidden: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, dim: int, heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, hidden=dim * 4, dropout=dropout)

    def forward(self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        h = self.norm1(x)
        h, _ = self.attn(h, h, h, attn_mask=attn_mask, need_weights=False)
        x = x + h
        x = x + self.mlp(self.norm2(x))
        return x


class TransformerEncoder(nn.Module):
    def __init__(self, dim: int, depth: int, heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.blocks = nn.ModuleList([TransformerBlock(dim, heads, dropout) for _ in range(depth)])

    def forward(self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        for blk in self.blocks:
            x = blk(x, attn_mask=attn_mask)
        return x


class LocalTransformerEncoder(nn.Module):
    def __init__(self, dim: int, depth: int, heads: int, window: int, dropout: float = 0.1):
        super().__init__()
        self.encoder = TransformerEncoder(dim, depth, heads, dropout)
        self.window = int(window)

    def _build_mask(self, n: int, device: torch.device) -> torch.Tensor:
        idx = torch.arange(n, device=device)
        dist = (idx[:, None] - idx[None, :]).abs()
        mask = dist > self.window
        # MultiheadAttention expects True => masked when bool mask is used.
        return mask

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mask = self._build_mask(x.shape[1], x.device)
        return self.encoder(x, attn_mask=mask)


# -----------------------------------------------------------------------------
# ROI pooling helpers
# -----------------------------------------------------------------------------

import torch.nn.functional as F

def resize_roi_masks(roi_masks, target_shape):
    target_shape = tuple(int(x) for x in target_shape)

    if roi_masks.ndim == 5:
        if roi_masks.shape[0] == 1 and roi_masks.shape[1] > 1:
            roi_masks = roi_masks[0]      # [K,D,H,W]
        elif roi_masks.shape[1] == 1:
            roi_masks = roi_masks[:, 0]   # [K,D,H,W]
        else:
            raise ValueError(
                f"Formato inesperado en resize_roi_masks: {tuple(roi_masks.shape)}"
            )

    if roi_masks.ndim != 4:
        raise ValueError(
            f"resize_roi_masks espera [K,D,H,W], pero llegó {tuple(roi_masks.shape)}"
        )

    masks = roi_masks.unsqueeze(1).float()  # [K,1,D,H,W]

    if tuple(masks.shape[-3:]) != target_shape:
        masks = F.interpolate(
            masks,
            size=target_shape,
            mode="trilinear",
            align_corners=False,
        )

    masks = masks[:, 0]  # [K,D,H,W]

    # normalización para pooling ponderado
    denom = masks.flatten(1).sum(-1, keepdim=True).clamp_min(1e-6)
    masks = masks / denom.view(-1, 1, 1, 1)

    return masks


def masked_roi_pool(feat, roi_masks):
    if feat.ndim != 5:
        raise ValueError(f"feat debe ser [B,C,D,H,W], pero llegó {tuple(feat.shape)}")

    if roi_masks.ndim == 5:
        if roi_masks.shape[0] == 1 and roi_masks.shape[1] > 1:
            roi_masks = roi_masks[0]
        elif roi_masks.shape[1] == 1:
            roi_masks = roi_masks[:, 0]

    if roi_masks.ndim != 4:
        raise ValueError(f"roi_masks debe ser [K,D,H,W], pero llegó {tuple(roi_masks.shape)}")

    if tuple(roi_masks.shape[-3:]) != tuple(feat.shape[-3:]):
        raise ValueError(
            f"Mismatch espacial: roi_masks={tuple(roi_masks.shape[-3:])} "
            f"vs feat={tuple(feat.shape[-3:])}"
        )

    B, C, D, H, W = feat.shape
    K = roi_masks.shape[0]

    feat_flat = feat.reshape(B, C, -1)
    masks_flat = roi_masks.reshape(K, -1).to(feat.dtype)

    pooled = torch.einsum("bcv,kv->bkc", feat_flat, masks_flat)
    return pooled                                # [B,K,C]


# -----------------------------------------------------------------------------
# Baselines
# -----------------------------------------------------------------------------
class BaseBaseline(nn.Module):
    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        raise NotImplementedError


class CNNDesignForADBaseline(BaseBaseline):
    """Wide 3D CNN inspired by CNN_design_for_AD."""
    def __init__(self, n_classes: int = 3, base_ch: int = 24):
        super().__init__()
        self.backbone = Small3DBackbone(in_ch=1, base_ch=base_ch, out_ch=base_ch * 8)
        self.head = nn.Sequential(
            nn.Linear(base_ch * 8, base_ch * 8),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(base_ch * 8, n_classes),
        )
        self.n_classes = n_classes

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        _, pooled = self.backbone(x)
        logits = self.head(pooled)
        return {"logits": logits, "features": pooled}


class _DenseLayer3D(nn.Module):
    def __init__(self, in_ch: int, growth: int):
        super().__init__()
        inter = growth * 4
        self.net = nn.Sequential(
            nn.InstanceNorm3d(in_ch, affine=True),
            nn.GELU(),
            nn.Conv3d(in_ch, inter, kernel_size=1, bias=False),
            nn.InstanceNorm3d(inter, affine=True),
            nn.GELU(),
            nn.Conv3d(inter, growth, kernel_size=3, padding=1, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.net(x)
        return torch.cat([x, y], dim=1)


class _DenseBlock3D(nn.Module):
    def __init__(self, in_ch: int, n_layers: int, growth: int):
        super().__init__()
        layers = []
        ch = in_ch
        for _ in range(n_layers):
            layers.append(_DenseLayer3D(ch, growth))
            ch += growth
        self.block = nn.Sequential(*layers)
        self.out_ch = ch

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class _Transition3D(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.InstanceNorm3d(in_ch, affine=True),
            nn.GELU(),
            nn.Conv3d(in_ch, out_ch, kernel_size=1, bias=False),
            nn.AvgPool3d(kernel_size=2, stride=2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class DenseNetCNNBaseline(BaseBaseline):
    def __init__(self, n_classes: int = 3, base_ch: int = 24, growth: int = 16):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv3d(1, base_ch, kernel_size=5, stride=2, padding=2, bias=False),
            nn.InstanceNorm3d(base_ch, affine=True),
            nn.GELU(),
        )
        self.db1 = _DenseBlock3D(base_ch, 3, growth)
        self.tr1 = _Transition3D(self.db1.out_ch, base_ch * 2)
        self.db2 = _DenseBlock3D(base_ch * 2, 4, growth)
        self.tr2 = _Transition3D(self.db2.out_ch, base_ch * 4)
        self.db3 = _DenseBlock3D(base_ch * 4, 4, growth)
        self.out_ch = self.db3.out_ch
        self.cls = nn.Linear(self.out_ch, n_classes)
        self.n_classes = n_classes

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        x = self.stem(x)
        x = self.db1(x)
        x = self.tr1(x)
        x = self.db2(x)
        x = self.tr2(x)
        x = self.db3(x)
        pooled = F.adaptive_avg_pool3d(x, 1).flatten(1)
        logits = self.cls(pooled)
        return {"logits": logits, "features": pooled}


class PatchEmbed3D(nn.Module):
    def __init__(self, in_ch: int, dim: int, patch_size: Tuple[int, int, int]):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv3d(in_ch, dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.proj(x)  # [B,dim,h,w,d]
        x = x.flatten(2).transpose(1, 2)  # [B,N,dim]
        return x


class ViTBaseline(BaseBaseline):
    def __init__(self, input_shape=(128,128,128), n_classes=3, embed_dim=128, heads=4, depth=4, patch_size=(16,16,16), dropout=0.1):
        super().__init__()
        self.patch = PatchEmbed3D(1, embed_dim, patch_size)
        n_tokens = (input_shape[0] // patch_size[0]) * (input_shape[1] // patch_size[1]) * (input_shape[2] // patch_size[2])
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos = nn.Parameter(torch.zeros(1, n_tokens + 1, embed_dim))
        self.enc = TransformerEncoder(embed_dim, depth, heads, dropout)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, n_classes)
        self.n_classes = n_classes

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        tok = self.patch(x)
        b = tok.size(0)
        cls = self.cls_token.expand(b, -1, -1)
        tok = torch.cat([cls, tok], dim=1)
        tok = tok + self.pos[:, :tok.size(1)]
        tok = self.enc(tok)
        feat = self.norm(tok[:, 0])
        logits = self.head(feat)
        return {"logits": logits, "features": feat}


class LongFormerBaseline(BaseBaseline):
    """3D patch transformer with local attention mask (LongFormer-style)."""
    def __init__(self, input_shape=(128,128,128), n_classes=3, embed_dim=128, heads=4, depth=4, patch_size=(16,16,16), window=32, dropout=0.1):
        super().__init__()
        self.patch = PatchEmbed3D(1, embed_dim, patch_size)
        n_tokens = (input_shape[0] // patch_size[0]) * (input_shape[1] // patch_size[1]) * (input_shape[2] // patch_size[2])
        self.pos = nn.Parameter(torch.zeros(1, n_tokens, embed_dim))
        self.enc = LocalTransformerEncoder(embed_dim, depth, heads, window, dropout)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, n_classes)
        self.n_classes = n_classes

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        tok = self.patch(x)
        tok = tok + self.pos[:, :tok.size(1)]
        tok = self.enc(tok)
        feat = self.norm(tok.mean(dim=1))
        logits = self.head(feat)
        return {"logits": logits, "features": feat}


class _SliceEncoder2D(nn.Module):
    def __init__(self, out_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 5, 2, 2, bias=False),
            nn.InstanceNorm2d(32, affine=True),
            nn.GELU(),
            nn.Conv2d(32, 64, 3, 2, 1, bias=False),
            nn.InstanceNorm2d(64, affine=True),
            nn.GELU(),
            nn.Conv2d(64, out_dim, 3, 2, 1, bias=False),
            nn.InstanceNorm2d(out_dim, affine=True),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).flatten(1)


class JointTransformerBaseline(BaseBaseline):
    """Adapted ViT-TST style baseline for 3D tensors using 3 plane slice sequences."""
    def __init__(self, n_classes=3, embed_dim=128, heads=4, depth=2, n_slices=24, dropout=0.1):
        super().__init__()
        self.slice_encoder = _SliceEncoder2D(embed_dim)
        self.n_slices = int(n_slices)
        self.axis_pos = nn.Parameter(torch.zeros(1, n_slices, embed_dim))
        self.seq_encoder = TransformerEncoder(embed_dim, depth, heads, dropout)
        self.plane_head = nn.Linear(embed_dim * 3, n_classes)
        self.n_classes = n_classes

    def _sample_indices(self, n: int, device: torch.device) -> torch.Tensor:
        if n <= self.n_slices:
            idx = torch.linspace(0, n - 1, steps=n, device=device).long()
            if n < self.n_slices:
                pad = idx[-1:].repeat(self.n_slices - n)
                idx = torch.cat([idx, pad], dim=0)
            return idx
        return torch.linspace(0, n - 1, steps=self.n_slices, device=device).round().long()

    def _encode_plane(self, x: torch.Tensor, axis: int) -> torch.Tensor:
        # x [B,1,H,W,D]
        B = x.size(0)
        spatial = x.shape[2:]
        idx = self._sample_indices(spatial[axis], x.device)
        seq = []
        for s in idx.tolist():
            if axis == 0:
                sl = x[:, :, s, :, :]
            elif axis == 1:
                sl = x[:, :, :, s, :]
            else:
                sl = x[:, :, :, :, s]
            seq.append(self.slice_encoder(sl))
        tok = torch.stack(seq, dim=1) + self.axis_pos[:, :len(seq)]
        tok = self.seq_encoder(tok)
        return tok.mean(dim=1)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        f_ax = self._encode_plane(x, axis=2)  # sagittal-ish depending on ordering
        f_cor = self._encode_plane(x, axis=1)
        f_sag = self._encode_plane(x, axis=0)
        feat = torch.cat([f_ax, f_cor, f_sag], dim=-1)
        logits = self.plane_head(feat)
        return {"logits": logits, "features": feat}


class SurrogateSpikeFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return (x > 0).float()

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        grad = 1.0 / (1.0 + x.abs()).pow(2)
        return grad_output * grad


class SpikeAct(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return SurrogateSpikeFn.apply(x)


class FasterSNNBaseline(BaseBaseline):
    """Spiking-style lightweight 3D surrogate inspired by FasterSNN."""
    def __init__(self, n_classes: int = 3, base_ch: int = 24):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv3d(1, base_ch, 3, 2, 1, bias=False),
            nn.InstanceNorm3d(base_ch, affine=True),
            SpikeAct(),
            nn.Conv3d(base_ch, base_ch * 2, 3, 2, 1, bias=False),
            nn.InstanceNorm3d(base_ch * 2, affine=True),
            SpikeAct(),
            nn.Conv3d(base_ch * 2, base_ch * 4, 3, 2, 1, bias=False),
            nn.InstanceNorm3d(base_ch * 4, affine=True),
            SpikeAct(),
            nn.Conv3d(base_ch * 4, base_ch * 8, 3, 2, 1, bias=False),
            nn.InstanceNorm3d(base_ch * 8, affine=True),
            SpikeAct(),
        )
        self.cls = nn.Linear(base_ch * 8, n_classes)
        self.n_classes = n_classes

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        feat = self.features(x)
        pooled = F.adaptive_avg_pool3d(feat, 1).flatten(1)
        logits = self.cls(pooled)
        return {"logits": logits, "features": pooled}


class ROIAwareGatingBaseline(BaseBaseline):
    """AAGN-style ROI-aware gating baseline."""
    def __init__(self, roi_masks: torch.Tensor, n_classes: int = 3, base_ch: int = 32, embed_dim: int = 128):
        super().__init__()
        self.backbone = Small3DBackbone(in_ch=1, base_ch=base_ch, out_ch=embed_dim)
        self.roi_masks = roi_masks.float()  # [K,H,W,D]
        self.K = roi_masks.shape[0]
        self.gate = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, 1),
        )
        self.cls = nn.Linear(embed_dim, n_classes)
        self.n_classes = n_classes

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        feat, _ = self.backbone(x)
        roi_masks = resize_roi_masks(self.roi_masks.to(feat.device), feat.shape[-3:])
        roi_feat = masked_roi_pool(feat, roi_masks)     # [B,K,C]
        alpha = torch.softmax(self.gate(roi_feat).squeeze(-1), dim=-1)  # [B,K]
        pooled = torch.einsum('bk,bkc->bc', alpha, roi_feat)
        logits = self.cls(pooled)
        return {"logits": logits, "features": pooled, "alpha": alpha}


def build_baseline_model(name: str, cfg: BaselineModelConfig, roi_masks: Optional[torch.Tensor] = None) -> nn.Module:
    key = name.strip().lower()
    if key in {"cnn_design_for_ad", "cnndesignforad", "cnn_design_for_adbaseline"}:
        return CNNDesignForADBaseline(n_classes=cfg.n_classes, base_ch=cfg.base_ch)
    if key in {"densenet-cnn", "densenet_cnn", "densenetcnn"}:
        return DenseNetCNNBaseline(n_classes=cfg.n_classes, base_ch=max(16, cfg.base_ch // 2))
    if key in {"vit", "visiontransformer"}:
        return ViTBaseline(
            input_shape=cfg.input_shape,
            n_classes=cfg.n_classes,
            embed_dim=cfg.embed_dim,
            heads=cfg.n_heads,
            depth=max(3, cfg.n_layers + 1),
            patch_size=cfg.patch_size,
            dropout=cfg.dropout,
        )
    if key in {"joint-transformer", "joint_transformer", "jointtransformer"}:
        return JointTransformerBaseline(
            n_classes=cfg.n_classes,
            embed_dim=cfg.embed_dim,
            heads=cfg.n_heads,
            depth=cfg.n_layers,
            n_slices=cfg.n_slice_tokens,
            dropout=cfg.dropout,
        )
    if key in {"longformer", "long_former"}:
        return LongFormerBaseline(
            input_shape=cfg.input_shape,
            n_classes=cfg.n_classes,
            embed_dim=cfg.embed_dim,
            heads=cfg.n_heads,
            depth=max(3, cfg.n_layers + 1),
            patch_size=cfg.patch_size,
            window=cfg.longformer_window,
            dropout=cfg.dropout,
        )
    if key in {"aagn", "roiawaregating", "aagnstyle"}:
        if roi_masks is None:
            raise ValueError("AAGN baseline requires roi_masks from the user's atlas manager.")
        return ROIAwareGatingBaseline(
            roi_masks=roi_masks,
            n_classes=cfg.n_classes,
            base_ch=cfg.base_ch,
            embed_dim=cfg.embed_dim,
        )
    if key in {"fastersnn", "faster_snn"}:
        return FasterSNNBaseline(n_classes=cfg.n_classes, base_ch=max(16, cfg.base_ch // 2))
    raise ValueError(f"Unknown baseline model name: {name}")


# -----------------------------------------------------------------------------
# Training harness
# -----------------------------------------------------------------------------
class ClassificationOnlyLoss(nn.Module):
    def __init__(self, label_smoothing: float = 0.0):
        super().__init__()
        self.label_smoothing = float(label_smoothing)

    def forward(self, logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        return F.cross_entropy(logits, y, label_smoothing=self.label_smoothing)


class ClassificationOnlyTrainer:
    def __init__(self, model: nn.Module, cfg: BaselineTrainConfig):
        self.model = model
        self.cfg = cfg
        self.device = torch.device(cfg.device)
        self.model.to(self.device)
        self.loss_fn = ClassificationOnlyLoss(cfg.label_smoothing)
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        if cfg.scheduler == "cosine":
            self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=max(1, cfg.n_epochs))
        else:
            self.scheduler = None
        self.scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp and self.device.type == "cuda")

    def _move_batch(self, batch: Dict[str, Any]) -> Dict[str, Any]:
        out = {}
        for k, v in batch.items():
            out[k] = v.to(self.device, non_blocking=True) if torch.is_tensor(v) else v
        return out

    def _forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        return self.model(x)

    @torch.no_grad()
    def evaluate(self, loader: DataLoader, prefix: str = "val") -> Dict[str, float]:
        self.model.eval()
        y_true, y_pred, y_prob = [], [], []
        losses = []
        for batch in loader:
            batch = self._move_batch(batch)
            out = self._forward(batch["x"])
            logits = out["logits"]
            loss = self.loss_fn(logits, batch["y"])
            prob = torch.softmax(logits, dim=-1)
            pred = prob.argmax(dim=-1)
            losses.append(float(loss.item()))
            y_true.append(batch["y"].detach().cpu())
            y_pred.append(pred.detach().cpu())
            y_prob.append(prob.detach().cpu())
        y_true = torch.cat(y_true).numpy()
        y_pred = torch.cat(y_pred).numpy()
        y_prob = torch.cat(y_prob).numpy()
        metrics = classification_metrics(y_true, y_pred, y_prob)
        metrics = {f"{prefix}_{k}": v for k, v in metrics.items()}
        metrics[f"{prefix}_loss"] = float(np.mean(losses)) if losses else float("nan")
        return metrics

    def train_epoch(self, loader: DataLoader) -> Dict[str, float]:
        self.model.train()
        losses = []
        for step, batch in enumerate(loader, start=1):
            batch = self._move_batch(batch)
            self.optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=self.device.type, enabled=self.cfg.use_amp and self.device.type == "cuda"):
                out = self._forward(batch["x"])
                loss = self.loss_fn(out["logits"], batch["y"])
            if self.scaler.is_enabled():
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                self.optimizer.step()
            losses.append(float(loss.item()))
        if self.scheduler is not None:
            self.scheduler.step()
        return {"train_loss": float(np.mean(losses)) if losses else float("nan")}

    def fit(self, train_loader: DataLoader, train_eval_loader: DataLoader, val_loader: DataLoader, external_loader: Optional[DataLoader] = None) -> Dict[str, Any]:
        best_state = None
        best_score = -float("inf")
        patience = 0
        history = []
        for epoch in range(1, self.cfg.n_epochs + 1):
            tr = self.train_epoch(train_loader)
            train_metrics = self.evaluate(train_eval_loader, prefix="train") if train_eval_loader is not None else {}
            val_metrics = self.evaluate(val_loader, prefix="val") if val_loader is not None else {}
            ext_metrics = self.evaluate(external_loader, prefix="external") if external_loader is not None else {}
            row = {"epoch": epoch, **tr, **train_metrics, **val_metrics, **ext_metrics}
            history.append(row)
            score = row.get("val_f1_macro", float("nan"))
            print(
                f"[Epoch {epoch:03d}] "
                f"train_loss={row.get('train_loss', float('nan')):.4f} | "
                f"VAL acc={row.get('val_accuracy', float('nan')):.4f} "
                f"f1={row.get('val_f1_macro', float('nan')):.4f} "
                f"rec={row.get('val_recall_macro', float('nan')):.4f} "
                f"prec={row.get('val_precision_macro', float('nan')):.4f} "
                f"auc={row.get('val_auc_macro_ovr', float('nan')):.4f}"
            )
            if np.isfinite(score) and score > best_score:
                best_score = score
                best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= self.cfg.early_stopping_patience:
                    print(f"Early stopping at epoch {epoch}.")
                    break
        if best_state is not None:
            self.model.load_state_dict(best_state)
        final_val = self.evaluate(val_loader, prefix="val") if val_loader is not None else {}
        final_ext = self.evaluate(external_loader, prefix="external") if external_loader is not None else {}
        return {
            "history": history,
            "best_score": best_score,
            "final_val": final_val,
            "final_external": final_ext,
            "state_dict": self.model.state_dict(),
        }


# -----------------------------------------------------------------------------
# CV runner reusing the user's data + artifact pipeline
# -----------------------------------------------------------------------------

def _make_loader_from_subset(dataset: Dataset, indices: Sequence[int], batch_size: int, shuffle: bool, num_workers: int) -> DataLoader:
    subset = Subset(dataset, indices)
    return DataLoader(
        subset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=shuffle,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )


def _resolve_roi_masks(
    namespace,
    atlas_path,
    target_shape,
):
    AtlasROIManager = _require(namespace, "AtlasROIManager")
    atlas_mgr = AtlasROIManager(atlas_path)

    if hasattr(atlas_mgr, "get_binary_masks"):
        masks = atlas_mgr.get_binary_masks(target_shape=target_shape)
    else:
        masks = atlas_mgr.get_masks(target_shape=target_shape, normalize=False)

    # Normalizar formatos posibles:
    # [K, D, H, W]
    # [K, 1, D, H, W]
    # [1, K, D, H, W]
    if masks.ndim == 5:
        if masks.shape[0] == 1 and masks.shape[1] > 1:
            masks = masks[0]          # [K, D, H, W]
        elif masks.shape[1] == 1:
            masks = masks[:, 0]       # [K, D, H, W]
        else:
            raise ValueError(
                f"Formato ROI no soportado. Se esperaba [K,1,D,H,W] o [1,K,D,H,W], "
                f"pero llegó {tuple(masks.shape)}"
            )

    if masks.ndim != 4:
        raise ValueError(
            f"roi_masks debe quedar como [K,D,H,W], pero llegó {tuple(masks.shape)}"
        )

    if tuple(masks.shape[-3:]) != tuple(target_shape):
        raise ValueError(
            f"Las máscaras quedaron con shape espacial {tuple(masks.shape[-3:])}, "
            f"pero target_shape={tuple(target_shape)}"
        )

    return masks.float().cpu()


def _infer_input_shape_from_dataset(dataset: Dataset) -> Tuple[int, int, int]:
    sample = dataset[0]["x"]
    if sample.ndim == 4:
        return tuple(int(v) for v in sample.shape[-3:])
    if sample.ndim == 3:
        return tuple(int(v) for v in sample.shape)
    raise ValueError(f"Unexpected MRI tensor shape: {tuple(sample.shape)}")


def train_baseline_cv_fold(
    namespace: Mapping[str, Any],
    baseline_name: str,
    base_dir: str,
    project_root: str,
    module_dir: str,
    atlas_path: str,
    precomputed_artifacts_dir: str,
    cohort: str = "target",
    n_splits: int = 5,
    fold_idx: int = 0,
    train_cfg: Optional[BaselineTrainConfig] = None,
    model_cfg: Optional[BaselineModelConfig] = None,
    save_dir: Optional[str] = None,
) -> Dict[str, Any]:
    set_seed((train_cfg.seed if train_cfg is not None else 42) + fold_idx)
    if train_cfg is None:
        train_cfg = BaselineTrainConfig()
    LoadArtifacts = _require(namespace, "load_precomputed_artifacts")
    BuildDataset = _require(namespace, "build_supervised_dataset_for_cohort")
    FindAtlas = _require(namespace, "find_existing_atlas_path")

    atlas_path = FindAtlas(atlas_path)
    cache = LoadArtifacts(
        base_dir=base_dir,
        module_dir=module_dir,
        atlas_path=atlas_path,
        precomputed_artifacts_dir=precomputed_artifacts_dir,
    )
    dataset, labels, _ = BuildDataset(cache, cohort=cohort)
    input_shape = _infer_input_shape_from_dataset(dataset)
    if model_cfg is None:
        model_cfg = BaselineModelConfig(input_shape=input_shape)
    else:
        model_cfg.input_shape = input_shape

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    splits = list(skf.split(np.arange(len(labels)), labels))
    train_idx, val_idx = splits[fold_idx]

    train_loader = _make_loader_from_subset(dataset, train_idx, train_cfg.batch_size, True, train_cfg.num_workers)
    train_eval_loader = _make_loader_from_subset(dataset, train_idx, train_cfg.batch_size, False, train_cfg.num_workers)
    val_loader = _make_loader_from_subset(dataset, val_idx, train_cfg.batch_size, False, train_cfg.num_workers)

    external_loader = None
    SupervisedMRIDatasetWired = _require(namespace, "SupervisedMRIDatasetWired")
    LABEL_MAP = _require(namespace, "LABEL_MAP")
    if cohort == "target":
        ext_dataset = SupervisedMRIDatasetWired(
            df_inventory=cache["df_source"],
            df_concepts=cache["df_concepts"],
            df_jac=cache["df_src_jac"],
            K=int(cache["K"]),
        )
        external_loader = DataLoader(ext_dataset, batch_size=train_cfg.batch_size, shuffle=False, num_workers=train_cfg.num_workers)
    elif cohort == "source" and cache.get("df_tgt_concepts", None) is not None:
        ext_dataset = SupervisedMRIDatasetWired(
            df_inventory=cache["df_target"],
            df_concepts=cache["df_tgt_concepts"],
            df_jac=cache["df_tgt_jac"],
            K=int(cache["K"]),
        )
        external_loader = DataLoader(ext_dataset, batch_size=train_cfg.batch_size, shuffle=False, num_workers=train_cfg.num_workers)

    roi_masks = (
        _resolve_roi_masks(namespace, atlas_path, input_shape)
        if baseline_name.lower() == "aagn"
        else None
    )
    model = build_baseline_model(baseline_name, model_cfg, roi_masks=roi_masks)
    trainer = ClassificationOnlyTrainer(model, train_cfg)
    result = trainer.fit(train_loader, train_eval_loader, val_loader, external_loader=external_loader)

    payload = {
        "baseline_name": baseline_name,
        "cohort": cohort,
        "fold_idx": fold_idx,
        "train_cfg": asdict(train_cfg),
        "model_cfg": asdict(model_cfg),
        "best_score": result["best_score"],
        "final_val": result["final_val"],
        "final_external": result["final_external"],
        "history": result["history"],
    }

    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        stem = f"{baseline_name}_cohort-{cohort}_fold-{fold_idx}"
        torch.save(result["state_dict"], os.path.join(save_dir, stem + "_weights.pt"))
        with open(os.path.join(save_dir, stem + "_metrics.json"), "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)
    return payload


REQUESTED_BASELINES = [
    "AAGN",
    "Joint-Transformer",
    "FasterSNN",
    "CNN_design_for_AD",
    "DenseNet-CNN",
    # "LongFormer",
    
    # "ViT",
]


def run_baseline_cv_for_cohort(
    namespace: Mapping[str, Any],
    baseline_name: str,
    base_dir: str,
    project_root: str,
    module_dir: str,
    atlas_path: str,
    precomputed_artifacts_dir: str,
    cohort: str = "target",
    n_splits: int = 5,
    train_cfg: Optional[BaselineTrainConfig] = None,
    model_cfg: Optional[BaselineModelConfig] = None,
    save_dir: Optional[str] = None,
) -> List[Dict[str, Any]]:
    results = []
    for fold_idx in range(n_splits):
        print("\n" + "=" * 88)
        print(f"Running {baseline_name} | cohort={cohort} | fold {fold_idx + 1}/{n_splits}")
        print("=" * 88)
        out = train_baseline_cv_fold(
            namespace=namespace,
            baseline_name=baseline_name,
            base_dir=base_dir,
            project_root=project_root,
            module_dir=module_dir,
            atlas_path=atlas_path,
            precomputed_artifacts_dir=precomputed_artifacts_dir,
            cohort=cohort,
            n_splits=n_splits,
            fold_idx=fold_idx,
            train_cfg=train_cfg,
            model_cfg=model_cfg,
            save_dir=save_dir,
        )
        results.append(out)
    return results


def run_all_requested_baselines(
    namespace: Mapping[str, Any],
    base_dir: str,
    project_root: str,
    module_dir: str,
    atlas_path: str,
    precomputed_artifacts_dir: str,
    cohort: str = "target",
    n_splits: int = 5,
    train_cfg: Optional[BaselineTrainConfig] = None,
    model_cfg: Optional[BaselineModelConfig] = None,
    save_dir: Optional[str] = None,
    baseline_names: Optional[Sequence[str]] = None,
) -> Dict[str, List[Dict[str, Any]]]:
    if baseline_names is None:
        baseline_names = REQUESTED_BASELINES
    all_results: Dict[str, List[Dict[str, Any]]] = {}
    for name in baseline_names:
        all_results[name] = run_baseline_cv_for_cohort(
            namespace=namespace,
            baseline_name=name,
            base_dir=base_dir,
            project_root=project_root,
            module_dir=module_dir,
            atlas_path=atlas_path,
            precomputed_artifacts_dir=precomputed_artifacts_dir,
            cohort=cohort,
            n_splits=n_splits,
            train_cfg=train_cfg,
            model_cfg=model_cfg,
            save_dir=save_dir,
        )
        if save_dir is not None:
            with open(os.path.join(save_dir, f"{name}_cohort-{cohort}_summary.json"), "w", encoding="utf-8") as f:
                json.dump(all_results[name], f, indent=2)
    return all_results




# =============================================================================
# PATCH: external validation + BiFPN3DViT and DA-ViT-style 3D baselines
# Paste this section after the baseline class definitions, or use the patched
# notebook generated from this conversation.
# =============================================================================

import os
import json
import math
from dataclasses import asdict
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset, Subset


# -----------------------------------------------------------------------------
# 1) Classification-only dataset: external validation must not depend on
#    c_target or g_bar, because architectural baselines are pure classifiers.
# -----------------------------------------------------------------------------
def _cohort_inventory_from_cache(cache: Mapping[str, Any], cohort: str) -> pd.DataFrame:
    cohort = str(cohort).lower()
    if cohort == "source":
        return cache["df_source"].copy()
    if cohort == "target":
        return cache["df_target"].copy()
    raise ValueError("cohort must be 'source' or 'target'.")


def _label_column(df: pd.DataFrame) -> str:
    if "label" in df.columns:
        return "label"
    if "Label" in df.columns:
        return "Label"
    raise KeyError("Inventory DataFrame must contain either 'label' or 'Label'.")


class ClassificationOnlyMRIDataset(Dataset):
    """Minimal MRI dataset for baselines and external validation.

    Expected inventory columns:
        x_path : path to .pt MRI tensor
        label or Label : CN/MCI/AD label string
        subject_id     : optional metadata
    """
    def __init__(self, df_inventory: pd.DataFrame, label_map: Optional[Dict[str, int]] = None):
        super().__init__()
        if label_map is None:
            label_map = globals().get("LABEL_MAP", {"CN": 0, "MCI": 1, "AD": 2})
        self.label_map = dict(label_map)
        self.data = df_inventory.reset_index(drop=True).copy()
        if "x_path" not in self.data.columns:
            raise KeyError("Inventory DataFrame must contain 'x_path'.")
        lab_col = _label_column(self.data)
        self.data["_label_name"] = self.data[lab_col].astype(str)
        self.labels_np = self.data["_label_name"].map(self.label_map).astype(int).to_numpy()

    def __len__(self) -> int:
        return len(self.data)

    def _load_x(self, path: str) -> torch.Tensor:
        load_tensor_like_fn = globals().get("load_tensor_like", None)
        if load_tensor_like_fn is not None:
            x = load_tensor_like_fn(path, expected_ndim=None, prefer_keys=["x", "image", "mri", "tensor", "volume"])
        else:
            obj = torch.load(path, map_location="cpu", weights_only=False)
            if torch.is_tensor(obj):
                x = obj
            elif isinstance(obj, dict):
                for key in ["x", "image", "mri", "tensor", "volume"]:
                    if key in obj:
                        x = obj[key]
                        break
                else:
                    raise KeyError(f"No valid tensor key found in {path}. Available keys: {list(obj.keys())}")
            else:
                x = torch.as_tensor(obj)
            x = x.detach().to(torch.float32).contiguous()

        if x.ndim == 3:
            x = x.unsqueeze(0)  # [1,D,H,W]
        if x.ndim != 4:
            raise ValueError(f"Expected MRI tensor [C,D,H,W] or [D,H,W], got {tuple(x.shape)} from {path}")
        if x.shape[0] != 1:
            # The current project uses single-channel MRI. This keeps the first channel
            # rather than crashing on tensors with accidental repeated channels.
            x = x[:1]
        return x.to(torch.float32).contiguous()

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.data.iloc[idx]
        label_name = str(row["_label_name"])
        y = torch.tensor(self.label_map[label_name], dtype=torch.long)
        subject_id = str(row["subject_id"]) if "subject_id" in self.data.columns else str(idx)
        return {
            "x": self._load_x(str(row["x_path"])),
            "y": y,
            "subject_id": subject_id,
            "label_name": label_name,
        }


def _make_classification_dataset(cache: Mapping[str, Any], cohort: str) -> ClassificationOnlyMRIDataset:
    return ClassificationOnlyMRIDataset(
        _cohort_inventory_from_cache(cache, cohort),
        label_map=globals().get("LABEL_MAP", {"CN": 0, "MCI": 1, "AD": 2}),
    )


def _make_loader(dataset: Dataset, batch_size: int, shuffle: bool, num_workers: int, drop_last: Optional[bool] = None) -> DataLoader:
    if drop_last is None:
        drop_last = bool(shuffle)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )


def _make_loader_from_subset_v2(dataset: Dataset, indices: Sequence[int], batch_size: int, shuffle: bool, num_workers: int) -> DataLoader:
    return _make_loader(Subset(dataset, list(indices)), batch_size, shuffle, num_workers, drop_last=shuffle)


def _resolve_external_cohort(train_cohort: str, external_cohort: Optional[str]) -> Optional[str]:
    if external_cohort is None:
        return None
    external_cohort = str(external_cohort).lower()
    if external_cohort in {"none", "false", "no"}:
        return None
    if external_cohort == "other":
        return "target" if str(train_cohort).lower() == "source" else "source"
    if external_cohort not in {"source", "target"}:
        raise ValueError("external_cohort must be one of: 'other', 'source', 'target', None.")
    return external_cohort


# Visible trainer override: reports external validation during training and stores
# the final external metrics after loading the best internal-validation checkpoint.
_ClassificationOnlyTrainerBase = ClassificationOnlyTrainer


class ClassificationOnlyTrainer(_ClassificationOnlyTrainerBase):
    def fit(
        self,
        train_loader: DataLoader,
        train_eval_loader: DataLoader,
        val_loader: DataLoader,
        external_loader: Optional[DataLoader] = None,
    ) -> Dict[str, Any]:
        best_state = None
        best_score = -float("inf")
        best_epoch = -1
        patience = 0
        history = []

        for epoch in range(1, self.cfg.n_epochs + 1):
            tr = self.train_epoch(train_loader)
            train_metrics = self.evaluate(train_eval_loader, prefix="train") if train_eval_loader is not None else {}
            val_metrics = self.evaluate(val_loader, prefix="val") if val_loader is not None else {}
            ext_metrics = self.evaluate(external_loader, prefix="external") if external_loader is not None else {}
            row = {"epoch": epoch, **tr, **train_metrics, **val_metrics, **ext_metrics}
            history.append(row)

            score = row.get("val_f1_macro", float("nan"))
            msg = (
                f"[Epoch {epoch:03d}] "
                f"train_loss={row.get('train_loss', float('nan')):.4f} | "
                f"VAL acc={row.get('val_accuracy', float('nan')):.4f} "
                f"f1={row.get('val_f1_macro', float('nan')):.4f} "
                f"rec={row.get('val_recall_macro', float('nan')):.4f} "
                f"prec={row.get('val_precision_macro', float('nan')):.4f} "
                f"auc={row.get('val_auc_macro_ovr', float('nan')):.4f}"
            )
            if external_loader is not None:
                msg += (
                    f" | EXT acc={row.get('external_accuracy', float('nan')):.4f} "
                    f"f1={row.get('external_f1_macro', float('nan')):.4f} "
                    f"rec={row.get('external_recall_macro', float('nan')):.4f} "
                    f"prec={row.get('external_precision_macro', float('nan')):.4f} "
                    f"auc={row.get('external_auc_macro_ovr', float('nan')):.4f}"
                )
            print(msg)

            if np.isfinite(score) and score > best_score:
                best_score = score
                best_epoch = epoch
                best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= self.cfg.early_stopping_patience:
                    print(f"Early stopping at epoch {epoch}. Best epoch={best_epoch}.")
                    break

        if best_state is not None:
            self.model.load_state_dict(best_state)

        final_val = self.evaluate(val_loader, prefix="val") if val_loader is not None else {}
        final_ext = self.evaluate(external_loader, prefix="external") if external_loader is not None else {}
        return {
            "history": history,
            "best_score": best_score,
            "best_epoch": best_epoch,
            "final_val": final_val,
            "final_external": final_ext,
            "state_dict": self.model.state_dict(),
        }


# -----------------------------------------------------------------------------
# 2) New architecture-style baselines requested by the user.
#    They are adapted to the existing 3D MRI tensor contract [B,1,D,H,W].
# -----------------------------------------------------------------------------
def _conv_out_stride2(n: int) -> int:
    # Conv3d with k=5/3, stride=2 and same-like padding used in this notebook.
    return int((int(n) + 1) // 2)


def _downsampled_shape(input_shape: Tuple[int, int, int], n_down: int = 4) -> Tuple[int, int, int]:
    d, h, w = [int(v) for v in input_shape]
    for _ in range(n_down):
        d, h, w = _conv_out_stride2(d), _conv_out_stride2(h), _conv_out_stride2(w)
    return max(d, 1), max(h, 1), max(w, 1)


class BiFPNLayer3D(nn.Module):
    """Lightweight 3D BiFPN fusion block for multi-scale volumetric features."""
    def __init__(self, in_channels: Sequence[int], out_ch: int):
        super().__init__()
        self.proj = nn.ModuleList([nn.Conv3d(c, out_ch, kernel_size=1, bias=False) for c in in_channels])
        self.smooth = nn.ModuleList([
            nn.Sequential(
                nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.InstanceNorm3d(out_ch, affine=True),
                nn.GELU(),
            )
            for _ in in_channels
        ])
        self.eps = 1e-4
        self.w_td = nn.Parameter(torch.ones(len(in_channels) - 1, 2))
        self.w_bu = nn.Parameter(torch.ones(len(in_channels) - 1, 2))

    def _fuse2(self, a: torch.Tensor, b: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
        w = F.relu(w)
        w = w / (w.sum() + self.eps)
        return w[0] * a + w[1] * b

    def forward(self, feats: Sequence[torch.Tensor]) -> torch.Tensor:
        p = [proj(f) for proj, f in zip(self.proj, feats)]

        # top-down path
        td = list(p)
        for i in range(len(td) - 2, -1, -1):
            up = F.interpolate(td[i + 1], size=td[i].shape[-3:], mode="trilinear", align_corners=False)
            td[i] = self._fuse2(td[i], up, self.w_td[i])

        # bottom-up path
        out = list(td)
        for i in range(1, len(out)):
            down = F.adaptive_avg_pool3d(out[i - 1], output_size=out[i].shape[-3:])
            out[i] = self._fuse2(out[i], down, self.w_bu[i - 1])

        out = [smooth(x) for smooth, x in zip(self.smooth, out)]
        return out[-1]


class BiFPN3DViTBaseline(BaseBaseline):
    """BiFPN3DViT-style baseline: 3D CNN pyramid + BiFPN + ViT encoder."""
    def __init__(
        self,
        input_shape: Tuple[int, int, int] = (128, 128, 128),
        n_classes: int = 3,
        base_ch: int = 24,
        embed_dim: int = 128,
        heads: int = 4,
        depth: int = 2,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.stem = ConvNormAct3D(1, base_ch, k=5, s=2, p=2)
        self.s1 = nn.Sequential(ResidualBlock3D(base_ch, base_ch), ResidualBlock3D(base_ch, base_ch))
        self.s2 = nn.Sequential(ResidualBlock3D(base_ch, base_ch * 2, stride=2), ResidualBlock3D(base_ch * 2, base_ch * 2))
        self.s3 = nn.Sequential(ResidualBlock3D(base_ch * 2, base_ch * 4, stride=2), ResidualBlock3D(base_ch * 4, base_ch * 4))
        self.s4 = nn.Sequential(ResidualBlock3D(base_ch * 4, base_ch * 8, stride=2), ResidualBlock3D(base_ch * 8, base_ch * 8))
        self.bifpn = BiFPNLayer3D([base_ch, base_ch * 2, base_ch * 4, base_ch * 8], embed_dim)

        token_shape = _downsampled_shape(input_shape, n_down=4)
        n_tokens = int(np.prod(token_shape))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_tokens + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)
        self.encoder = TransformerEncoder(embed_dim, depth=max(1, depth), heads=heads, dropout=dropout)
        self.norm = nn.LayerNorm(embed_dim)
        self.attn_pool = nn.Linear(embed_dim, 1)
        self.head = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, n_classes),
        )
        self.n_classes = n_classes
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        x = self.stem(x)
        f1 = self.s1(x)
        f2 = self.s2(f1)
        f3 = self.s3(f2)
        f4 = self.s4(f3)
        fused = self.bifpn([f1, f2, f3, f4])
        tok = fused.flatten(2).transpose(1, 2)
        b = tok.size(0)
        cls = self.cls_token.expand(b, -1, -1)
        tok = torch.cat([cls, tok], dim=1)
        tok = self.pos_drop(tok + self.pos_embed[:, :tok.size(1)])
        tok = self.encoder(tok)
        tok = self.norm(tok)
        cls_feat = tok[:, 0]
        patch_tok = tok[:, 1:]
        alpha = torch.softmax(self.attn_pool(patch_tok), dim=1)
        patch_summary = torch.sum(alpha * patch_tok, dim=1)
        feat = torch.cat([cls_feat, patch_summary], dim=-1)
        logits = self.head(feat)
        return {"logits": logits, "features": feat, "token_attention": alpha.squeeze(-1), "feature_map": fused}


class DeformableMHSA3D(nn.Module):
    """3D patch-token deformable MHSA approximation.

    The public DA-ViT repository exposes learnable offsets but uses standard
    attention as placeholder. Here the offsets are injected as a learnable
    spatial bias over the 3D patch grid, keeping the model executable for MRI.
    """
    def __init__(self, dim: int, heads: int, grid_size: Tuple[int, int, int], dropout: float = 0.1, offset_scale: float = 0.25):
        super().__init__()
        if dim % heads != 0:
            raise ValueError("embed_dim must be divisible by n_heads.")
        self.dim = dim
        self.heads = heads
        self.head_dim = dim // heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)
        self.offset_scale = float(offset_scale)
        d, h, w = [int(v) for v in grid_size]
        zz, yy, xx = torch.meshgrid(
            torch.linspace(-1.0, 1.0, d),
            torch.linspace(-1.0, 1.0, h),
            torch.linspace(-1.0, 1.0, w),
            indexing="ij",
        )
        grid = torch.stack([zz, yy, xx], dim=-1).reshape(-1, 3)
        self.register_buffer("grid", grid, persistent=False)
        self.offsets = nn.Parameter(torch.zeros(1, heads, grid.shape[0], 3))
        self.bias_gain = nn.Parameter(torch.tensor(1.0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, n, d = x.shape
        qkv = self.qkv(x).reshape(b, n, 3, self.heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale

        if n == self.grid.shape[0]:
            grid = self.grid.to(device=x.device, dtype=x.dtype)
            deformed = grid.unsqueeze(0) + torch.tanh(self.offsets[0].to(dtype=x.dtype)) * self.offset_scale
            dist2 = torch.cdist(deformed, grid.unsqueeze(0).expand(self.heads, -1, -1), p=2).pow(2)
            attn = attn - F.softplus(self.bias_gain).to(dtype=x.dtype) * dist2.unsqueeze(0)

        attn = torch.softmax(attn, dim=-1)
        attn = self.drop(attn)
        out = torch.matmul(attn, v).transpose(1, 2).reshape(b, n, d)
        return self.proj(out)


class DeformableTransformerBlock3D(nn.Module):
    def __init__(self, dim: int, heads: int, grid_size: Tuple[int, int, int], mlp_ratio: float = 4.0, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = DeformableMHSA3D(dim, heads, grid_size, dropout=dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(int(dim * mlp_ratio), dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class DAViT3DBaseline(BaseBaseline):
    """DA-ViT-style baseline adapted from 2D MRI patches to 3D MRI patches."""
    def __init__(
        self,
        input_shape: Tuple[int, int, int] = (128, 128, 128),
        n_classes: int = 3,
        embed_dim: int = 128,
        heads: int = 4,
        depth: int = 4,
        patch_size: Tuple[int, int, int] = (16, 16, 16),
        dropout: float = 0.1,
    ):
        super().__init__()
        self.patch = PatchEmbed3D(1, embed_dim, patch_size)
        grid_size = tuple(max(1, int(s) // int(p)) for s, p in zip(input_shape, patch_size))
        n_tokens = int(np.prod(grid_size))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_tokens, embed_dim))
        self.pos_drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            DeformableTransformerBlock3D(embed_dim, heads, grid_size, mlp_ratio=4.0, dropout=dropout)
            for _ in range(max(1, depth))
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, n_classes)
        self.n_classes = n_classes
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        tok = self.patch(x)
        tok = self.pos_drop(tok + self.pos_embed[:, :tok.size(1)])
        for blk in self.blocks:
            tok = blk(tok)
        tok = self.norm(tok)
        feat = tok.mean(dim=1)
        logits = self.head(feat)
        return {"logits": logits, "features": feat, "patch_tokens": tok}


# -----------------------------------------------------------------------------
# 3) Override the model factory to include the requested frameworks.
# -----------------------------------------------------------------------------
def build_baseline_model(name: str, cfg: BaselineModelConfig, roi_masks: Optional[torch.Tensor] = None) -> nn.Module:
    key = name.strip().lower()
    if key in {"cnn_design_for_ad", "cnndesignforad", "cnn_design_for_adbaseline"}:
        return CNNDesignForADBaseline(n_classes=cfg.n_classes, base_ch=cfg.base_ch)
    if key in {"densenet-cnn", "densenet_cnn", "densenetcnn"}:
        return DenseNetCNNBaseline(n_classes=cfg.n_classes, base_ch=max(16, cfg.base_ch // 2))
    if key in {"vit", "visiontransformer", "vision-transformer"}:
        return ViTBaseline(
            input_shape=cfg.input_shape,
            n_classes=cfg.n_classes,
            embed_dim=cfg.embed_dim,
            heads=cfg.n_heads,
            depth=max(3, cfg.n_layers + 1),
            patch_size=cfg.patch_size,
            dropout=cfg.dropout,
        )
    if key in {"da-vit", "davit", "deformable-vit", "deformable_vit", "deformablemhsa-vit"}:
        return DAViT3DBaseline(
            input_shape=cfg.input_shape,
            n_classes=cfg.n_classes,
            embed_dim=cfg.embed_dim,
            heads=cfg.n_heads,
            depth=max(2, cfg.n_layers + 1),
            patch_size=cfg.patch_size,
            dropout=cfg.dropout,
        )
    if key in {"bifpn3dvit", "bifpn-3d-vit", "bifpn_3d_vit", "bifpn3dvitbaseline"}:
        return BiFPN3DViTBaseline(
            input_shape=cfg.input_shape,
            n_classes=cfg.n_classes,
            base_ch=max(8, cfg.base_ch),
            embed_dim=cfg.embed_dim,
            heads=cfg.n_heads,
            depth=max(1, cfg.n_layers),
            dropout=cfg.dropout,
        )
    if key in {"joint-transformer", "joint_transformer", "jointtransformer"}:
        return JointTransformerBaseline(
            n_classes=cfg.n_classes,
            embed_dim=cfg.embed_dim,
            heads=cfg.n_heads,
            depth=cfg.n_layers,
            n_slices=cfg.n_slice_tokens,
            dropout=cfg.dropout,
        )
    if key in {"longformer", "long_former"}:
        return LongFormerBaseline(
            input_shape=cfg.input_shape,
            n_classes=cfg.n_classes,
            embed_dim=cfg.embed_dim,
            heads=cfg.n_heads,
            depth=max(3, cfg.n_layers + 1),
            patch_size=cfg.patch_size,
            window=cfg.longformer_window,
            dropout=cfg.dropout,
        )
    if key in {"aagn", "roiawaregating", "aagnstyle"}:
        if roi_masks is None:
            raise ValueError("AAGN baseline requires roi_masks from the atlas manager.")
        return ROIAwareGatingBaseline(
            roi_masks=roi_masks,
            n_classes=cfg.n_classes,
            base_ch=cfg.base_ch,
            embed_dim=cfg.embed_dim,
        )
    if key in {"fastersnn", "faster_snn"}:
        return FasterSNNBaseline(n_classes=cfg.n_classes, base_ch=max(16, cfg.base_ch // 2))
    raise ValueError(f"Unknown baseline model name: {name}")


# -----------------------------------------------------------------------------
# 4) Override CV routines: validation split = internal; full other cohort = external.
# -----------------------------------------------------------------------------
def train_baseline_cv_fold(
    namespace: Mapping[str, Any],
    baseline_name: str,
    base_dir: str,
    project_root: str,
    module_dir: str,
    atlas_path: str,
    precomputed_artifacts_dir: str,
    cohort: str = "source",
    external_cohort: Optional[str] = "other",
    n_splits: int = 5,
    fold_idx: int = 0,
    train_cfg: Optional[BaselineTrainConfig] = None,
    model_cfg: Optional[BaselineModelConfig] = None,
    save_dir: Optional[str] = None,
) -> Dict[str, Any]:
    if train_cfg is None:
        train_cfg = BaselineTrainConfig()
    set_seed(int(train_cfg.seed) + int(fold_idx))

    LoadArtifacts = _require(namespace, "load_precomputed_artifacts")
    FindAtlas = _require(namespace, "find_existing_atlas_path")
    atlas_path = FindAtlas(atlas_path)
    cache = LoadArtifacts(
        base_dir=base_dir,
        module_dir=module_dir,
        atlas_path=atlas_path,
        precomputed_artifacts_dir=precomputed_artifacts_dir,
    )

    cohort = str(cohort).lower()
    dataset = _make_classification_dataset(cache, cohort=cohort)
    labels = dataset.labels_np
    input_shape = _infer_input_shape_from_dataset(dataset)

    if model_cfg is None:
        model_cfg = BaselineModelConfig(input_shape=input_shape)
    else:
        model_cfg.input_shape = input_shape

    if n_splits > np.min(np.bincount(labels)):
        raise ValueError(
            f"n_splits={n_splits} is larger than the smallest class count in cohort='{cohort}'. "
            f"Class counts: {dict(zip(*np.unique(labels, return_counts=True)))}"
        )

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    splits = list(skf.split(np.arange(len(labels)), labels))
    if fold_idx < 0 or fold_idx >= len(splits):
        raise ValueError(f"fold_idx must be in [0, {len(splits)-1}], got {fold_idx}")
    train_idx, val_idx = splits[fold_idx]

    train_loader = _make_loader_from_subset_v2(dataset, train_idx, train_cfg.batch_size, True, train_cfg.num_workers)
    train_eval_loader = _make_loader_from_subset_v2(dataset, train_idx, train_cfg.batch_size, False, train_cfg.num_workers)
    val_loader = _make_loader_from_subset_v2(dataset, val_idx, train_cfg.batch_size, False, train_cfg.num_workers)

    external_name = _resolve_external_cohort(cohort, external_cohort)
    external_loader = None
    external_n = 0
    if external_name is not None:
        external_dataset = _make_classification_dataset(cache, cohort=external_name)
        external_n = len(external_dataset)
        external_loader = _make_loader(external_dataset, train_cfg.batch_size, False, train_cfg.num_workers, drop_last=False)

    roi_masks = (
        _resolve_roi_masks(namespace, atlas_path, input_shape)
        if baseline_name.strip().lower() in {"aagn", "roiawaregating", "aagnstyle"}
        else None
    )

    model = build_baseline_model(baseline_name, model_cfg, roi_masks=roi_masks)
    trainer = ClassificationOnlyTrainer(model, train_cfg)
    result = trainer.fit(train_loader, train_eval_loader, val_loader, external_loader=external_loader)

    payload = {
        "baseline_name": baseline_name,
        "cohort": cohort,
        "external_cohort": external_name,
        "fold_idx": int(fold_idx),
        "n_train": int(len(train_idx)),
        "n_val": int(len(val_idx)),
        "n_external": int(external_n),
        "train_cfg": asdict(train_cfg),
        "model_cfg": asdict(model_cfg),
        "best_score": float(result["best_score"]),
        "final_val": result["final_val"],
        "final_external": result["final_external"],
        "history": result["history"],
    }

    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        stem = f"{baseline_name}_train-{cohort}_external-{external_name}_fold-{fold_idx}"
        safe_stem = stem.replace("/", "_").replace(" ", "_")
        torch.save(result["state_dict"], os.path.join(save_dir, safe_stem + "_weights.pt"))
        with open(os.path.join(save_dir, safe_stem + "_metrics.json"), "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)
    return payload


REQUESTED_BASELINES = [
    "AAGN",
    "Joint-Transformer",
    "FasterSNN",
    "CNN_design_for_AD",
    "DenseNet-CNN",
    "ViT",
    "DA-ViT",
    "BiFPN3DViT",
]


def run_baseline_cv_for_cohort(
    namespace: Mapping[str, Any],
    baseline_name: str,
    base_dir: str,
    project_root: str,
    module_dir: str,
    atlas_path: str,
    precomputed_artifacts_dir: str,
    cohort: str = "source",
    external_cohort: Optional[str] = "other",
    n_splits: int = 5,
    train_cfg: Optional[BaselineTrainConfig] = None,
    model_cfg: Optional[BaselineModelConfig] = None,
    save_dir: Optional[str] = None,
) -> List[Dict[str, Any]]:
    results = []
    for fold_idx in range(n_splits):
        print("\n" + "=" * 96)
        print(
            f"Running {baseline_name} | train cohort={cohort} | "
            f"external cohort={_resolve_external_cohort(cohort, external_cohort)} | "
            f"fold {fold_idx + 1}/{n_splits}"
        )
        print("=" * 96)
        out = train_baseline_cv_fold(
            namespace=namespace,
            baseline_name=baseline_name,
            base_dir=base_dir,
            project_root=project_root,
            module_dir=module_dir,
            atlas_path=atlas_path,
            precomputed_artifacts_dir=precomputed_artifacts_dir,
            cohort=cohort,
            external_cohort=external_cohort,
            n_splits=n_splits,
            fold_idx=fold_idx,
            train_cfg=train_cfg,
            model_cfg=model_cfg,
            save_dir=save_dir,
        )
        results.append(out)
    return results


def run_all_requested_baselines(
    namespace: Mapping[str, Any],
    base_dir: str,
    project_root: str,
    module_dir: str,
    atlas_path: str,
    precomputed_artifacts_dir: str,
    cohort: str = "source",
    external_cohort: Optional[str] = "other",
    n_splits: int = 5,
    train_cfg: Optional[BaselineTrainConfig] = None,
    model_cfg: Optional[BaselineModelConfig] = None,
    save_dir: Optional[str] = None,
    baseline_names: Optional[Sequence[str]] = None,
) -> Dict[str, List[Dict[str, Any]]]:
    if baseline_names is None:
        baseline_names = REQUESTED_BASELINES
    all_results: Dict[str, List[Dict[str, Any]]] = {}
    for name in baseline_names:
        all_results[name] = run_baseline_cv_for_cohort(
            namespace=namespace,
            baseline_name=name,
            base_dir=base_dir,
            project_root=project_root,
            module_dir=module_dir,
            atlas_path=atlas_path,
            precomputed_artifacts_dir=precomputed_artifacts_dir,
            cohort=cohort,
            external_cohort=external_cohort,
            n_splits=n_splits,
            train_cfg=train_cfg,
            model_cfg=model_cfg,
            save_dir=save_dir,
        )
        if save_dir is not None:
            os.makedirs(save_dir, exist_ok=True)
            with open(os.path.join(save_dir, f"{name}_train-{cohort}_external-{_resolve_external_cohort(cohort, external_cohort)}_summary.json"), "w", encoding="utf-8") as f:
                json.dump(all_results[name], f, indent=2)
    return all_results


def summarize_baseline_cv_results(all_results: Mapping[str, Sequence[Mapping[str, Any]]]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Return per-fold and mean±std summaries for internal and external validation."""
    rows = []
    for baseline, folds in all_results.items():
        for r in folds:
            row = {
                "baseline": baseline,
                "train_cohort": r.get("cohort"),
                "external_cohort": r.get("external_cohort"),
                "fold": r.get("fold_idx"),
                "n_train": r.get("n_train"),
                "n_val": r.get("n_val"),
                "n_external": r.get("n_external"),
            }
            for scope, key in [("val", "final_val"), ("external", "final_external")]:
                metrics = r.get(key, {}) or {}
                for m, v in metrics.items():
                    clean_m = m.replace(f"{scope}_", "")
                    row[f"{scope}_{clean_m}"] = v
            rows.append(row)
    df = pd.DataFrame(rows)
    metric_cols = [c for c in df.columns if c.startswith("val_") or c.startswith("external_")]
    summary = df.groupby(["baseline", "train_cohort", "external_cohort"], dropna=False)[metric_cols].agg(["mean", "std"])
    return df, summary





In [ ]:
# -----------------------------------------------------------------------------
# Example usage: external validation enabled by default
# -----------------------------------------------------------------------------
BASE_DIR = "/kaggle/input/notebooks/alejopatio/preprocess-alzheimer/model_ready_data"
PROJECT_ROOT = "/kaggle/input/notebooks/alejopatio/precompute-artifacts-alzheimer"
MODULE_DIR = "/kaggle/working/mri_da_missing"
ATLAS_PATH = "/kaggle/input/notebooks/alejopatio/precompute-artifacts-alzheimer/cerebra_prepared/CerebrA_discrete_ready.nii.gz"
PRECOMP_DIR = "/kaggle/input/notebooks/alejopatio/precompute-artifacts-alzheimer/precomputed_artifacts_cerebra"
SAVE_DIR = "/kaggle/working/baseline_runs_external"

train_cfg = BaselineTrainConfig(
    n_epochs=30,
    lr=1e-4,
    weight_decay=1e-4,
    batch_size=8,       # lower if DA-ViT or BiFPN3DViT exceeds GPU memory
    num_workers=0,
    device="cuda" if torch.cuda.is_available() else "cpu",
    early_stopping_patience=6,
)

model_cfg = BaselineModelConfig(
    n_classes=3,
    base_ch=16,         # 16 is safer for 3D ViT-like baselines on Kaggle GPUs
    embed_dim=128,
    n_heads=4,
    n_layers=2,
    patch_size=(16, 16, 16),
    dropout=0.1,
)

# Recommended protocol for clinically meaningful generalization:
#   train cohort = source  -> internal CV on source, external validation on target
#   train cohort = target  -> internal CV on target, external validation on source

all_results = run_all_requested_baselines(
    namespace=globals(),
    base_dir=BASE_DIR,
    project_root=PROJECT_ROOT,
    module_dir=MODULE_DIR,
    atlas_path=ATLAS_PATH,
    precomputed_artifacts_dir=PRECOMP_DIR,
    cohort="target",
    external_cohort="other",
    n_splits=5,
    train_cfg=train_cfg,
    model_cfg=model_cfg,
    save_dir=SAVE_DIR,
    baseline_names=[
        # "DA-ViT",
        # "BiFPN3DViT",
        # "CNN_design_for_AD",
        # "DenseNet-CNN",
        # "ViT",
        # "Joint-Transformer",
        "AAGN",
        "FasterSNN",
    ],
)
df_folds, df_summary = summarize_baseline_cv_results(all_results)
display(df_folds)
display(df_summary)
df_folds.to_csv(os.path.join(SAVE_DIR, "baseline_external_per_fold.csv"), index=False)
df_summary.to_csv(os.path.join(SAVE_DIR, "baseline_external_summary.csv"))